## **Part 1: Overview:** The notebook is run in the following order: the knowledge base, retrieval, the Phi-3 answer generator, the danger-sign safety layer, the Kiswahili layer, and the evaluation using precision@k, recall@k, MRR and nDCG. RAGAS evaluation is the last phase.


# Part 2: The English pipeline (Phases 1 to 5)

## Phase 1: Data Collection and Storage

## 1.1 Import json and packages

In [1]:
# Import json
import json

!pip install langchain-text-splitters sentence-transformers faiss-cpu -q

## 1.2 Construct the Knowledge base

In [2]:
maternal_documents = [
    # Antenatal Care
      {
  "id": "DOC_ANC_001",
  "domain": "maternal_health",
  "topic": "antenatal_care",
  "subtopic": "antenatal_care_overview",
  "source": "World Health Organization",
  "year": 2016,
  "url": "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content",
  "content": "The 2016 WHO guideline on antenatal care (ANC) sets out 49 recommendations across five areas: nutritional interventions, maternal and fetal assessment, preventive measures, management of common physiological symptoms, and health-system interventions to improve ANC quality. The guideline's central aim is a positive pregnancy experience, defined as maintaining physical and social wellbeing, preventing or managing complications, and supporting a woman's confidence and autonomy through pregnancy, not simply preventing death or illness."
},

    {
  "id": "DOC_ANC_002",
  "domain": "maternal_health",
  "topic": "antenatal_care",
  "subtopic": "antenatal_visits",
  "source": "World Health Organization",
  "year": 2016,
  "url": "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content",
  "content": "WHO's 2016 ANC model recommends a minimum of eight contacts between a pregnant woman and a health provider, replacing the earlier four-visit model, to reduce perinatal mortality and improve a woman's experience of care. The recommended schedule is: Contact 1 by 12 weeks of gestation, Contact 2 at 20 weeks, Contact 3 at 26 weeks, Contact 4 at 30 weeks, Contact 5 at 34 weeks, Contact 6 at 36 weeks, Contact 7 at 38 weeks, and Contact 8 at 40 weeks. If the woman has not given birth by 41 weeks, she should return for assessment. Any recommended intervention missed at one contact should be provided at the next. These antenatal visits allow health professionals to monitor the health of the mother and baby. These visits help detect complications early and provide appropriate counselling and treatment."
},

    {
  "id": "DOC_ANC_003",
  "domain": "maternal_health",
  "topic": "antenatal_care",
  "subtopic": "nutrition_and_lifestyle_pregnancy",
  "source": "World Health Organization",
  "year": 2016,
  "url": "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content",
  "content": "Counselling on healthy eating and staying physically active during pregnancy is recommended to support maternal health and prevent excessive weight gain. A healthy diet includes adequate energy, protein, vitamins and minerals from a variety of foods such as vegetables, meat, fish, beans, nuts, whole grains and fruit. For women with high daily caffeine intake (more than 300 mg per day, from sources such as coffee, tea, cola drinks, energy drinks or chocolate), lowering caffeine intake during pregnancy is recommended to reduce the risk of pregnancy loss and low birth weight."
},

    {
  "id": "DOC_ANC_004",
  "domain": "maternal_health",
  "topic": "antenatal_care",
  "subtopic": "supplementation_pregnancy",
  "source": "World Health Organization",
  "year": 2016,
  "url": "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content",
  "content": "Daily oral iron and folic acid supplementation (30-60 mg elemental iron and 400 micrograms/0.4 mg folic acid) is recommended for all pregnant women to prevent maternal anaemia, low birth weight and preterm birth. Where daily dosing causes side-effects and anaemia prevalence among pregnant women is below 20%, intermittent dosing (120 mg iron and 2.8 mg folic acid once weekly) is a recommended alternative. In populations with low dietary calcium intake, daily calcium supplementation (1.5-2.0 g elemental calcium) is recommended to reduce the risk of pre-eclampsia. Vitamin A supplementation is recommended only in areas where vitamin A deficiency is a severe public health problem, to prevent night blindness; it is not recommended as routine practice elsewhere."
},

    {
  "id": "DOC_ANC_005",
  "domain": "maternal_health",
  "topic": "antenatal_care",
  "subtopic": "maternal_assessment",
  "source": "World Health Organization",
  "year": 2016,
  "url": "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content",
  "content": "Routine maternal assessment during ANC includes: haemoglobin testing (full blood count, or an on-site haemoglobinometer where lab testing is unavailable) to diagnose anaemia; midstream urine culture, or on-site Gram-staining where culture is unavailable, to detect asymptomatic bacteriuria; screening for gestational diabetes, generally between 24 and 28 weeks; asking about tobacco use and about alcohol or other substance use at every visit; and HIV and syphilis testing, which is a routine part of care in high-HIV-prevalence settings. In areas where tuberculosis prevalence is 100 per 100,000 population or higher, systematic TB screening is also recommended. Providers should also consider asking about intimate partner violence where a private setting, confidentiality and a referral pathway are in place."
},

    {
  "id": "DOC_ANC_006",
  "domain": "maternal_health",
  "topic": "antenatal_care",
  "subtopic": "ultrasound_pregnancy",
  "source": "World Health Organization",
  "year": 2016,
  "url": "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content",
  "content": "One ultrasound scan before 24 weeks of gestation is recommended for all pregnant women. This early scan improves the accuracy of gestational age estimation, increases detection of fetal abnormalities and multiple pregnancies, reduces unnecessary induction for perceived post-term pregnancy, and is associated with a better overall pregnancy experience. Routine Doppler ultrasound of fetal blood vessels, by contrast, is not recommended as it has not been shown to improve maternal or perinatal outcomes."
},

    {
  "id": "DOC_ANC_007",
  "domain": "maternal_health",
  "topic": "antenatal_care",
  "subtopic": "preventive_measures_pregnancy",
  "source": "World Health Organization",
  "year": 2016,
  "url": "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content",
  "content": "Preventive measures recommended during pregnancy include: tetanus toxoid vaccination for all pregnant women, with the specific schedule depending on prior vaccination history, to prevent neonatal tetanus; a seven-day antibiotic course for women diagnosed with asymptomatic bacteriuria, to prevent preterm birth and low birth weight; and, in areas with high worm-infection burden, anthelminthic treatment after the first trimester. In malaria-endemic areas of Africa, intermittent preventive treatment with sulfadoxine-pyrimethamine (IPTp-SP) is recommended for all pregnant women, starting in the second trimester, with doses spaced at least one month apart, aiming for a minimum of three doses over the pregnancy."
},

    {
  "id": "DOC_ANC_008",
  "domain": "maternal_health",
  "topic": "antenatal_care",
  "subtopic": "pregnancy_education",
  "source": "World Health Organization",
  "year": 2016,
  "url": "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content",
  "content": "At every ANC contact, providers should support birth-preparedness and complication-readiness planning, which is associated with increased use of skilled birth attendance and faster use of facility care when complications arise. Individualized counselling should also cover healthy eating and physical activity, tobacco and substance use, malaria and HIV prevention, and the meaning of any test results. Women should be advised to seek care immediately if they notice danger signs, rather than waiting for their next scheduled contact."
},

    # Physiological symptoms
    {
    "id": "DOC_ANC_009",
    "domain": "maternal_health",
    "topic": "physiological_symptoms",
    "subtopic": "nausea_and_vomiting",
    "source": "World Health Organization",
    "year": 2016,
    "url": "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content",
    "content": "Ginger, chamomile, vitamin B6 and/or acupuncture are recommended for relieving nausea in early pregnancy, based on a woman's preferences and what's available; these non-pharmacological options are considered unlikely to harm mother or baby. Nausea and vomiting affects around 70% of pregnant women, usually in the first trimester, and symptoms generally resolve by the second half of pregnancy. Pharmacological treatments such as doxylamine or metoclopramide should be reserved for distressing symptoms, and used only under a doctor's supervision."
    },

    {
    "id": "DOC_ANC_010",
    "domain": "maternal_health",
    "topic": "physiological_symptoms",
    "subtopic": "heartburn",
    "source": "World Health Organization",
    "year": 2016,
    "url": "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content",
    "content": "Diet and lifestyle advice is recommended first for preventing and relieving heartburn in pregnancy: avoiding large or fatty meals and alcohol, stopping smoking, and raising the head of the bed while sleeping. If symptoms persist, antacid preparations (such as magnesium carbonate or aluminium hydroxide) can be offered and are considered unlikely to cause harm at recommended doses. Antacids can impair absorption of other medications, so they should not be taken within two hours of iron and folic acid supplements."
    },

    {
    "id": "DOC_ANC_011",
    "domain": "maternal_health",
    "topic": "physiological_symptoms",
    "subtopic": "leg_cramps",
    "source": "World Health Organization",
    "year": 2016,
    "url": "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content",
    "content": "Magnesium, calcium, or non-pharmacological approaches can be used to relieve leg cramps in pregnancy, based on a woman's preferences and what's available. Evidence on magnesium and calcium for this purpose is limited, but they are considered unlikely to be harmful. There is currently no strong evidence for non-drug approaches such as stretching, heat therapy, or massage, though they are commonly tried."
    },

    {
    "id": "DOC_ANC_012",
    "domain": "maternal_health",
    "topic": "physiological_symptoms",
    "subtopic": "low_back_and_pelvic_pain",
    "source": "World Health Organization",
    "year": 2016,
    "url": "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content",
    "content": "Regular exercise throughout pregnancy, on land or in water, is recommended to prevent low back and pelvic pain, as part of the same lifestyle advice that helps prevent excessive weight gain. Physiotherapy, support belts, and acupuncture are treatment options once pain occurs, based on a woman's preference. Exercise may relieve low back pain but is not recommended specifically for pelvic pain caused by symphysis pubis dysfunction, as it can worsen that condition. Most women find these symptoms improve within a few months after birth."
    },

    {
    "id": "DOC_ANC_013",
    "domain": "maternal_health",
    "topic": "physiological_symptoms",
    "subtopic": "constipation",
    "source": "World Health Organization",
    "year": 2016,
    "url": "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content",
    "content": "For constipation in pregnancy, dietary advice is the first step: adequate water intake and dietary fibre from vegetables, nuts, fruit, and whole grains. If constipation persists despite dietary changes, wheat bran or other fibre supplements are recommended. For troublesome constipation not relieved by diet or fibre, intermittent use of poorly-absorbed laxatives may be considered."
    },

    {
    "id": "DOC_ANC_014",
    "domain": "maternal_health",
    "topic": "physiological_symptoms",
    "subtopic": "varicose_veins_and_oedema",
    "source": "World Health Organization",
    "year": 2016,
    "url": "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content",
    "content": "Non-pharmacological options are recommended for managing varicose veins and oedema (swelling) in pregnancy, based on a woman's preference: compression stockings, leg elevation, and water immersion. These are low-cost and considered unlikely to cause harm. Women should be informed that varicose veins may worsen as pregnancy progresses, but most see improvement within a few months of giving birth."
    },

    # Continuity of care
  {
    "id": "DOC_ANC_015",
    "domain": "maternal_health",
    "topic": "continuity_of_care",
    "subtopic": "woman_held_case_notes",
    "source": "World Health Organization",
    "year": 2016,
    "url": "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content",
    "content": "Each pregnant woman is recommended to carry her own case notes during pregnancy, whether paper-based or electronic, to improve continuity and quality of care and her overall pregnancy experience. Women who carry their own notes report greater access to pregnancy-related information and a stronger sense of involvement in their own care. Health systems implementing this should be mindful that women with lower health literacy may need support understanding their notes, and that data privacy and sensitive information (such as HIV status) need appropriate safeguards."
  },

  {
    "id": "DOC_ANC_016",
    "domain": "maternal_health",
    "topic": "continuity_of_care",
    "subtopic": "midwife_led_continuity_of_care",
    "source": "World Health Organization",
    "year": 2016,
    "url": "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content",
    "content": "In settings with well-functioning midwifery programmes, midwife-led continuity-of-care models are recommended, where a known midwife or small group of known midwives supports a woman through pregnancy, birth, and the postnatal period. This model requires enough trained midwives for each woman to see the same one or two providers throughout, and is not recommended as a starting point in settings without an already well-functioning midwifery workforce; those settings should focus first on scaling up midwife numbers and quality."
  },

    # Danger signs during pregnancy
    {
  "id": "DOC_DS_017",
  "domain": "maternal_health",
  "topic": "danger_signs_pregnancy",
  "subtopic": "danger_signs_overview",
  "source": ["World Health Organization", "Centers for Disease Control and Prevention", "UNICEF", "Kenya Ministry of Health"],
   "url": [
    "https://www.ncbi.nlm.nih.gov/books/NBK304190/pdf/Bookshelf_NBK304190.pdf",
    "https://www.cdc.gov/hearher/resources/download-share/docs/pdf/Warning-Signs-Poster-LTR-English.pdf",
    "https://www.unicef.org/timorleste/media/2386/file/Danger%20signs%20for%20women%20-%20English%20.pdf",
    "https://health.go.ke/sites/default/files/2023-05/Mother%20Child%20Health%20Handbook%20MOH-FOR%20PRINT.pdf"
  ],
  "year": 2024,
  "content": "Danger signs in pregnancy are symptoms that can signal a life-threatening complication for the mother or baby and require immediate care at a health facility, not a wait-and-see approach. Across WHO, CDC, UNICEF, and Kenya's Ministry of Health guidance, the core danger signs are consistent: vaginal bleeding, severe or persistent headache, blurred vision, convulsions or fits, difficulty breathing or chest pain, severe abdominal pain, high fever, severe swelling of the face or hands, reduced or absent fetal movement, severe nausea and vomiting, and fluid leaking from the vagina before the due date. A pregnant woman experiencing any of these should go to a health facility right away rather than waiting for her next scheduled antenatal visit."
},

    {
  "id": "DOC_DS_018",
  "domain": "maternal_health",
  "topic": "danger_signs_pregnancy",
  "subtopic": "vaginal_bleeding",
  "source": ["World Health Organization", "Centers for Disease Control and Prevention", "UNICEF", "Kenya Ministry of Health"],
  "url": [
    "https://www.ncbi.nlm.nih.gov/books/NBK304190/pdf/Bookshelf_NBK304190.pdf",
    "https://www.cdc.gov/hearher/resources/download-share/docs/pdf/Warning-Signs-Poster-LTR-English.pdf",
    "https://www.unicef.org/timorleste/media/2386/file/Danger%20signs%20for%20women%20-%20English%20.pdf",
    "https://health.go.ke/sites/default/files/2023-05/Mother%20Child%20Health%20Handbook%20MOH-FOR%20PRINT.pdf"
  ],
  "year": 2024,
  "content": "Vaginal bleeding at any point in pregnancy is one of the most urgent danger signs and requires going to a hospital or health centre immediately, not waiting to see if it stops. In early pregnancy, bleeding can be a sign of miscarriage or an ectopic pregnancy (where the pregnancy develops outside the womb). After around 28 weeks, bleeding is more often related to the placenta and is treated as an emergency requiring urgent skilled care. Bleeding is especially dangerous in women who are anaemic, since significant blood loss can become life-threatening within hours. Vaginal bleeding accompanied by fluid leakage may also indicate the membranes have ruptured."
},

   {
  "id": "DOC_DS_019",
  "domain": "maternal_health",
  "topic": "danger_signs_pregnancy",
  "subtopic": "headache_vision",
  "source": ["World Health Organization", "Centers for Disease Control and Prevention", "Kenya Ministry of Health"],
  "url": [
    "https://www.ncbi.nlm.nih.gov/books/NBK304190/pdf/Bookshelf_NBK304190.pdf",
    "https://www.cdc.gov/hearher/resources/download-share/docs/pdf/Warning-Signs-Poster-LTR-English.pdf",
    "https://health.go.ke/sites/default/files/2023-05/Mother%20Child%20Health%20Handbook%20MOH-FOR%20PRINT.pdf"
  ],
  "year": 2024,
  "content": "A severe headache that won't go away or keeps getting worse, especially with blurred vision, seeing spots, or other visual changes, is a danger sign that calls for immediate medical care. These symptoms most often appear after 20 weeks of pregnancy and can be a sign of preeclampsia, a blood pressure condition. Other symptoms that can appear alongside it include pain in the upper abdomen (especially the right side), sudden swelling of the face or hands, and nausea or vomiting that starts suddenly in the second half of pregnancy. Preeclampsia can be present with no symptoms at all, which is why blood pressure is checked at every antenatal visit; but once a woman feels this combination of symptoms, she should not wait for her next scheduled visit. If left untreated, preeclampsia can progress to eclampsia, a life-threatening condition involving seizures."
},

    {
  "id": "DOC_DS_020",
  "domain": "maternal_health",
  "topic": "danger_signs_pregnancy",
  "subtopic": "convulsions",
  "source": ["World Health Organization", "UNICEF", "Kenya Ministry of Health"],
  "url": [
    "https://www.ncbi.nlm.nih.gov/books/NBK304190/pdf/Bookshelf_NBK304190.pdf",
    "https://www.unicef.org/timorleste/media/2386/file/Danger%20signs%20for%20women%20-%20English%20.pdf",
    "https://health.go.ke/sites/default/files/2023-05/Mother%20Child%20Health%20Handbook%20MOH-FOR%20PRINT.pdf"
  ],
  "year": 2024,
  "content": "Convulsions or fits during pregnancy are a medical emergency and may be associated with severe conditions such as eclampsia. This condition can be life-threatening for both the mother and the baby. It sometimes follows warning signs such as severe headache, blurred vision, or upper abdominal pain, but a seizure can also occur suddenly without any prior symptoms. Eclampsia is not limited to pregnancy itself; it can also occur during labour or in the days and weeks after birth, so danger-sign awareness should continue through the postpartum period. At a health facility, treatment typically includes magnesium sulfate to control and prevent further seizures alongside close monitoring of the mother and baby."
},

    {
  "id": "DOC_DS_021",
  "domain": "maternal_health",
  "topic": "danger_signs_pregnancy",
  "subtopic": "breathing_chest_pain",
  "source": ["World Health Organization", "Centers for Disease Control and Prevention"],
  "url": [
    "https://www.ncbi.nlm.nih.gov/books/NBK304190/pdf/Bookshelf_NBK304190.pdf",
    "https://www.cdc.gov/hearher/resources/download-share/docs/pdf/Warning-Signs-Poster-LTR-English.pdf"
  ],
  "year": 2024,
  "content": "Trouble breathing, fast or difficult breathing, chest pain, or a rapidly beating heart are urgent danger signs requiring immediate medical care, whether they occur during pregnancy or in the year after giving birth. These symptoms can point to a blood clot that has travelled to the lungs, or to peripartum cardiomyopathy, a weakening of the heart muscle that can develop in the last weeks of pregnancy or in the months after delivery. Both conditions can become life-threatening quickly, so a woman should not wait to see if the symptoms pass; she should seek emergency care right away and tell the provider she is pregnant or recently gave birth."
},

    {
  "id": "DOC_DS_022",
  "domain": "maternal_health",
  "topic": "danger_signs_pregnancy",
  "subtopic": "abdominal_pain",
  "source": ["World Health Organization", "Centers for Disease Control and Prevention", "UNICEF", "Kenya Ministry of Health"],
  "url": [
    "https://www.ncbi.nlm.nih.gov/books/NBK304190/pdf/Bookshelf_NBK304190.pdf",
    "https://www.cdc.gov/hearher/resources/download-share/docs/pdf/Warning-Signs-Poster-LTR-English.pdf",
    "https://www.unicef.org/timorleste/media/2386/file/Danger%20signs%20for%20women%20-%20English%20.pdf",
    "https://health.go.ke/sites/default/files/2023-05/Mother%20Child%20Health%20Handbook%20MOH-FOR%20PRINT.pdf"
  ],
  "year": 2024,
  "content": "Severe abdominal pain that does not go away is a danger sign that calls for urgent medical evaluation. Mild, brief cramping is common as the uterus grows, but persistent or intense pain is different and its likely cause depends on timing. Early in pregnancy, severe lower abdominal pain with bleeding can signal an ectopic pregnancy or miscarriage. In the third trimester, sudden abdominal pain, especially with the belly feeling constantly hard or tender and possibly with vaginal bleeding, can indicate placental abruption, where the placenta separates from the uterus before birth; this is a medical emergency even if visible bleeding is minimal, since blood can become trapped inside the uterus. Pain concentrated under the ribs, particularly on the right side, can be a sign of preeclampsia. Any of these patterns warrants going to a health facility immediately rather than waiting."
},

    {
  "id": "DOC_DS_023",
  "domain": "maternal_health",
  "topic": "danger_signs_pregnancy",
  "subtopic": "fever",
  "source": ["World Health Organization", "Centers for Disease Control and Prevention", "UNICEF", "Kenya Ministry of Health"],
  "url": [
    "https://www.ncbi.nlm.nih.gov/books/NBK304190/pdf/Bookshelf_NBK304190.pdf",
    "https://www.cdc.gov/hearher/resources/download-share/docs/pdf/Warning-Signs-Poster-LTR-English.pdf",
    "https://www.unicef.org/timorleste/media/2386/file/Danger%20signs%20for%20women%20-%20English%20.pdf",
    "https://health.go.ke/sites/default/files/2023-05/Mother%20Child%20Health%20Handbook%20MOH-FOR%20PRINT.pdf"
  ],
  "year": 2024,
  "content": "A fever of 100.4°F (38°C) or higher during pregnancy is a danger sign, especially when the woman feels too weak to get out of bed. Fever can point to an infection of the womb and amniotic fluid (chorioamnionitis), often linked to foul-smelling vaginal discharge or waters that have already broken, or to other infections such as a urinary tract infection. In malaria-endemic areas, fever in pregnancy should also be evaluated for malaria, since it can be more severe in pregnant women and carries added risk for the baby. Because fever in pregnancy can progress quickly to a serious infection for both mother and baby, prompt evaluation at a health facility is needed rather than waiting to see if it resolves on its own."
},

    {
  "id": "DOC_DS_024",
  "domain": "maternal_health",
  "topic": "danger_signs_pregnancy",
  "subtopic": "swelling",
  "source": ["World Health Organization", "Centers for Disease Control and Prevention", "UNICEF", "Kenya Ministry of Health"],
  "url": [
    "https://www.ncbi.nlm.nih.gov/books/NBK304190/pdf/Bookshelf_NBK304190.pdf",
    "https://www.cdc.gov/hearher/resources/download-share/docs/pdf/Warning-Signs-Poster-LTR-English.pdf",
    "https://www.unicef.org/timorleste/media/2386/file/Danger%20signs%20for%20women%20-%20English%20.pdf",
    "https://health.go.ke/sites/default/files/2023-05/Mother%20Child%20Health%20Handbook%20MOH-FOR%20PRINT.pdf"
  ],
  "year": 2024,
  "content": "Mild swelling of the feet and ankles is common and normal in pregnancy, especially later in the day and in the third trimester, and usually eases with rest and leg elevation. It becomes a danger sign when it is sudden, severe, or affects the face and hands, particularly after 20 weeks, and does not improve with rest. This kind of swelling can be a sign of preeclampsia, especially when it appears together with a severe headache, blurred vision, or pain under the ribs. Swelling that affects one leg much more than the other, especially with pain, warmth, or redness, is a different concern and can indicate a blood clot rather than preeclampsia. Either pattern requires prompt medical assessment rather than waiting."
},

    {
  "id": "DOC_DS_025",
  "domain": "maternal_health",
  "topic": "danger_signs_pregnancy",
  "subtopic": "fetal_movement",
  "source": ["Centers for Disease Control and Prevention", "UNICEF", "Kenya Ministry of Health"],
  "url": [
    "https://www.cdc.gov/hearher/resources/download-share/docs/pdf/Warning-Signs-Poster-LTR-English.pdf",
    "https://www.unicef.org/timorleste/media/2386/file/Danger%20signs%20for%20women%20-%20English%20.pdf",
    "https://health.go.ke/sites/default/files/2023-05/Mother%20Child%20Health%20Handbook%20MOH-FOR%20PRINT.pdf"
  ],
  "year": 2024,
  "content": "A decrease, slowing, or complete absence of fetal movement during pregnancy is a danger sign that requires prompt medical attention. Pregnant women should not wait if they notice that the baby is moving less than usual or has stopped moving. They should seek care promptly at a health facility so that the wellbeing of the mother and baby can be assessed."
},

    {
  "id": "DOC_DS_026",
  "domain": "maternal_health",
  "topic": "danger_signs_pregnancy",
  "subtopic": "nausea_vomiting",
  "source": ["Centers for Disease Control and Prevention","UNICEF"],
  "url": [
    "https://www.cdc.gov/hearher/resources/download-share/docs/pdf/Warning-Signs-Poster-LTR-English.pdf",
    "https://www.unicef.org/timorleste/media/2386/file/Danger%20signs%20for%20women%20-%20English%20.pdf"
  ],
  "year": 2024,
  "content": "Severe or persistent nausea and vomiting during pregnancy is a danger sign that requires medical attention. A pregnant woman who keeps vomiting or is unable or unwilling to eat should not wait for the symptoms to resolve before seeking care. Prompt medical assessment at a health facility is recommended so that the woman's condition can be evaluated and appropriate care provided."
},

    {
  "id": "DOC_DS_027",
  "domain": "maternal_health",
  "topic": "danger_signs_pregnancy",
  "subtopic": "fluid_leakage",
  "source": ["Centers for Disease Control and Prevention","UNICEF","Kenya Ministry of Health"],
  "url": [
    "https://www.cdc.gov/hearher/resources/download-share/docs/pdf/Warning-Signs-Poster-LTR-English.pdf",
    "https://www.unicef.org/timorleste/media/2386/file/Danger%20signs%20for%20women%20-%20English%20.pdf",
    "https://health.go.ke/sites/default/files/2023-05/Mother%20Child%20Health%20Handbook%20MOH-FOR%20PRINT.pdf"
  ],
  "year": 2024,
    "content": "Leaking of fluid from the vagina during pregnancy is a danger sign that requires prompt medical attention. Fluid or amniotic discharge occurring before the due date should not be ignored or managed by waiting for it to stop. Pregnant women who experience fluid leakage should seek care promptly at a health facility so that their condition and the wellbeing of the baby can be assessed. Breaking of the water is also identified as a danger sign during pregnancy, and women are advised to seek skilled care at a health facility when this occurs."
},

    {
  "id": "DOC_DS_028",
  "domain": "maternal_health",
  "topic": "danger_signs_pregnancy",
  "subtopic": "dizziness_fainting",
  "source": "Centers for Disease Control and Prevention",
  "year": 2024,
  "url": "https://www.cdc.gov/hearher/resources/download-share/docs/pdf/Warning-Signs-Poster-LTR-English.pdf",
  "content": "Dizziness or fainting during pregnancy may indicate underlying health problems such as anemia or low blood pressure. Persistent or severe episodes should be evaluated by a healthcare provider."
},

    {
  "id": "DOC_DS_029",
  "domain": "maternal_health",
  "topic": "danger_signs_pregnancy",
  "subtopic": "anemia_signs",
  "source": "Kenya Ministry of Health",
  "year": 2020,
  "url": "https://health.go.ke/sites/default/files/2023-05/Mother%20Child%20Health%20Handbook%20MOH-FOR%20PRINT.pdf",
  "content": "Paleness of the eyes and tongue during pregnancy may indicate anemia. Anemia reduces the oxygen-carrying capacity of the blood and can affect both the mother and baby. Pregnant women showing paleness or signs of anemia should seek skilled care at a health facility immediately. Hemoglobin (Hb) levels and pallor are routinely monitored during antenatal care contacts, with Iron and Folic Acid Supplementation (IFAS) provided to prevent anemia."
},

    {
  "id": "DOC_DS_030",
  "domain": "maternal_health",
  "topic": "danger_signs_pregnancy",
  "subtopic": "mental_health",
  "source": "Centers for Disease Control and Prevention",
  "year": 2024,
  "url": "https://www.cdc.gov/hearher/resources/download-share/docs/pdf/Warning-Signs-Poster-LTR-English.pdf",
  "content": "Thoughts of harming oneself or the baby during pregnancy are serious warning signs that require immediate attention. These symptoms may indicate severe emotional or mental health challenges. Urgent professional support is necessary."
},

    #Preeclampsia
    {
  "id": "DOC_PRE_031",
  "domain": "maternal_health",
  "topic": "preeclampsia",
  "subtopic": "preeclampsia_definition",
  "source": "World Health Organization",
  "year": 2025,
  "url":"https://www.who.int/news-room/fact-sheets/detail/pre-eclampsia",
  "content": "Preeclampsia is a high blood pressure disorder that typically develops after 20 weeks of pregnancy. It is clinically diagnosed by the onset of hypertension (blood pressure ≥140/90 mmHg) and proteinuria (protein in the urine) or end-organ dysfunction after 20 weeks of gestation. It poses serious health risks to both mother and baby if not properly managed, and early detection is crucial to prevent progression to eclampsia."
},

    {
  "id": "DOC_PRE_032",
  "domain": "maternal_health",
  "topic": "preeclampsia",
  "subtopic": "preeclampsia_diagnosis",
  "source": "World Health Organization",
  "year": 2025,
  "url":"https://www.who.int/news-room/fact-sheets/detail/pre-eclampsia",
  "content": "Pre-eclampsia is clinically diagnosed by the onset of hypertension (blood pressure ≥140/90 mmHg) and proteinuria (protein in the urine ≥0.3 g/24 hours) occurring after 20 weeks of gestation. These clinical markers indicate potential maternal and fetal complications that require prompt medical evaluation and management."
},

    {
  "id": "DOC_PRE_033",
  "domain": "maternal_health",
  "topic": "preeclampsia",
  "subtopic": "eclampsia_progression",
  "source": "World Health Organization",
  "year": 2025,
  "url":"https://www.who.int/news-room/fact-sheets/detail/pre-eclampsia",
  "content": "If pre-eclampsia is not properly managed, it can progress to eclampsia. Eclampsia is a severe, life-threatening complication characterized by the onset of generalized seizures (convulsions) or coma in a pregnant or postpartum woman with pre-eclampsia. Eclampsia is an acute obstetric emergency requiring immediate medical intervention to protect both mother and baby."
},

    {
  "id": "DOC_PRE_034",
  "domain": "maternal_health",
  "topic": "preeclampsia",
  "subtopic": "preeclampsia_risk_groups",
  "source": "World Health Organization",
  "year": 2025,
  "url":"https://www.who.int/news-room/fact-sheets/detail/pre-eclampsia",
  "content": "Certain groups of women are at higher risk of developing preeclampsia. These include first-time mothers, women carrying multiple pregnancies, and those with conditions such as obesity, hypertension, diabetes or kidney disease."
},

    {
  "id": "DOC_PRE_035",
  "domain": "maternal_health",
  "topic": "preeclampsia",
  "subtopic": "preeclampsia_risk_factors",
  "source": "World Health Organization",
  "year": 2025,
  "url":"https://www.who.int/news-room/fact-sheets/detail/pre-eclampsia",
  "content": "A personal history of pre-eclampsia in a previous pregnancy or a family history of hypertensive disorders (such as a mother or sister having pre-eclampsia) substantially increases a woman's risk of developing the condition. Pre-existing chronic conditions—including chronic hypertension, pre-gestational diabetes, and renal disease further elevate the likelihood of developing pre-eclampsia and related pregnancy complications."
},

    {
  "id": "DOC_PRE_036",
  "domain": "maternal_health",
  "topic": "preeclampsia",
  "subtopic": "preeclampsia_asymptomatic",
  "source": "World Health Organization",
  "year": 2025,
  "url":"https://www.who.int/news-room/fact-sheets/detail/pre-eclampsia",
  "content": "Pre-eclampsia can develop without noticeable symptoms in its early stages. Because elevated blood pressure and protein in the urine may not cause obvious physical discomfort initially, routine blood pressure monitoring and urinalysis during regular antenatal care (ANC) visits are essential for early detection, prevention, and timely management."
},

    {
  "id": "DOC_PRE_037",
  "domain": "maternal_health",
  "topic": "preeclampsia",
  "subtopic": "preeclampsia_neurological_symptoms",
  "source": "World Health Organization",
  "year": 2025,
  "url":"https://www.who.int/news-room/fact-sheets/detail/pre-eclampsia",
  "content": "Persistent severe headaches and visual disturbances—such as blurred vision, seeing spots, flashing lights, or sensitivity to light—are key warning signs of pre-eclampsia. These neurological symptoms indicate potential cerebral involvement or worsening severe disease, requiring immediate emergency clinical evaluation to prevent complications such as eclampsia or organ damage."
},

    {
  "id": "DOC_PRE_038",
  "domain": "maternal_health",
  "topic": "preeclampsia",
  "subtopic": "preeclampsia_physical_symptoms",
  "source": "World Health Organization",
  "year": 2025,
  "url":"https://www.who.int/news-room/fact-sheets/detail/pre-eclampsia",
  "content": "Other symptoms of preeclampsia include swelling of the hands and face, upper abdominal pain, and nausea or vomiting after the first trimester. These signs should be taken seriously and evaluated by a healthcare provider."
},

    {
  "id": "DOC_PRE_039",
  "domain": "maternal_health",
  "topic": "preeclampsia",
  "subtopic": "preeclampsia_organ_damage",
  "source": "World Health Organization",
  "year": 2025,
  "url":"https://www.who.int/news-room/fact-sheets/detail/pre-eclampsia",
  "content": "Preeclampsia can affect multiple organs, including the liver, kidneys and brain. Severe cases may result in long-term or permanent organ damage."
},

    {
  "id": "DOC_PRE_040",
  "domain": "maternal_health",
  "topic": "preeclampsia",
  "subtopic": "hellp_syndrome",
  "source": "World Health Organization",
  "year": 2025,
  "url":"https://www.who.int/news-room/fact-sheets/detail/pre-eclampsia",
  "content": "A severe complication of preeclampsia is HELLP syndrome, which affects the liver and blood. It is an acronym for Hemolysis (breakdown of red blood cells), Elevated Liver enzymes, and Low Platelet count. HELLP syndrome primarily affects the blood and liver function, presenting serious medical risks to both mother and baby. It is a life-threatening condition that requires immediate medical intervention."
},

    {
  "id": "DOC_PRE_041",
  "domain": "maternal_health",
  "topic": "preeclampsia",
  "subtopic": "preeclampsia_pregnancy_risks",
  "source": "World Health Organization",
  "year": 2025,
  "url":"https://www.who.int/news-room/fact-sheets/detail/pre-eclampsia",
  "content": "Pre-eclampsia increases the risk of serious complications for both mother and baby. For the mother, it can lead to placental abruption (where the placenta peels away from the wall of the uterus before delivery) and organ damage. For the baby, it can cause poor growth in the womb (fetal growth restriction), low amniotic fluid, and early birth (preterm delivery) before the baby is fully developed."
},

    {
  "id": "DOC_PRE_042",
  "domain": "maternal_health",
  "topic": "preeclampsia",
  "subtopic": "magnesium_sulfate_treatment",
  "source": "World Health Organization",
  "year": 2025,
  "url":"https://www.who.int/news-room/fact-sheets/detail/pre-eclampsia",
  "content": "Magnesium sulfate is the main medicine used to prevent and treat seizures (fits) in women with severe pre-eclampsia. It is highly effective at reducing the risk of the condition progressing to eclampsia, protecting the health and safety of both the mother and the baby during care."
},

    {
  "id": "DOC_PRE_043",
  "domain": "maternal_health",
  "topic": "preeclampsia",
  "subtopic": "blood_pressure_control",
  "source": "World Health Organization",
  "year": 2025,
  "url":"https://www.who.int/news-room/fact-sheets/detail/pre-eclampsia",
  "content": "Antihypertensive medications are used to control high blood pressure in women with preeclampsia. Managing blood pressure helps prevent severe complications, such as stroke or organ damage in the mother, and protects the baby's health."
},

    {
  "id": "DOC_PRE_044",
  "domain": "maternal_health",
  "topic": "preeclampsia",
  "subtopic": "fetal_support",
  "source": "World Health Organization",
  "year": 2025,
  "url":"https://www.who.int/news-room/fact-sheets/detail/pre-eclampsia",
  "content": "Corticosteroids may be given to pregnant women with preeclampsia when early delivery is expected. These medications help mature the baby's lungs quickly, lowering the risk of severe breathing problems and other complications if the baby is born premature."
},

    {
  "id": "DOC_PRE_045",
  "domain": "maternal_health",
  "topic": "preeclampsia",
  "subtopic": "preeclampsia_prevention",
  "source": "World Health Organization",
  "year": 2025,
  "url":"https://www.who.int/news-room/fact-sheets/detail/pre-eclampsia",
  "content": "Low-dose aspirin starting in early pregnancy can help prevent preeclampsia in women who are at high risk. In addition, taking calcium supplements helps lower the risk of preeclampsia for mothers who do not get enough calcium in their daily diet."
},

    #Postpartum Hemorrhage
    {
  "id": "DOC_PPH_046",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "pph_definition",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Postpartum hemorrhage is heavy, excessive bleeding that occurs after childbirth. It is a critical medical emergency that can happen right after delivery or up to twelve weeks postpartum. Severe blood loss can cause a sudden drop in blood pressure, requiring immediate emergency care to locate and stop the bleeding."
},

    {
  "id": "DOC_PPH_047",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "pph_severity",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Severe postpartum hemorrhage can cause hypovolemic shock, a dangerous condition where organs such as the heart, brain, and kidneys do not receive enough blood flow due to massive blood loss. If the bleeding is not stopped promptly, hypovolemic shock can lead to multi-organ failure and life-threatening complications."
},

    {
  "id": "DOC_PPH_048",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "pph_diagnosis",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Postpartum hemorrhage is diagnosed when total blood loss exceeds one liter after childbirth, whether it is a vaginal delivery or a cesarean section. Healthcare providers also diagnose it if the mother shows signs of excessive blood loss, such as a sharp drop in blood pressure, a rapid heart rate, or pale skin. Doctors often weigh blood-soaked pads and monitor vital signs to evaluate the severity of the bleeding."
},

    {
  "id": "DOC_PPH_049",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "pph_timing",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Postpartum hemorrhage can happen immediately after childbirth or several weeks later. Primary postpartum hemorrhage occurs within the first 24 hours after delivery, when most cases happen. Secondary or late postpartum hemorrhage occurs between 24 hours and 12 weeks after giving birth, requiring prompt medical evaluation if heavy bleeding begins at home."
},

    {
  "id": "DOC_PPH_050",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "pph_symptoms",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Key warning signs of postpartum hemorrhage include continuous heavy vaginal bleeding, passing blood clots larger than a golf ball, dizziness, feeling faint, blurred vision, a fast heart rate, and pale or clammy skin. These symptoms point to significant blood loss and mean you need immediate emergency medical care."
},

    {
  "id": "DOC_PPH_051",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "pph_bleeding_signs",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "A primary sign of postpartum hemorrhage is continuous, heavy vaginal bleeding that rapidly soaks through sanitary pads or does not slow down. Passing large blood clots, especially those bigger than a golf ball, is another clear warning sign that requires urgent medical evaluation to check for excessive bleeding."
},

    {
  "id": "DOC_PPH_052",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "pph_shock_signs",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Warning signs of hypovolemic shock caused by severe postpartum hemorrhage include blurred vision, cold, pale, or clammy skin, intense dizziness, weakness, and fainting. These symptoms happen when heavy blood loss prevents organs from getting enough blood, signaling a critical medical emergency that needs immediate lifesaving intervention."
},

    {
  "id": "DOC_PPH_053",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "pph_physical_changes",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Physical signs of postpartum hemorrhage include a rapid heartbeat, shortness of breath, and unusual pain, pressure, or swelling in the vaginal or perineal area. Swelling and severe pain in these areas can happen when blood pools outside blood vessels, forming a hematoma that requires medical evaluation."
},

    {
  "id": "DOC_PPH_054",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "uterine_atony",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Uterine atony is the most common cause of postpartum hemorrhage, responsible for up to eighty percent of cases. It happens when the uterine muscles remain soft and weak after delivery instead of contracting firmly. Strong contractions are necessary to squeeze shut the open blood vessels where the placenta attached, so when the uterus fails to contract, heavy bleeding continues."
},

    {
  "id": "DOC_PPH_055",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "pph_trauma",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Physical trauma during childbirth, such as tears or lacerations to the vagina, cervix, perineum, or uterus, is a major cause of postpartum hemorrhage. These birth canal injuries can cause rapid, significant blood loss or allow blood to collect internally, requiring surgical repair or stitches to stop the bleeding."
},

    {
  "id": "DOC_PPH_056",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "retained_placenta",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Postpartum hemorrhage can occur if parts of the placenta remain attached to or trapped inside the uterus after childbirth. Retained placental tissue physically prevents the uterus from contracting firmly shut, allowing blood vessels at the placental site to keep bleeding until the remaining tissue is completely removed."
},

    {
  "id": "DOC_PPH_057",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "clotting_disorders",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Blood clotting disorders, such as von Willebrand disease or pregnancy-related conditions that lower blood platelets, significantly increase the risk of postpartum hemorrhage. When the blood cannot form normal clots, even minor bleeding from the placenta site or tissue tears can rapidly become severe and dangerous."
},

    {
  "id": "DOC_PPH_058",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "pph_risk_factors",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Risk factors for postpartum hemorrhage include multiple pregnancies, large babies, prolonged or rapid labour, obesity and advanced maternal age. However, about forty percent of postpartum hemorrhages occur in mothers with no known risk factors, making close monitoring essential for all births."
},

    {
  "id": "DOC_PPH_059",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "pph_medical_risks",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Specific medical conditions and clinical treatments significantly raise the risk of postpartum hemorrhage. These include placental complications, blood clotting disorders, uterine infections, and medical interventions like receiving general anesthesia or medications such as magnesium sulfate, which can relax the uterus and impair its ability to contract after birth."
},

    {
  "id": "DOC_PPH_060",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "pph_emergency_management",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Immediate treatment of postpartum hemorrhage focuses on stopping the bleeding and stabilizing the patient. Initial medical steps include firm uterine massage to stimulate contractions, providing intravenous fluids to support blood pressure, and giving oxygen through a mask to ensure organs receive adequate oxygenation."
},

    {
  "id": "DOC_PPH_061",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "pph_medical_treatment",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Medical treatments for postpartum hemorrhage involve administering medications to stimulate firm uterine contractions and stop active bleeding. If blood loss is severe, blood transfusions are given to replace lost red blood cells and restore blood volume, while iron supplements are often prescribed during recovery to help rebuild red blood cell levels."
},

    {
  "id": "DOC_PPH_062",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "pph_surgical_treatment",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "When non-surgical measures fail to stop postpartum hemorrhage, advanced interventions are required. These include stitching birth canal tears, inserting a special balloon catheter inside the uterus to exert pressure on bleeding vessels, or performing procedures to block the uterine blood supply. In rare, life-threatening cases where bleeding cannot be controlled, an emergency hysterectomy or exploratory surgery may be necessary."
},

    {
  "id": "DOC_PPH_063",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "pph_recovery",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Recovery from postpartum hemorrhage requires adequate rest, hydration, and nutritional support to help the body rebuild lost blood. Eating iron-rich foods like red meat, leafy greens, and fortified cereals helps boost red blood cell production and manage postpartum anemia, while close self-monitoring ensures bleeding steadily decreases over the following weeks."
},

    {
  "id": "DOC_PPH_064",
  "domain": "maternal_health",
  "topic": "postpartum_hemorrhage",
  "subtopic": "pph_emotional_health",
  "source": "Cleveland Clinic",
  "year": 2024,
  "url":"https://my.clevelandclinic.org/health/diseases/22228-postpartum-hemorrhage",
  "content": "Experiencing a postpartum hemorrhage and emergency medical interventions can be frightening and emotionally traumatic. Full recovery involves both physical and emotional healing, making it essential to lean on family, join support groups, or talk to healthcare professionals to process birth trauma and manage anxiety after severe childbirth complications."
},

    #Gestational diabetes Mellitus
    {
  "id": "DOC_GDM_065",
  "domain": "maternal_health",
  "topic": "gestational_diabetes_mellitus",
  "subtopic": "gdm_definition",
  "source": "International Diabetes Federation",
  "year": 2024,
  "url":"https://idf.org/about-diabetes/types-of-diabetes/gestational-diabetes/",
  "content": "Gestational diabetes mellitus is a condition characterized by high blood glucose levels that are first diagnosed during pregnancy. While blood sugar levels typically return to normal after the baby is born, having gestational diabetes increases the long-term risk of developing type 2 diabetes later in life."
},

    {
  "id": "DOC_GDM_066",
  "domain": "maternal_health",
  "topic": "gestational_diabetes_mellitus",
  "subtopic": "gdm_cause",
  "source": "International Diabetes Federation",
  "year": 2024,
  "url":"https://idf.org/about-diabetes/types-of-diabetes/gestational-diabetes/",
  "content": "Gestational diabetes occurs when the pancreas cannot produce enough insulin to meet the increased demands of pregnancy. During gestation, hormones released by the placenta cause insulin resistance by blocking how the mother's body uses insulin, leading to elevated blood sugar levels if the body cannot compensate."
},

    {
  "id": "DOC_GDM_067",
  "domain": "maternal_health",
  "topic": "gestational_diabetes_mellitus",
  "subtopic": "gdm_prevalence",
  "source": "International Diabetes Federation",
  "year": 2024,
  "url":"https://idf.org/about-diabetes/types-of-diabetes/gestational-diabetes/",
  "content": "Gestational diabetes is a common pregnancy complication globally. According to estimates by the International Diabetes Federation, approximately fifteen point six percent of live births worldwide—affecting roughly one in six pregnancies—involve some form of elevated blood glucose during pregnancy."
},

    {
  "id": "DOC_GDM_068",
  "domain": "maternal_health",
  "topic": "gestational_diabetes_mellitus",
  "subtopic": "gdm_risk_weight",
  "source": "International Diabetes Federation",
  "year": 2024,
  "url":"https://idf.org/about-diabetes/types-of-diabetes/gestational-diabetes/",
  "content": "Women who are overweight or obese before conceiving have a significantly higher risk of developing gestational diabetes. Carrying excess body weight amplifies the natural insulin resistance triggered by pregnancy hormones, making it harder for the body to keep blood glucose levels in a healthy range."
},

    {
  "id": "DOC_GDM_069",
  "domain": "maternal_health",
  "topic": "gestational_diabetes_mellitus",
  "subtopic": "gdm_risk_age",
  "source": "International Diabetes Federation",
  "year": 2024,
  "url":"https://idf.org/about-diabetes/types-of-diabetes/gestational-diabetes/",
  "content": "Advanced maternal age is a primary risk factor for gestational diabetes. While the risk begins to rise incrementally in women over twenty-five years of age, women over thirty-five and particularly those over forty-five face a significantly higher probability of developing high blood sugar during pregnancy due to age-related changes in pancreatic beta-cell function and insulin sensitivity."
},

    {
  "id": "DOC_GDM_070",
  "domain": "maternal_health",
  "topic": "gestational_diabetes_mellitus",
  "subtopic": "gdm_risk_medical_history",
  "source": "International Diabetes Federation",
  "year": 2024,
  "url":"https://idf.org/about-diabetes/types-of-diabetes/gestational-diabetes/",
  "content": "A personal history of gestational diabetes in a prior pregnancy or an underlying endocrine condition like polycystic ovary syndrome substantially increases the risk of developing gestational diabetes. These conditions are associated with baseline insulin resistance, which impairs the body's ability to maintain healthy blood glucose levels under the additional metabolic strain of pregnancy."
},

    {
  "id": "DOC_GDM_071",
  "domain": "maternal_health",
  "topic": "gestational_diabetes_mellitus",
  "subtopic": "gdm_risk_genetics",
  "source": "International Diabetes Federation",
  "year": 2024,
  "url":"https://idf.org/about-diabetes/types-of-diabetes/gestational-diabetes/",
  "content": "A family history of diabetes, particularly in a first-degree relative such as a parent or sibling, significantly increases the likelihood of developing gestational diabetes. Genetic factors influence pancreatic function and insulin sensitivity, making some women naturally more susceptible to impaired glucose processing when pregnancy hormones create insulin resistance."
},

    {
  "id": "DOC_GDM_072",
  "domain": "maternal_health",
  "topic": "gestational_diabetes_mellitus",
  "subtopic": "gdm_asymptomatic",
  "source": "International Diabetes Federation",
  "year": 2024,
  "url":"https://idf.org/about-diabetes/types-of-diabetes/gestational-diabetes/",
  "content": "Gestational diabetes often develops without any noticeable symptoms, or its signs such as mild fatigue and increased thirst are easily mistaken for typical pregnancy changes. Because the condition can go undetected while still affecting fetal growth, routine laboratory screening during prenatal care is essential for early detection and timely management."
},

    {
  "id": "DOC_GDM_073",
  "domain": "maternal_health",
  "topic": "gestational_diabetes_mellitus",
  "subtopic": "gdm_symptoms",
  "source": "International Diabetes Federation",
  "year": 2024,
  "url":"https://idf.org/about-diabetes/types-of-diabetes/gestational-diabetes/",
  "content": "When symptoms of gestational diabetes do appear, they typically stem from elevated blood sugar levels and include excessive thirst, frequent urination, unusual fatigue, and temporary blurred vision. Because these symptoms can overlap with common pregnancy discomforts, noticing them warrants prompt blood glucose testing."
},

    {
  "id": "DOC_GDM_074",
  "domain": "maternal_health",
  "topic": "gestational_diabetes_mellitus",
  "subtopic": "gdm_additional_symptoms",
  "source": "International Diabetes Federation",
  "year": 2024,
  "url":"https://idf.org/about-diabetes/types-of-diabetes/gestational-diabetes/",
  "content": "Additional symptoms associated with gestational diabetes include persistent nausea and recurrent infections, particularly vaginal yeast infections and urinary tract infections. Elevated glucose levels in bodily fluids create a favorable environment for fungal and bacterial growth, making these frequent infections a potential indicator of unmanaged high blood sugar."
},

    {
  "id": "DOC_GDM_075",
  "domain": "maternal_health",
  "topic": "gestational_diabetes_mellitus",
  "subtopic": "gdm_management_importance",
  "source": "International Diabetes Federation",
  "year": 2024,
  "url":"https://idf.org/about-diabetes/types-of-diabetes/gestational-diabetes/",
  "content": "Properly managing gestational diabetes is essential for maintaining a healthy pregnancy and preventing adverse outcomes. Maintaining target blood glucose levels significantly reduces maternal risks such as high blood pressure and surgical delivery, while protecting the developing baby from excessive birth weight, respiratory difficulties, and metabolic complications at birth."
},

    {
  "id": "DOC_GDM_076",
  "domain": "maternal_health",
  "topic": "gestational_diabetes_mellitus",
  "subtopic": "gdm_management",
  "source": "International Diabetes Federation",
  "year": 2024,
  "url":"https://idf.org/about-diabetes/types-of-diabetes/gestational-diabetes/",
  "content": "Gestational diabetes is primarily managed through structured lifestyle modifications, including a balanced diet tailored to control carbohydrate intake and regular moderate physical activity. When blood sugar targets cannot be maintained through diet and exercise alone, healthcare providers prescribe safe medications such as insulin injections or oral glucose-lowering therapy to keep blood glucose within target ranges."
},

    {
  "id": "DOC_GDM_077",
  "domain": "maternal_health",
  "topic": "gestational_diabetes_mellitus",
  "subtopic": "gdm_maternal_risks",
  "source": "International Diabetes Federation",
  "year": 2024,
  "url":"https://idf.org/about-diabetes/types-of-diabetes/gestational-diabetes/",
  "content": "Gestational diabetes increases maternal risks during and after pregnancy. In the short term, it elevates the risk of high blood pressure, preeclampsia, and labor complications such as difficult births or the need for a cesarean section. Long-term, women who have had gestational diabetes face a significantly higher lifetime risk of developing type 2 diabetes."
},

    {
  "id": "DOC_GDM_078",
  "domain": "maternal_health",
  "topic": "gestational_diabetes_mellitus",
  "subtopic": "gdm_fetal_risks",
  "source": "International Diabetes Federation",
  "year": 2024,
  "url":"https://idf.org/about-diabetes/types-of-diabetes/gestational-diabetes/",
  "content": "Babies born to mothers with unmanaged gestational diabetes face immediate and long-term health risks. Excess maternal blood glucose crosses the placenta, causing the fetus to produce extra insulin and gain excessive weight, which increases the likelihood of birth injuries and low blood sugar at delivery. In the long term, these children have an elevated risk of developing childhood obesity and type 2 diabetes."
},

    {
  "id": "DOC_GDM_079",
  "domain": "maternal_health",
  "topic": "gestational_diabetes_mellitus",
  "subtopic": "gdm_outcomes",
  "source": "International Diabetes Federation",
  "year": 2024,
  "url":"https://idf.org/about-diabetes/types-of-diabetes/gestational-diabetes/",
  "content": "With timely diagnosis and proper blood glucose management, the majority of women with gestational diabetes experience successful pregnancies and deliver healthy babies. Consistently following healthcare advice regarding diet, physical activity, and prescribed treatment effectively mitigates potential risks and leads to positive maternal and infant health outcomes."
},

    #Maternal Infections during pregnancy
    {
  "id": "DOC_INF_080",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "maternal_infections_overview",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Maternal infections during pregnancy pose significant risks to both maternal and fetal health. These infections can be bacterial, viral, or parasitic in origin, and without prompt screening, diagnosis, and treatment, they can lead to severe complications such as congenital defects, fetal growth restriction, preterm labor, or birth injury."
},

    {
  "id": "DOC_INF_081",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "chickenpox",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Chickenpox infection during early pregnancy, particularly within the first twenty weeks of gestation, can result in congenital varicella syndrome in the developing fetus. This serious condition can lead to permanent birth injuries, including neurological damage, microcephaly, hydrocephalus, vision impairment, skin scarring, and limb defects."
},

    {
  "id": "DOC_INF_082",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "chorioamnionitis",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Chorioamnionitis is a bacterial infection of the amniotic fluid and fetal membranes, typically caused by bacteria ascending from the mother's vagina into the uterus. It requires urgent medical treatment with antibiotics because it poses severe risks to the newborn, including neonatal sepsis, meningitis, pneumonia, and permanent neurological injury."
},

    {
  "id": "DOC_INF_083",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "chorioamnionitis_management",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Chorioamnionitis is treated with immediate intravenous antibiotics. In some cases, early delivery may be necessary to protect the baby from infection-related complications."
},

    {
  "id": "DOC_INF_084",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "gb_streptococcus",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Group B Streptococcus is a bacterial organism commonly present in the vagina and rectum of pregnant women without producing any symptoms or harm to the mother. However, if transmitted to the newborn during vaginal delivery, Group B Strep can cause life-threatening early-onset neonatal infections, including sepsis, pneumonia, and bacterial meningitis."
},

    {
  "id": "DOC_INF_085",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "gb_screening",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Pregnant women are routinely screened for Group B Streptococcus colonization using a vaginal-rectal swab between 36 and 37 weeks of gestation. If the test returns positive, intravenous antibiotics are administered during labor to effectively prevent mother-to-child bacterial transmission and protect the newborn from severe infection."
},

    {
  "id": "DOC_INF_086",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "hepatitis_b",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Hepatitis B infection can be transmitted from mother to infant during childbirth through contact with maternal blood and secretions. Without immediate post-exposure immunoprophylaxis at birth, most infected newborns develop chronic Hepatitis B, which significantly increases their risk of long-term liver disease, cirrhosis, and liver failure later in life."
},

    {
  "id": "DOC_INF_087",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "hepatitis_b_prevention",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "To prevent perinatal transmission of Hepatitis B, infants born to infected mothers must receive the Hepatitis B vaccine alongside Hepatitis B Immune Globulin within twelve hours of birth. This combination provides both immediate antibodies and long-term immune protection, preventing chronic infection in over ninety percent of exposed newborns."
},

    {
  "id": "DOC_INF_088",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "listeriosis",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Listeriosis is a foodborne bacterial infection caused by consuming food contaminated with Listeria monocytogenes, such as unpasteurized dairy products, soft cheeses, or deli meats. Although pregnant mothers may experience only mild, flu-like symptoms, the bacteria can cross the placenta and cause severe complications, including miscarriage, stillbirth, premature delivery, or life-threatening newborn infections."
},

    {
  "id": "DOC_INF_089",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "toxoplasmosis",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Toxoplasmosis is a parasitic infection caused by Toxoplasma gondii, typically contracted through contact with infected cat feces, contaminated soil, or raw and undercooked meat. Mothers often show no symptoms, but transplacental infection can cause congenital toxoplasmosis in the fetus, leading to severe long-term complications such as hydrocephalus, brain damage, intellectual disability, hearing loss, and chorioretinitis resulting in vision impairment."
},

    {
  "id": "DOC_INF_090",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "rubella",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Rubella infection contracted during early pregnancy, particularly in the first trimester, carries a critical risk of congenital rubella syndrome in the developing fetus. This severe viral condition can cause permanent birth defects, including congenital heart anomalies, sensorineural hearing loss, cataracts, microcephaly, and long-term intellectual and developmental delays."
},

    {
  "id": "DOC_INF_091",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "syphilis",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Syphilis is a sexually transmitted bacterial infection that can pass from mother to fetus during pregnancy or delivery. If left untreated, maternal syphilis causes congenital syphilis, leading to critical health risks such as stillbirth, premature birth, low birth weight, skeletal deformities, severe anemia, and permanent neurological and organ damage in the infant."
},

    {
  "id": "DOC_INF_092",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "syphilis_management",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Syphilis screening is a routine component of early prenatal care. When diagnosed promptly, maternal syphilis can be effectively cured using penicillin antibiotics, such as Benzathine Penicillin G, which cross the placenta to treat both the mother and fetus, successfully preventing congenital syphilis transmission and associated birth complications."
},

    {
  "id": "DOC_INF_093",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "uti_management",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Urinary tract infections are diagnosed through routine prenatal urinalysis and urine cultures and treated using pregnancy-safe antibiotics. Prompt diagnosis and early treatment are essential to prevent the infection from escalating into acute kidney infection or pyelonephritis, which significantly increases the risk of premature labor, low birth weight, and maternal sepsis."
},

    {
  "id": "DOC_INF_094",
  "domain": "maternal_health",
   "topic": "maternal_infections",
  "subtopic": "newborn_infection_signs",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Newborns exposed to maternal infections during pregnancy or delivery may display early signs of neonatal sepsis or infection. Key warning symptoms include difficulty breathing or rapid respiration, jaundice, poor feeding, body temperature instability, and extreme lethargy or unresponsiveness, all of which require immediate emergency medical evaluation."
},

    {
  "id": "DOC_INF_095",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "severe_newborn_symptoms",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Severe infection in newborns, such as neonatal meningitis or systemic sepsis, presents acute neurological and systemic warning signs including high fever or hypothermia, seizures, sensitivity to light, a bulging fontanelle, and extreme irritability or unresponsiveness. These severe symptoms represent critical medical emergencies requiring immediate intensive care to prevent permanent brain damage or mortality."
},

    {
  "id": "DOC_INF_096",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "prenatal_screening",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Routine prenatal screening tests—including blood panels, urine cultures, and vaginal-rectal swabs—are essential for identifying maternal infections early in pregnancy. Timely diagnosis allows healthcare providers to initiate targeted treatments and prophylactic interventions, significantly reducing the risk of vertical transmission, birth injuries, and long-term health complications for both mother and newborn."
},

    {
  "id": "DOC_INF_097",
  "domain": "maternal_health",
  "topic": "maternal_infections",
  "subtopic": "prenatal_tests",
  "source": "Birth Injury Justice Center",
  "year": 2026,
  "url":"https://www.childbirthinjuries.com/birth-injury/maternal-infections/",
  "content": "Standard prenatal screening relies on routine diagnostic procedures including blood serology tests to detect infection or immunity for syphilis, rubella, Hepatitis B, and HIV, alongside vaginal-rectal swab cultures to identify Group B Streptococcus colonization. These diagnostic tests provide crucial clinical data to guide timely medical interventions and intrapartum antibiotic prophylaxis during pregnancy and labor."
},

    #Nutrition during pregnancy
    {
  "id": "DOC_NUT_098",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "nutrition_importance",
  "source": ["World Health Organization, UNICEF"],
  "year": 2012,
  "url": ["https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
          "https://www.unicef.org/nutrition/maternal"
          ],
  "content": "Proper maternal nutrition during pregnancy is fundamental to optimizing both maternal health and fetal development. Adequate intake of essential macronutrients and micronutrients supports placental function, healthy weight gain, and optimal organ formation in the fetus. Conversely, maternal undernutrition or key nutrient deficiencies increase the risks of intrauterine growth restriction, low birth weight, preterm birth, stillbirth, maternal anemia, and severe pregnancy complications."
},

    {
  "id": "DOC_NUT_099",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "increased_nutritional_needs",
  "source": ["World Health Organization, UNICEF"],
  "year": 2012,
  "url": ["https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
          "https://www.unicef.org/nutrition/maternal"
          ],
  "content": "Pregnancy significantly increases a woman's physiological demand for energy, protein, vitamins, and minerals. These additional nutrients are essential to build maternal tissues, expand uterine and placental structures, support a fifty percent increase in maternal blood volume, and fuel rapid fetal cellular growth and organ development."
},

    {
  "id": "DOC_NUT_100",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "energy_requirements",
  "source": ["World Health Organization, Johns Hopkins Medicine"],
  "year": 2012,
  "url": ["https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
          "https://www.hopkinsmedicine.org/health/wellness-and-prevention/nutrition-during-pregnancy"
          ],
  "content": "Pregnant women require a moderate increase in total energy intake rather than doubling their food consumption. Energy demands remain minimal during the first trimester, but step up to an additional three hundred to four hundred fifty calories daily during the second and third trimesters, representing a ten to fifteen percent increase over pre-pregnancy baseline needs to fuel healthy fetal tissue development and maternal weight gain."
},

    {
  "id": "DOC_NUT_101",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "weight_gain",
  "source": ["World Health Organization, Kenya Ministry of Health"],
  "year": 2012,
  "url": ["https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
          "http://guidelines.health.go.ke:8000/media/National_Guidelines_for_Quality_Obstetrics_and_Perinatal_Care.pdf"
          ],
  "content": "Healthy gestational weight gain is a key indicator of maternal and fetal well-being, typically ranging between ten to sixteen kilograms for women entering pregnancy at a normal body mass index. Target weight gain varies according to pre-pregnancy weight status, requiring gradual progression with minimal increase in the first trimester followed by steady weekly gains throughout the second and third trimesters to lower risks of low birth weight, macrosomia, and surgical delivery."
},

    {
  "id": "DOC_NUT_102",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "balanced_diet",
  "source": ["World Health Organization, Johns Hopkins Medicine"],
  "year": 2012,
  "url": ["https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
          "https://www.hopkinsmedicine.org/health/wellness-and-prevention/nutrition-during-pregnancy"],
  "content": "Pregnant women should consume a balanced diet that includes fruits, vegetables, whole grains, dairy products and protein-rich foods. A varied diet helps ensure adequate intake of essential nutrients needed for maternal health and fetal development."
},

    {
  "id": "DOC_NUT_103",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "protein",
  "source": ["World Health Organization, Aga Khan University Hospital"],
  "year": 2012,
  "url": ["https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
          "https://hospitals.aku.edu/pakistan/patients-families/Documents/Nutrition%20During%20Pregnancy.pdf"],
  "content": "Dietary protein provides essential amino acids necessary for rapid cellular growth, tissue synthesis, uterine enlargement, placental maintenance, and expanded maternal blood volume during pregnancy. Meeting increased daily protein requirements through varied animal and plant sources such as lean poultry, fish, eggs, dairy, beans, lentils, nuts, and seeds supports optimal fetal brain and muscular development and reduces risks of intrauterine growth restriction."
},

    {
  "id": "DOC_NUT_104",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "carbohydrates",
  "source": "World Health Organization",
  "year": 2012,
  "url": "https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
  "content": "Carbohydrates serve as the primary source of metabolic energy during pregnancy, ideally comprising fifty to sixty percent of total daily caloric intake to fuel maternal cellular functions and supply constant glucose across the placenta for fetal brain development. Prioritizing complex carbohydrates and whole grains over refined sugars delivers sustained glucose release, essential dietary fiber to prevent constipation, and vital B vitamins that help lower the risk of gestational diabetes."
},

    {
  "id": "DOC_NUT_105",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "fats_omega3",
  "source": ["World Health Organization, Aga Khan University Hospital"],
  "year": 2012,
  "url": ["https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
          "https://hospitals.aku.edu/pakistan/patients-families/Documents/Nutrition%20During%20Pregnancy.pdf"],
  "content": "Dietary fats play an essential structural role during pregnancy, particularly essential omega-3 fatty acids such as docosahexaenoic acid and eicosapentaenoic acid, which are vital for fetal brain morphogenesis, central nervous system growth, and visual acuity development. Consuming safe, low-mercury sources including salmon, sardines, flaxseeds, and walnuts supports placental development and fetal neurogenesis, whereas limiting saturated and industrial trans fats protects maternal vascular health."
},

    {
  "id": "DOC_NUT_106",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "dietary_fibre",
  "source": "World Health Organization",
  "year": 2012,
  "url": "https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
  "content": "Adequate dietary fiber intake during pregnancy is essential for counteracting progesterone-induced intestinal slowing, promoting regular bowel motility, and preventing gestational constipation and hemorrhoids. High-fiber diets rich in whole grains, legumes, vegetables, and fruits slow glucose absorption to improve glycemic control, lower the risk of gestational diabetes, and support vascular health to reduce susceptibility to preeclampsia."
},

    {
  "id": "DOC_NUT_107",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "micronutrients",
  "source": ["World Health Organization, Kenya Ministry of Health"],
  "year": 2012,
  "url": ["https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
          "http://guidelines.health.go.ke:8000/media/National_Guidelines_for_Quality_Obstetrics_and_Perinatal_Care.pdf"],
  "content": "Pregnancy significantly increases maternal requirements for essential micronutrients, particularly iron, folic acid, calcium, iodine, and zinc. Daily intake or targeted supplementation of these vitamins and minerals is necessary to support rapid fetal cell division, enable neural tube development, expand maternal red blood cell mass to prevent anemia, protect maternal bone health, and reduce the risks of pre-eclampsia, low birth weight, and congenital anomalies."
},

    {
  "id": "DOC_NUT_108",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "iron",
  "source": ["World Health Organization, Kenya Ministry of Health"],
  "year": 2012,
  "url": ["https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
          "http://guidelines.health.go.ke:8000/media/National_Guidelines_for_Quality_Obstetrics_and_Perinatal_Care.pdf"],
  "content": "Iron is an essential micronutrient during pregnancy required for maternal hemoglobin production, red blood cell synthesis, and fetal oxygen supply. Severe iron deficiency leads to maternal anemia, increased fatigue, heightened risk of infection, impaired fetal growth, and higher susceptibility to severe complications during childbirth such as postpartum hemorrhage and low birth weight."
},

    {
  "id": "DOC_NUT_109",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "folic_acid",
  "source": ["World Health Organization, Johns Hopkins Medicine"],
  "year": 2012,
  "url": ["https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
          "https://www.hopkinsmedicine.org/health/wellness-and-prevention/nutrition-during-pregnancy"],
  "content": "Folic acid is essentia for cellular division, rapid tissue growth, and proper neural tube development in early pregnancy. Starting supplementation prior to conception and continuing through the first trimester enables complete closure of the embryonic neural tube within twenty-eight days of conception, reducing the risk of major congenital brain and spinal cord defects such as spina bifida and anencephaly by up to seventy percent."
},

    {
  "id": "DOC_NUT_110",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "calcium",
  "source": ["World Health Organization, Aga Khan University Hospital"],
  "year": 2012,
  "url": ["https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
          "https://hospitals.aku.edu/pakistan/patients-families/Documents/Nutrition%20During%20Pregnancy.pdf"],
  "content": "Calcium is an essential mineral required for fetal skeletal mineralization, tooth bud formation, heart rate regulation, and neuromuscular development. Adequate daily maternal intake ensures the developing fetus receives sufficient calcium without depleting maternal bone density reserves, while simultaneously supporting vascular tone to lower the risk of gestational hypertension and preeclampsia."
},

    {
  "id": "DOC_NUT_111",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "iodine",
  "source": ["World Health Organization, Kenya Ministry of Health"],
  "year": 2012,
  "url": ["https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
          "http://guidelines.health.go.ke:8000/media/National_Guidelines_for_Quality_Obstetrics_and_Perinatal_Care.pdf"],
  "content": "Iodine is an essential nutrient that helps the thyroid gland produce hormones needed for healthy brain growth and body development in the baby. Getting enough iodine during pregnancy protects the baby's developing brain and nervous system, helping to prevent learning disabilities, growth delays, and thyroid problems after birth."
},

    {
  "id": "DOC_NUT_112",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "hydration",
  "source": ["World Health Organization, Johns Hopkins Medicine"],
  "year": 2012,
  "url": ["https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
          "https://www.hopkinsmedicine.org/health/wellness-and-prevention/nutrition-during-pregnancy"],
  "content": "Drinking enough water during pregnancy is essential to help form amniotic fluid, build extra blood volume, and carry nutrients to the developing baby. Pregnant women should drink about eight to ten glasses which is two to two and a half liters of water daily to support digestion, flush out bodily waste, and lower the risk of constipation, swelling, and urinary tract infections."
},

    {
  "id": "DOC_NUT_113",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "substances_to_avoid",
  "source": ["World Health Organization, Johns Hopkins Medicine"],
  "year": 2012,
  "url": ["https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
          "https://www.hopkinsmedicine.org/health/wellness-and-prevention/nutrition-during-pregnancy"],
  "content": "Pregnant women should completely avoid alcohol and strictly limit caffeine during pregnancy. Alcohol passes directly to the baby and can cause severe lifelong physical and brain damage known as fetal alcohol spectrum disorders. High caffeine intake should be reduced to less than two hundred milligrams per day because excessive amounts can increase the risk of low birth weight and miscarriage."
},

    {
  "id": "DOC_NUT_114",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "poor_nutrition_risks",
  "source": "UNICEF",
  "year": 2026,
  "url": "https://www.unicef.org/nutrition/maternal",
  "content": "Poor maternal nutrition during pregnancy greatly increases the risk of severe health complications such as severe anemia, high blood pressure disorders like preeclampsia, heavy bleeding during delivery, and maternal death. For the baby, lack of proper maternal nutrients can lead to premature birth, low birth weight, poor physical growth, and lifelong brain or learning delays."
},

    {
  "id": "DOC_NUT_115",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "dietary_diversity",
  "source": ["Kenya Ministry of Health, UNICEF"],
  "year": 2022,
  "url": ["http://guidelines.health.go.ke:8000/media/National_Guidelines_for_Quality_Obstetrics_and_Perinatal_Care.pdf",
          "https://www.unicef.org/nutrition/maternal"],
  "content": "Eating a wide variety of locally available foods during pregnancy helps ensure mothers get all the essential vitamins and minerals needed for a healthy pregnancy. Eating from multiple food groups such as fresh vegetables, fruits, whole grains, beans, and lean meats prevents maternal malnutrition and supports healthy growth and development for the baby."
},

    {
  "id": "DOC_NUT_116",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "supplements",
  "source": ["World Health Organization, UNICEF"],
  "year": 2012,
  "url": ["https://cdn.who.int/media/docs/librariesprovider2/country-sites/maternal-nutrition-eng.pdf",
          "https://www.unicef.org/nutrition/maternal"],
  "content": "While a balanced diet provides most nutrients, pregnant women often need daily supplements like iron, folic acid, and calcium to meet their increased body needs. Taking these recommended supplements helps fill nutritional gaps, prevents maternal anemia and bone loss, and lowers the risk of birth defects and pregnancy complications."
},

    {
  "id": "DOC_NUT_117",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "diet_symptom_management",
  "source": "Aga Khan University Hospital",
  "year": 2018,
  "url": "https://hospitals.aku.edu/pakistan/patients-families/Documents/Nutrition%20During%20Pregnancy.pdf",
  "content": "Eating small frequent meals throughout the day helps ease common pregnancy uncomfortable symptoms like morning sickness, nausea, vomiting, heartburn, and constipation. Staying hydrated with water and choosing light, easily digestible foods keeps stomach acid down, improves digestion, and helps mothers feel more comfortable throughout pregnancy."
},

    #Iron and Folic acid supplementation
    {
  "id": "DOC_IFA_118",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "iron_deficiency_overview",
  "source": "World Health Organization",
  "year": 2024,
  "url": "https://www.who.int/tools/elena/interventions/daily-iron-pregnancy",
  "content": "Iron deficiency anemia is a very common global health issue that affects more than forty percent of pregnant women worldwide. Pregnant women need higher amounts of iron and folic acid to help build extra blood, keep their bodies strong, and support healthy development for the growing baby."
},

    {
  "id": "DOC_IFA_119",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "deficiency_risks",
  "source": "World Health Organization",
  "year": 2024,
  "url": "https://www.who.int/tools/elena/interventions/daily-iron-pregnancy",
  "content": "Deficiencies in iron and folic acid during pregnancy can can lead to serious health problems for both mother and baby. Low iron causes anemia, severe tiredness, and a higher chance of getting sick from infections, while low folic acid raises the risk of severe birth defects affecting the baby's brain and spine."
},

    {
  "id": "DOC_IFA_120",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "ifa_dosage",
  "source": "World Health Organization",
  "year": 2024,
  "url": "https://www.who.int/tools/elena/interventions/daily-iron-pregnancy",
  "content": "The World Health Organization recommends that pregnant women take a daily supplement containing thirty to sixty milligrams of elemental iron and four hundred micrograms or zero point four milligrams of folic acid. Taking this daily combination helps maintain healthy iron levels, prevents maternal anemia, and supports the baby's early physical and brain development."
},

    {
  "id": "DOC_IFA_121",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "ifa_benefits",
  "source": "World Health Organization",
  "year": 2024,
  "url": "https://www.who.int/tools/elena/interventions/daily-iron-pregnancy",
  "content": "Taking iron and folic acid supplements during pregnancy protects against maternal anemia, lowers the risk of serious infections after childbirth, and reduces the chances of premature birth or low birth weight in newborns. Regular use ensures both mother and baby stay healthy throughout the pregnancy."
},

    {
  "id": "DOC_IFA_122",
  "domain": "maternal_health",
  "topic": "nutrition_pregnancy",
  "subtopic": "folic_acid_timing",
  "source": "World Health Organization",
  "year": 2024,
  "url": "https://www.who.int/tools/elena/interventions/daily-iron-pregnancy",
  "content": "Folic acid supplementation should ideally begin before getting pregnant or as early as possible during the first weeks of pregnancy. Taking folic acid early in pregnancy provides critical protection when the baby's spine and brain are forming, significantly lowering the risk of severe birth defects known as neural tube defects."
},

    #Birth preparedness
    {
  "id": "DOC_BP_123",
  "domain": "maternal_health",
  "topic": "birth_preparedness",
  "subtopic": "birth_preparedness_overview",
  "source": ["World Health Organization, Kenyatta University Hospital"],
  "year": 2022,
  "url": ["https://www.who.int/tools/your-life-your-health/life-phase/pregnancy--birth-and-after-childbirth/making-plans-for-childbirth-when-pregnant",
          "https://ir-library.ku.ac.ke/server/api/core/bitstreams/b447bb8e-3ab1-414c-bd1b-cdc7a1af9c6d/content"],
  "content": "Birth preparedness is a key part of pregnancy care that helps mothers and their families plan ahead for a safe delivery and respond quickly if emergency complications happen. Making a clear plan during pregnancy prevents dangerous delays in seeking medical help, arranging transport, and getting quality health care when labor begins."
},

    {
  "id": "DOC_BP_124",
  "domain": "maternal_health",
  "topic": "birth_preparedness",
  "subtopic": "importance_planning",
  "source": "Kenyatta University Hospital",
  "year": 2014,
  "url": "https://ir-library.ku.ac.ke/server/api/core/bitstreams/b447bb8e-3ab1-414c-bd1b-cdc7a1af9c6d/content",
  "content": "Many families see pregnancy as a routine life event and do not make specific plans for labor or sudden health emergencies. Failing to prepare early often causes critical delays in recognizing warning signs and reaching a health facility. Preparing for birth ahead of time ensures families can quickly get skilled medical care as soon as labor begins or if complications occur."
},

    {
  "id": "DOC_BP_125",
  "domain": "maternal_health",
  "topic": "birth_preparedness",
  "subtopic": "birth_plan",
  "source": "World Health Organization",
  "year": 2022,
  "url": "https://www.who.int/tools/your-life-your-health/life-phase/pregnancy--birth-and-after-childbirth/making-plans-for-childbirth-when-pregnant",
  "content": "The World Health Organization recommends that every pregnant woman creates a written birth preparedness and emergency readiness plan. Having a clear written plan ensures key arrangements like selecting a health facility and saving emergency money are ready before labor starts. It also helps families make fast, confident decisions if sudden health complications arise."
},

    {
  "id": "DOC_BP_126",
  "domain": "maternal_health",
  "topic": "birth_preparedness",
  "subtopic": "health_facility_transport",
  "source": ["World Health Organization, Kenyatta University Hospital"],
  "year": 2022,
  "url": ["https://www.who.int/tools/your-life-your-health/life-phase/pregnancy--birth-and-after-childbirth/making-plans-for-childbirth-when-pregnant",
          "https://ir-library.ku.ac.ke/server/api/core/bitstreams/b447bb8e-3ab1-414c-bd1b-cdc7a1af9c6d/content"],
  "content": "A complete birth preparedness plan includes choosing a specific health facility for delivery and arranging reliable transportation ahead of time. Identifying a preferred health center and organizing emergency travel options early ensures a mother can travel quickly when labor starts or if urgent medical complications develop."
},

    {
  "id": "DOC_BP_127",
  "domain": "maternal_health",
  "topic": "birth_preparedness",
  "subtopic": "financial_preparation",
  "source": ["World Health Organization, Kenyatta University Hospital"],
  "year": 2022,
  "url": ["https://www.who.int/tools/your-life-your-health/life-phase/pregnancy--birth-and-after-childbirth/making-plans-for-childbirth-when-pregnant",
          "https://ir-library.ku.ac.ke/server/api/core/bitstreams/b447bb8e-3ab1-414c-bd1b-cdc7a1af9c6d/content"],
  "content": "Families should plan early for the financial expenses of childbirth and unexpected medical emergencies. Setting aside saved money in advance prevents dangerous delays in getting care due to a lack of immediate funds. Financial readiness is an essential part of ensuring a safe delivery and quick emergency response."
},

    {
  "id": "DOC_BP_128",
  "domain": "maternal_health",
  "topic": "birth_preparedness",
  "subtopic": "skilled_birth_attendant",
  "source": ["World Health Organization, Kenyatta University Hospital"],
  "year": 2022,
  "url": ["https://www.who.int/tools/your-life-your-health/life-phase/pregnancy--birth-and-after-childbirth/making-plans-for-childbirth-when-pregnant",
          "https://ir-library.ku.ac.ke/server/api/core/bitstreams/b447bb8e-3ab1-414c-bd1b-cdc7a1af9c6d/content"],
  "content": "Pregnant women should identify a skilled health worker such as a doctor, midwife, or nurse to assist during childbirth. Having a qualified healthcare professional present during delivery significantly increases the chances of spotting complications early and providing life-saving treatment."
},

    {
  "id": "DOC_BP_129",
  "domain": "maternal_health",
  "topic": "birth_preparedness",
  "subtopic": "support_system",
  "source": ["World Health Organization, Kenyatta University Hospital"],
  "year": 2022,
  "url": ["https://www.who.int/tools/your-life-your-health/life-phase/pregnancy--birth-and-after-childbirth/making-plans-for-childbirth-when-pregnant",
          "https://ir-library.ku.ac.ke/server/api/core/bitstreams/b447bb8e-3ab1-414c-bd1b-cdc7a1af9c6d/content"],
  "content": "Every pregnant woman should choose trusted family members or friends who can accompany her and offer physical and emotional help during labor and delivery. Having a supportive birth companion present reduces stress, improves comfort, and helps advocate for the mother's needs during childbirth."
},

    {
  "id": "DOC_BP_130",
  "domain": "maternal_health",
  "topic": "birth_preparedness",
  "subtopic": "household_support",
  "source": "Kenyatta University Hospital",
  "year": 2014,
  "url": "https://ir-library.ku.ac.ke/server/api/core/bitstreams/b447bb8e-3ab1-414c-bd1b-cdc7a1af9c6d/content",
  "content": "A thorough birth plan includes arranging for family members or trusted neighbors to take care of household chores, cooking, and child care while the mother is at the hospital. Families should also designate a trusted decision maker who can step in immediately to authorize medical care or travel if labor starts suddenly or complications occur."
},

    {
  "id": "DOC_BP_131",
  "domain": "maternal_health",
  "topic": "birth_preparedness",
  "subtopic": "emergency_preparedness",
  "source": "Kenyatta University Hospital",
  "year": 2014,
  "url": "https://ir-library.ku.ac.ke/server/api/core/bitstreams/b447bb8e-3ab1-414c-bd1b-cdc7a1af9c6d/content",
  "content": "To prepare for sudden medical emergencies, families should identify potential compatible blood donors in advance and locate the nearest referral health facility capable of managing severe childbirth complications. Making these emergency arrangements early prevents critical delays in getting life-saving care when urgent complications arise."
},

    {
  "id": "DOC_BP_132",
  "domain": "maternal_health",
  "topic": "birth_preparedness",
  "subtopic": "childbirth_supplies",
  "source": "World Health Organization",
  "year": 2022,
  "url": "https://www.who.int/tools/your-life-your-health/life-phase/pregnancy--birth-and-after-childbirth/making-plans-for-childbirth-when-pregnant",
  "content": "Pregnant women should gather essential childbirth supplies ahead of time, including clean clothes and soft blankets for drying, warming, and wrapping the newborn baby. Having these basic items ready before labor begins promotes good hygiene and ensures proper immediate care right after birth."
},

    {
  "id": "DOC_BP_133",
  "domain": "maternal_health",
  "topic": "birth_preparedness",
  "subtopic": "plan_review",
  "source": "Kenyatta University Hospital",
  "year": 2014,
  "url": "https://ir-library.ku.ac.ke/server/api/core/bitstreams/b447bb8e-3ab1-414c-bd1b-cdc7a1af9c6d/content",
  "content": "Pregnant women should discuss and review their birth preparedness plan with a midwife or healthcare provider during regular antenatal clinic visits. Reviewing and updating the plan as the expected due date gets closer ensures that all arrangements for travel, funds, and emergency contacts remain clear and ready."
},

    #Post natal warning signs
    {
  "id": "DOC_POST_134",
  "domain": "maternal_health",
  "topic": "postnatal_care",
  "subtopic": "postnatal_overview",
  "source": ["World Health Organization", "National Library of Medicine"],
  "year": 2013,
  "url": "https://www.ncbi.nlm.nih.gov/books/NBK304191/",
  "content": "Mothers and their families must learn to recognize danger signs during the postnatal period because many life threatening complications happen within the first week after giving birth. Spotting these warning signs early and getting immediate medical care at a hospital can prevent severe illness and save lives."
},

    {
  "id": "DOC_POST_135",
  "domain": "maternal_health",
  "topic": "postnatal_care",
  "subtopic": "postnatal_bleeding",
  "source": ["World Health Organization", "National Library of Medicine"],
  "year": 2013,
  "url": "https://www.ncbi.nlm.nih.gov/books/NBK304191/",
  "content": "Heavy vaginal bleeding after childbirth is a life-threatening emergency that can indicate postpartum hemorrhage. Mothers who soak through more than two sanitary towels in an hour, pass large blood clots, or experience sudden severe bleeding must go to a health facility immediately for emergency medical care."
},

    {
  "id": "DOC_POST_136",
  "domain": "maternal_health",
  "topic": "postnatal_care",
  "subtopic": "postnatal_convulsions",
  "source": ["World Health Organization", "National Library of Medicine"],
  "year": 2013,
  "url": "https://www.ncbi.nlm.nih.gov/books/NBK304191/",
  "content": "Convulsions, seizures, or fits after childbirth are critical danger signs that often indicate severe high blood pressure complications known as eclampsia. Sudden fits after delivery can quickly become life-threatening and require emergency transport to a hospital for immediate medical treatment."
},

    {
  "id": "DOC_POST_137",
  "domain": "maternal_health",
  "topic": "postnatal_care",
  "subtopic": "postnatal_breathing",
  "source": ["World Health Organization", "National Library of Medicine"],
  "year": 2013,
  "url": "https://www.ncbi.nlm.nih.gov/books/NBK304191/",
  "content": "Fast or difficult breathing, chest pain, or painful swelling and redness in the legs after childbirth are serious danger signs that may point to severe chest infections or blood clots. Any mother who experiences shortness of breath, sudden chest discomfort, or leg pain after giving birth should seek emergency medical care immediately."
},

    {
  "id": "DOC_POST_138",
  "domain": "maternal_health",
  "topic": "postnatal_care",
  "subtopic": "postnatal_headache_fever",
  "source": ["World Health Organization", "National Library of Medicine"],
  "year": 2013,
  "url": "https://www.ncbi.nlm.nih.gov/books/NBK304191/",
  "content": "Severe headaches accompanied by blurred vision, as well as a high fever with extreme weakness or chills, are critical warning signs after giving birth. These symptoms can indicate serious postpartum hypertension or dangerous uterine and systemic infections, requiring urgent medical evaluation and treatment at a health facility."
},

    {
  "id": "DOC_POST_139",
  "domain": "maternal_health",
  "topic": "postnatal_care",
  "subtopic": "postnatal_breast_issues",
  "source": ["World Health Organization", "National Library of Medicine"],
  "year": 2013,
  "url": "https://www.ncbi.nlm.nih.gov/books/NBK304191/",
  "content": "Swollen, tender, unusually warm, or painful breasts and cracked or red nipples after childbirth often signal breastfeeding complications or localized breast infections like mastitis. Women experiencing persistent breast soreness, fever, or painful lumps should be evaluated promptly by a healthcare provider for safe treatment and breastfeeding support."
},

    {
  "id": "DOC_POST_140",
  "domain": "maternal_health",
  "topic": "postnatal_care",
  "subtopic": "postnatal_urinary_perineum",
  "source": ["World Health Organization", "National Library of Medicine"],
  "year": 2013,
  "url": "https://www.ncbi.nlm.nih.gov/books/NBK304191/",
  "content": "Pain or difficulty when urinating, inability to control urine, worsening pain in the perineum area, or red, swollen, and leaking delivery wounds are key warning signs after childbirth. These symptoms can indicate urinary tract infections or infected tear and incision wounds that require prompt medical evaluation and treatment to prevent further health complications."
},

    {
  "id": "DOC_POST_141",
  "domain": "maternal_health",
  "topic": "postnatal_care",
  "subtopic": "postnatal_urinary_perineum",
  "source": ["World Health Organization", "National Library of Medicine"],
  "year": 2013,
  "url": "https://www.ncbi.nlm.nih.gov/books/NBK304191/",
  "content": "Burning sensation during urination, inability to pass urine, severe perineal soreness, or foul smelling discharge from surgical incisions and vaginal tears are major postnatal warning signs. Leaving these symptoms untreated can cause severe tissue infections or kidney complications, making immediate clinical examination and treatment essential."
},

    {
  "id": "DOC_POST_142",
  "domain": "maternal_health",
  "topic": "postnatal_care",
  "subtopic": "postnatal_blues",
  "source": ["World Health Organization", "National Library of Medicine"],
  "year": 2013,
  "url": "https://www.ncbi.nlm.nih.gov/books/NBK304191/",
  "content": "Many mothers experience mild emotional shifts shortly after childbirth known as postnatal blues or baby blues. Symptoms include sudden mood swings, crying spells, anxiety, fatigue, and sleep disruptions, which typically peak within the first few days and resolve naturally within two weeks with rest and family support."
},

    {
  "id": "DOC_POST_143",
  "domain": "maternal_health",
  "topic": "postnatal_care",
  "subtopic": "postnatal_depression",
  "source": ["World Health Organization", "National Library of Medicine"],
  "year": 2013,
  "url": "https://www.ncbi.nlm.nih.gov/books/NBK304191/",
  "content": "Persistent sadness, inability to bond with the baby, overwhelming anxiety, loss of interest in daily life, or recurring thoughts of harming oneself or the infant are serious signs of postnatal depression. Unlike temporary baby blues, postpartum depression lasts longer than two weeks and requires professional medical care, counseling, and strong social support."
},

    #Labour and delivery warning signs
    {
  "id": "DOC_LAB_144",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "early_labour_overview",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "Signs that labor is approaching vary from person to person and often develop gradually. Common early physical indicators show that the cervix and body are preparing for birth, even though active labor and real contractions might still be days or weeks away."
},

    {
  "id": "DOC_LAB_145",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "labour_cramps_back_pain",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "Mild cramping similar to period cramps is a common early indicator that labor is drawing near. These cramps happen as the cervix softens, thins, and prepares to open for delivery. Persistent lower back ache or pressure in the lower back also occurs as the baby drops deeper into the pelvic region."
},

    {
  "id": "DOC_LAB_146",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "pelvic_pressure_lightening",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "Pelvic pressure often increases in the final weeks of pregnancy when the baby drops lower into the pelvis, a process called lightening. While this shift relieves pressure on the diaphragm and makes breathing easier, it puts extra weight on the bladder and hips, causing lower abdominal discomfort and a frequent urge to urinate."
},

    {
  "id": "DOC_LAB_147",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "mucus_plug_discharge",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "Losing the mucus plug is a normal sign that labor is near. As the cervix thins and starts to open, mothers may notice a thick, clear, pinkish, or slightly bloody vaginal discharge, showing that childbirth preparations are underway."
},

    {
  "id": "DOC_LAB_148",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "fatigue_insomnia_labour",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "Fatigue or sudden exhaustion is common in the final days before labour. Some women may also experience insomnia due to discomfort, hormonal changes or anticipation of childbirth."
},

    {
  "id": "DOC_LAB_149",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "digestive_changes_labour",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "Digestive shifts like loose stools, mild diarrhea, nausea, or a sudden decrease in appetite often occur right before labor starts. These natural changes are caused by birth hormones relaxing muscles and clearing the digestive tract to make space for delivery."
},

    {
  "id": "DOC_LAB_150",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "hormonal_physical_changes",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "Hormonal changes right before labor cause increased joint flexibility and occasional feelings of unsteadiness or clumsiness. The body produces higher levels of a hormone called relaxin, which relaxes pelvic ligaments and loosens joints to help the birth canal expand during delivery."
},

    {
  "id": "DOC_LAB_151",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "nesting_energy",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "Many mothers experience a sudden burst of energy shortly before labor begins, a natural urge known as the nesting instinct. This sudden surge in motivation often prompts expectant mothers to clean, organize, cook, and finish preparing the home for the arrival of the baby."
},

    {
  "id": "DOC_LAB_152",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "pelvic_nerve_pain",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "Sharp, sudden pelvic pain or shooting nerve twinges, often referred to as lightning crotch, can happen when the baby drops lower and presses against sensitive nerves in the pelvis. While uncomfortable, these brief sharp sensations are a normal indicator that the baby is moving into position for birth."
},

    {
  "id": "DOC_LAB_153",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "braxton_hicks",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "Braxton Hicks contractions are irregular practice contractions that tighten the uterus to help prepare the body for childbirth. Unlike true labor contractions, Braxton Hicks do not follow a regular timing pattern, do not grow stronger or closer together over time, and usually ease when changing positions, walking, or resting."
},

    {
  "id": "DOC_LAB_154",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "active_labour_overview",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "Active labor is the phase of childbirth where contractions become strong, regular, and progressively closer together while the cervix opens significantly. Identifying the transition into active labor is crucial so expectant mothers can head to a healthcare facility or contact their birth team for timely, safe delivery support."
},

    {
  "id": "DOC_LAB_155",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "labour_contractions",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "True labor contractions become progressively stronger, longer, and closer together in a predictable pattern. Unlike false labor, true contractions continue to intensify regardless of walking, changing body positions, drinking water, or resting, indicating that active labor is underway."
},

    {
  "id": "DOC_LAB_156",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "water_breaking",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "Water breaking happens when the fluid filled amniotic sac surrounding the baby ruptures, causing a trickling flow or sudden gush of clear fluid from the vagina. This is a clear indicator that labor is starting or about to begin, requiring immediate contact with a doctor, midwife, or healthcare facility."
},

    {
  "id": "DOC_LAB_157",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "urge_to_push",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "As labor advances, mothers often experience an intense, involuntary reflex and powerful pressure or urge to push as the baby moves down into the birth canal. This strong sensation signals that the cervix is fully dilated and that the delivery phase of childbirth is starting."
},

    {
  "id": "DOC_LAB_158",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "intense_pelvic_pressure",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "Intense pelvic pressure and a heavy downward sensation occur when the baby moves deeper into the pelvic structure and birth canal. This increasing weight and pressure confirm that labor is actively advancing toward final delivery."
},

    {
  "id": "DOC_LAB_159",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "labour_emergency_signs",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "Certain symptoms during labour require immediate medical attention. Seek urgent help at a hospital or health facility if you experience chest pain, severe lightheadedness, fainting, continuous nausea, or sudden extreme swelling in the face, hands, arms, or legs."
},

    {
  "id": "DOC_LAB_160",
  "domain": "maternal_health",
  "topic": "labour_and_delivery",
  "subtopic": "labour_critical_signs",
  "source": "The Mother Baby Center",
  "year": 2025,
  "url": "https://www.themotherbabycenter.org/blog/2025/08/signs-labor-is-near/",
  "content": "Heavy vaginal bleeding similar to a menstrual period or a noticeable drop in the baby movement are critical emergency warning signs. Immediate medical attention at the nearest hospital or health center is required to prevent severe complications for both mother and child."
},

    # IPV Screening
    {
  "id": "DOC_IPV_161",
  "domain": "maternal_health",
  "topic": "intimate_partner_violence",
  "subtopic": "definition_and_prevalence_scope",
  "source": ["American College of Obstetricians and Gynecologists", "National Library of Medicine"],
  "year": 2012,
  "url": ["https://www.acog.org/clinical/clinical-guidance/committee-opinion/articles/2012/02/intimate-partner-violence"
          "https://pmc.ncbi.nlm.nih.gov/articles/PMC6067906/"],
  "content": "Intimate partner violence is a pattern of assaultive and coercive behavior including physical injury psychological abuse sexual assault progressive isolation stalking and reproductive coercion perpetrated to establish control over a partner. Globally 30 percent of women experience physical or sexual violence in their lifetime. Homicide by a current or former intimate partner remains a leading cause of maternal mortality."
},

    {
  "id": "DOC_IPV_162",
  "domain": "maternal_health",
  "topic": "intimate_partner_violence",
  "subtopic": "reproductive_coercion_and_sti_risks",
  "source": ["American College of Obstetricians and Gynecologists", "National Library of Medicine"],
  "year": 2012,
  "url": ["https://www.acog.org/clinical/clinical-guidance/committee-opinion/articles/2012/02/intimate-partner-violence"
          "https://pmc.ncbi.nlm.nih.gov/articles/PMC6067906/"],
  "content": "Reproductive coercion involves explicit behaviors to control pregnancy outcomes including birth control sabotage refusing safe sex intentional exposure to sexually transmitted infections and forcing or forbidding abortion. Approximately 20 percent of abused women seeking family planning care report pregnancy coercion and 15 percent report birth control sabotage. Expedited partner therapy for sexually transmitted infections should only be offered after confirming there is no risk of partner violence."
},

    {
  "id": "DOC_IPV_163",
  "domain": "maternal_health",
  "topic": "intimate_partner_violence",
  "subtopic": "perinatal_and_maternal_health_outcomes",
  "source": ["American College of Obstetricians and Gynecologists", "National Library of Medicine"],
  "year": 2012,
  "url": ["https://www.acog.org/clinical/clinical-guidance/committee-opinion/articles/2012/02/intimate-partner-violence"
          "https://pmc.ncbi.nlm.nih.gov/articles/PMC6067906/"],
  "content": "Intimate partner violence during pregnancy affects over 300000 women globally and is associated with adverse outcomes such as preterm delivery low birth weight intrauterine growth restriction stillbirth placental abruption and neonatal intensive care admission. Maternal complications include gestational hypertension severe nausea vaginal bleeding urinary tract infections chronic pelvic pain and severe depression."
},

    {
  "id": "DOC_IPV_164",
  "domain": "maternal_health",
  "topic": "intimate_partner_violence",
  "subtopic": "special_populations_disabilities_and_elderly",
  "source": ["American College of Obstetricians and Gynecologists", "National Library of Medicine"],
  "year": 2012,
  "url": ["https://www.acog.org/clinical/clinical-guidance/committee-opinion/articles/2012/02/intimate-partner-violence"
          "https://pmc.ncbi.nlm.nih.gov/articles/PMC6067906/"],
  "content": "Abuse against women with disabilities often involves caregivers withholding essential medications preventing access to wheelchairs or canes and refusing assistance with personal hygiene or eating. Elder abuse affects up to 2 million older adults where perpetrators are adult children or partners who inflict physical emotional or financial exploitation."
},

    {
  "id": "DOC_IPV_165",
  "domain": "maternal_health",
  "topic": "intimate_partner_violence",
  "subtopic": "clinical_screening_barriers_and_protocols",
  "source": ["American College of Obstetricians and Gynecologists", "National Library of Medicine"],
  "year": 2012,
  "url": ["https://www.acog.org/clinical/clinical-guidance/committee-opinion/articles/2012/02/intimate-partner-violence"
          "https://pmc.ncbi.nlm.nih.gov/articles/PMC6067906/"],
  "content": "Only 9 to 40 percent of clinicians routinely screen for relationship violence due to discomfort, lack of time training or privacy fear of offending patients and uncertainty about handling disclosures. Establishing a standardized systematic screening protocol with clear roles and tools helps overcome these barriers across clinical settings."
},

    {
  "id": "DOC_IPV_166",
  "domain": "maternal_health",
  "topic": "intimate_partner_violence",
  "subtopic": "universal_screening_timing_and_frequency",
  "source": ["American College of Obstetricians and Gynecologists", "National Library of Medicine"],
  "year": 2012,
  "url": ["https://www.acog.org/clinical/clinical-guidance/committee-opinion/articles/2012/02/intimate-partner-violence"
          "https://pmc.ncbi.nlm.nih.gov/articles/PMC6067906/"],
  "content": "Clinicians should routinely screen all women of reproductive age at initial consultations well-woman visits and annual exams. Obstetric screening must occur at the first prenatal visit at least once during each trimester and at the postpartum checkup because abuse may begin or escalate during pregnancy."
},

    {
  "id": "DOC_IPV_167",
  "domain": "maternal_health",
  "topic": "intimate_partner_violence",
  "subtopic": "privacy_controls_and_interpreter_policy",
  "source": ["American College of Obstetricians and Gynecologists", "National Library of Medicine"],
  "year": 2012,
  "url": ["https://www.acog.org/clinical/clinical-guidance/committee-opinion/articles/2012/02/intimate-partner-violence"
          "https://pmc.ncbi.nlm.nih.gov/articles/PMC6067906/"],
  "content": "Screening must occur privately without partners family members or friends present. Normalizing privacy as standard clinic policy or conducting screens during private bathroom visits preserves confidentiality. Trained professional medical interpreters must be used instead of family members to protect patient safety and ensure exact communication."
},

    {
  "id": "DOC_IPV_168",
  "domain": "maternal_health",
  "topic": "intimate_partner_violence",
  "subtopic": "scripting_framing_and_language_tactics",
  "source": ["American College of Obstetricians and Gynecologists", "National Library of Medicine"],
  "year": 2012,
  "url": ["https://www.acog.org/clinical/clinical-guidance/committee-opinion/articles/2012/02/intimate-partner-violence"
          "https://pmc.ncbi.nlm.nih.gov/articles/PMC6067906/"],
  "content": "Screening should begin with a normalizing framing statement reassuring patients that safety questions are standard for everyone. Clinicians must avoid stigmatizing terms like victim, battered, rape or abuse and instead use behaviorally specific language like hit, slapped, pushed or forced into sexual activity."
},

    {
  "id": "DOC_IPV_169",
  "domain": "maternal_health",
  "topic": "intimate_partner_violence",
  "subtopic": "validated_screening_tools_comparison",
  "source": ["American College of Obstetricians and Gynecologists", "National Library of Medicine"],
  "year": 2012,
  "url": ["https://www.acog.org/clinical/clinical-guidance/committee-opinion/articles/2012/02/intimate-partner-violence"
          "https://pmc.ncbi.nlm.nih.gov/articles/PMC6067906/"],
  "content": "Single vague questions like asking if a patient feels safe at home have low sensitivity and are insufficient for screening. Clinicians should use validated multi-item tools such as the Abuse Assessment Screen HARK Partner Violence Screen or STaT tool where any single positive response indicates relationship violence."
},

    {
  "id": "DOC_IPV_170",
  "domain": "maternal_health",
  "topic": "intimate_partner_violence",
  "subtopic": "mandatory_reporting_transparency",
  "source": ["American College of Obstetricians and Gynecologists", "National Library of Medicine"],
  "year": 2012,
  "url": ["https://www.acog.org/clinical/clinical-guidance/committee-opinion/articles/2012/02/intimate-partner-violence"
          "https://pmc.ncbi.nlm.nih.gov/articles/PMC6067906/"],
  "content": "Mandatory reporting laws vary by state and jurisdiction. In regions where healthcare workers must report relationship injuries to police providers must clearly explain these legal obligations before asking screening questions to preserve patient trust autonomy and informed choice."
},

    {
  "id": "DOC_IPV_171",
  "domain": "maternal_health",
  "topic": "intimate_partner_violence",
  "subtopic": "therapeutic_communication_environment",
  "source": ["American College of Obstetricians and Gynecologists", "National Library of Medicine"],
  "year": 2012,
  "url": ["https://www.acog.org/clinical/clinical-guidance/committee-opinion/articles/2012/02/intimate-partner-violence"
          "https://pmc.ncbi.nlm.nih.gov/articles/PMC6067906/"],
  "content": "Therapeutic interaction requires sitting at eye level maintaining warm eye contact and avoiding screen distractions during assessment. Invasions of personal space can trigger post-traumatic stress symptoms whereas empathetic listening builds the clinical trust required for disclosure."
},

    {
  "id": "DOC_IPV_172",
  "domain": "maternal_health",
  "topic": "intimate_partner_violence",
  "subtopic": "empowering_immediate_disclosure_response",
  "source": ["American College of Obstetricians and Gynecologists", "National Library of Medicine"],
  "year": 2012,
  "url": ["https://www.acog.org/clinical/clinical-guidance/committee-opinion/articles/2012/02/intimate-partner-violence"
          "https://pmc.ncbi.nlm.nih.gov/articles/PMC6067906/"],
  "content": "When a patient discloses abuse immediate responses should validate her experience with supportive statements like I believe you and this is not your fault. Clinicians must avoid forcing police contact or making unrequested choices which can heighten fear and hinder patient autonomy."
},

    {
  "id": "DOC_IPV_173",
  "domain": "maternal_health",
  "topic": "intimate_partner_violence",
  "subtopic": "lethality_assessment_and_danger_factors",
  "source": ["American College of Obstetricians and Gynecologists", "National Library of Medicine"],
  "year": 2012,
  "url": ["https://www.acog.org/clinical/clinical-guidance/committee-opinion/articles/2012/02/intimate-partner-violence"
          "https://pmc.ncbi.nlm.nih.gov/articles/PMC6067906/"],
  "content": "Because survivors often underestimate physical danger, clinicians should perform a lethality assessment using tools like the Danger Assessment. Key risk factors for homicide include partner access to guns, threats with weapons, history of nonfatal strangulation, life threats, estrangement, and severe jealousy."
},

    {
  "id": "DOC_IPV_174",
  "domain": "maternal_health",
  "topic": "intimate_partner_violence",
  "subtopic": "safety_planning_and_discreet_resources",
  "source": ["American College of Obstetricians and Gynecologists", "National Library of Medicine"],
  "year": 2012,
  "url": ["https://www.acog.org/clinical/clinical-guidance/committee-opinion/articles/2012/02/intimate-partner-violence"
          "https://pmc.ncbi.nlm.nih.gov/articles/PMC6067906/"],
  "content": "Leaving an abusive partner without a structured safety plan sharply increases immediate physical danger. Key safety strategies include securing copies of vital identification documents, hiding emergency exit funds in a safe location, establishing secret code words with trusted contacts, and placing educational safety cards in private clinic areas like restrooms so women can access support without fear of partner discovery."
},

    {
  "id": "DOC_IPV_175",
  "domain": "maternal_health",
  "topic": "intimate_partner_violence",
  "subtopic": "warm_referrals_and_documentation",
  "source": ["American College of Obstetricians and Gynecologists", "National Library of Medicine"],
  "year": 2012,
  "url": ["https://www.acog.org/clinical/clinical-guidance/committee-opinion/articles/2012/02/intimate-partner-violence"
          "https://pmc.ncbi.nlm.nih.gov/articles/PMC6067906/"],
  "content": "Effective referrals involve offering warm, direct phone connections or sitting alongside patients during calls to crisis hotlines, using a private clinic phone to prevent digital monitoring by controlling partners. Clinical documentation must detail physical injuries accurately using body maps, direct patient quotations, and pre-arranged safe communication methods to protect survivor safety in legal or medical follow-ups."
},

    # High Risk Pregnancy Factors
    {
  "id": "DOC_HRP_176",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "definition_age_and_past_history_factors",
  "source": ["Perth Obstetrics and Gynaecology Specialists", "National Institute of Child Health Development"],
  "year": 2025,
  "url": ["https://www.pogs.com.au/high-risk-pregnancy"
          "https://www.nichd.nih.gov/health/topics/high-risk/conditioninfo/factors"],
  "content": "A high-risk pregnancy poses potential health threats to the mother or baby before, during, or after delivery, requiring regular monitoring by an obstetrician or perinatologist in specialized care centers. Age-related risks occur in young adolescents under 15 years, who face higher rates of anemia, hypertension, preterm labor, and undetected infections, as well as first-time mothers over 35 to 38 years, who experience higher rates of gestational diabetes, preeclampsia, ectopic pregnancy, labor dystocia, cesarean delivery, and chromosomal anomalies such as Down syndrome. Additional maternal risk factors include pre-pregnancy weight under 100 pounds, obesity, height under 5 feet, grand multiparity over five pregnancies, and a history of difficult deliveries, rapid labor, or prior C-sections."
},

    {
  "id": "DOC_HRP_177",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "chronic_maternal_medical_conditions",
  "source": ["Perth Obstetrics and Gynaecology Specialists", "National Institute of Child Health Development"],
  "year": 2025,
  "url": ["https://www.pogs.com.au/high-risk-pregnancy"
          "https://www.nichd.nih.gov/health/topics/high-risk/conditioninfo/factors"],
  "content": "Health conditions that a woman has before getting pregnant can make a pregnancy high risk for both mother and baby. High blood pressure before pregnancy can damage the mother's kidneys and cause low birth weight or severe high blood pressure during pregnancy. Having diabetes before pregnancy means blood sugar must be carefully controlled before and during pregnancy to prevent birth defects, unusually large babies, and low blood sugar in the newborn. Polycystic ovary syndrome (PCOS) increases the risk of miscarriage before 20 weeks, pregnancy diabetes, and preeclampsia. Kidney disease makes it harder to get pregnant and raises the chance of early delivery or preeclampsia—in fact, nearly 1 in 5 women who get early preeclampsia actually have hidden kidney disease. Autoimmune diseases like lupus increase the risk of early labor and stillbirth, meaning doctors must carefully manage the mother's treatment so her medications do not harm the developing baby."
},

    {
  "id": "DOC_HRP_178",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "lifestyle_factors_and_infections",
  "source": ["Perth Obstetrics and Gynaecology Specialists", "National Institute of Child Health Development"],
  "year": 2025,
  "url": ["https://www.pogs.com.au/high-risk-pregnancy"
          "https://www.nichd.nih.gov/health/topics/high-risk/conditioninfo/factors"],
  "content": "Lifestyle choices and certain infections during pregnancy can seriously affect the health of the baby. Drinking alcohol while pregnant can lead to fetal alcohol spectrum disorders, causing long-term learning disabilities, behavior problems, and facial abnormalities; there is no safe amount of alcohol during pregnancy. Smoking cigarettes doubles or triples the risk of stillbirth and causes premature birth, low birth weight, and sudden infant death syndrome (SIDS). Using illegal drugs or marijuana can double the risk of stillbirth and harm normal brain development in the fetus. Infections like HIV can pass from mother to baby during pregnancy, birth, or breastfeeding, but taking proper medication, having a C-section before the water breaks, and formula feeding reduces this risk to less than 1%. Getting infected with the Zika virus during pregnancy can cause severe brain damage in the baby, including microcephaly (an abnormally small head), as well as stillbirth."
},

    {
  "id": "DOC_HRP_179",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "pregnancy_specific_conditions",
  "source": ["Perth Obstetrics and Gynaecology Specialists", "National Institute of Child Health Development"],
  "year": 2025,
  "url": ["https://www.pogs.com.au/high-risk-pregnancy"
          "https://www.nichd.nih.gov/health/topics/high-risk/conditioninfo/factors"],
  "content": "Some complications develop only after a woman becomes pregnant. Gestational diabetes is a type of high blood sugar that starts during pregnancy; it can cause the baby to grow too large, lead to early delivery, and raise the mother's long-term risk for type 2 diabetes. Preeclampsia is a serious condition after 20 weeks of pregnancy marked by sudden high blood pressure and signs of damage to organs like the liver or kidneys, which can progress to life-threatening seizures (eclampsia). Carrying twins, triplets, or more increases the chance of early birth, low birth weight, and needing a C-section. Having a history of early labor in a past pregnancy increases the risk of it happening again, though treatments like progesterone shots can help delay birth. When birth defects or genetic conditions are found before birth, medical teams can prepare specialized care, surgery, or treatment immediately after the baby is born."
},

    {
  "id": "DOC_HRP_180",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "diagnostic_testing_and_fetal_monitoring",
  "source": ["Perth Obstetrics and Gynaecology Specialists", "National Institute of Child Health Development"],
  "year": 2025,
  "url": ["https://www.pogs.com.au/high-risk-pregnancy"
          "https://www.nichd.nih.gov/health/topics/high-risk/conditioninfo/factors"],
  "content": "Specialized tests help doctors check the health of the mother and the developing baby throughout a high-risk pregnancy. Ultrasound scans track fetal growth, check the placenta, measure cervical length to predict early labor, and look for physical abnormalities. Genetic tests include chorionic villus sampling, which takes a small sample of placenta cells between weeks 10 and 12, and amniocentesis, which tests a sample of fluid from around the baby after 15 weeks. Blood tests, such as cordocentesis after week 18, check for infections, blood disorders, and chromosome changes. Doctors also use a biophysical profile, which combines ultrasound monitoring with fetal heart rate checks, and vaginal fluid tests to detect signs of early labor."
},

    {
  "id": "DOC_HRP_181",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "clinical_management_and_prevention_strategies",
  "source": ["Perth Obstetrics and Gynaecology Specialists", "National Institute of Child Health Development"],
  "year": 2025,
  "url": ["https://www.pogs.com.au/high-risk-pregnancy"
          "https://www.nichd.nih.gov/health/topics/high-risk/conditioninfo/factors"],
  "content": "Managing a high-risk pregnancy involves frequent checkups, tailored treatment plans, and self-care. Women with high-risk factors often receive care from a perinatologist, a specialist doctor in maternal-fetal medicine, in centers equipped with neonatal intensive care units. Treatment strategies include taking prescribed medications, making diet changes, getting enough rest, and monitoring blood pressure or blood sugar at home. Expectant mothers should eat nutritious foods rich in folic acid, calcium, and iron, maintain recommended weight gain, avoid alcohol, smoking, and recreational drugs, and limit strenuous activities when advised. Immediate medical care is necessary if warning signs appear, such as vaginal bleeding, lower belly pain or severe cramps, watery discharge, reduced baby movement, severe headaches, or sudden vision changes."
},

    {
  "id": "DOC_HRP_182",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "thyroid_disorders_in_pregnancy",
  "source": ["Perth Obstetrics and Gynaecology Specialists", "National Institute of Child Health Development"],
  "year": 2025,
  "url": ["https://www.pogs.com.au/high-risk-pregnancy"
          "https://www.nichd.nih.gov/health/topics/high-risk/conditioninfo/factors"],
  "content": "Thyroid disorders affect metabolic health and can develop before, during, or shortly after pregnancy. Thyroid hormones are essential for normal fetal brain and nervous system development, especially during the first trimester when the baby depends entirely on maternal thyroid supply. Overactive thyroid, often caused by Graves disease, can lead to fast heart rates, poor weight gain, and pregnancy complications. Underactive thyroid, frequently caused by Hashimotos disease or inadequate treatment, can impair fetal cognitive development if left untreated. After delivery, some women experience postpartum thyroiditis, an inflammation of the thyroid gland that causes temporary overactive thyroid followed by underactive thyroid, which may sometimes require long term treatment."
},

    {
  "id": "DOC_HRP_183",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "anemia_and_nutrient_deficiencies",
  "source": ["Perth Obstetrics and Gynaecology Specialists", "National Institute of Child Health Development"],
  "year": 2025,
  "url": ["https://www.pogs.com.au/high-risk-pregnancy"
          "https://www.nichd.nih.gov/health/topics/high-risk/conditioninfo/factors"],
  "content": "Anemia occurs when the blood has a reduced capacity to carry oxygen throughout the body. During pregnancy, blood volume increases significantly, raising the need for essential nutrients to produce healthy red blood cells. The three main types are iron deficiency anemia, folate deficiency anemia, and vitamin B12 deficiency anemia. Common symptoms include persistent fatigue, weakness, dizziness, shortness of breath, rapid heartbeat, and pale skin, lips, or nails. If left untreated, severe anemia increases the risk of premature birth and low birth weight. Prevention and treatment involve daily prenatal supplements, eating iron rich foods like green leafy vegetables and lean meats, and pairing iron sources with vitamin C rich foods to boost absorption."
},

    {
  "id": "DOC_HRP_184",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "recurrent_miscarriage_causes_and_testing",
  "source": ["Perth Obstetrics and Gynaecology Specialists", "National Institute of Child Health Development"],
  "year": 2025,
  "url": ["https://www.pogs.com.au/high-risk-pregnancy"
          "https://www.nichd.nih.gov/health/topics/high-risk/conditioninfo/factors"],
  "content": "Recurrent miscarriage is defined as having two or more consecutive pregnancy losses before twenty weeks of gestation. Potential causes include genetic or chromosomal abnormalities in either parent, structural uterine issues such as fibroids, hormonal imbalances like polycystic ovary syndrome, and immune conditions like antiphospholipid syndrome that cause abnormal blood clotting. Diagnostic evaluations involve detailed genetic karyotyping for both partners, uterine imaging scans, hormone checks, thyroid testing, and antibody screening. Treatment is tailored to the underlying cause, ranging from genetic counseling and hormonal therapy to blood thinning medications that help support a safe and healthy subsequent pregnancy."
},

    {
  "id": "DOC_HRP_185",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "advanced_maternal_age_and_labor_complications",
  "source": ["Perth Obstetrics and Gynaecology Specialists", "National Institute of Child Health Development"],
  "year": 2025,
  "url": ["https://www.pogs.com.au/high-risk-pregnancy"
          "https://www.nichd.nih.gov/health/topics/high-risk/conditioninfo/factors"],
  "content": "Pregnancy in women in their late thirties or early forties brings unique health considerations for both mother and baby. As maternal age increases, there is a higher chance of chromosomal variations such as Down syndrome, which can be identified early through first trimester screening, blood tests, and ultrasound scans. Older mothers also face higher risks of pre existing high blood pressure, gestational diabetes, ectopic pregnancy, and miscarriage. During labor, older women are more likely to experience prolonged labor stages, which can increase the need for assisted delivery methods like vacuum, forceps, or a planned cesarean section."
},

    {
  "id": "DOC_HRP_186",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "twin_pregnancies_and_specialized_care",
  "source": ["Perth Obstetrics and Gynaecology Specialists", "National Institute of Child Health Development"],
  "year": 2025,
  "url": ["https://www.pogs.com.au/high-risk-pregnancy"
          "https://www.nichd.nih.gov/health/topics/high-risk/conditioninfo/factors"],
  "content": "Carrying twins or higher order multiples requires specialized medical supervision throughout pregnancy. Twin pregnancies carry increased risks of preeclampsia, premature labor, low birth weight, and growth restrictions. To support healthy growth, mothers expecting twins need extra nutrition, higher intakes of folic acid, iron, calcium, and protein, and an appropriate weight gain of about twelve to sixteen kilograms. Frequent prenatal visits and regular ultrasounds help track baby development and spot early signs of preterm labor, while preparing the mother both mentally and physically for a likely cesarean section delivery."
},

    {
  "id": "DOC_HRP_187",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "environmental_and_occupational_heavy_metal_exposure",
  "source": "Centers for Disease Control and Prevention",
  "year": 2024,
  "url": "https://www.cdc.gov/niosh/reproductive-health/prevention/lead-metals.html",
  "content": "Occupational and environmental exposure to heavy metals such as lead, cadmium, mercury, cobalt, and nickel poses severe risks to reproductive and maternal-fetal health. Ingesting or inhaling metal dust and fumes during pregnancy increases the risks of miscarriage, stillbirth, birth defects, and altered brain development in the baby. Heavy metals also pass into breast milk, leading to long term neurodevelopmental issues in nursing infants. High risk occupations include construction, smelting, battery recycling, welding, and stained glass work. Protective measures involve wearing proper respirators, washing hands with specialized lead removing soap before eating, changing work clothes and shoes before going home, and testing blood lead levels during pregnancy or breastfeeding."
},

    {
  "id": "DOC_HRP_188",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "perinatal_mental_health_global_prevalence_and_risks",
  "source": ["World Health Organization", "Policy Center for Maternal Mental Health Synthesis"],
  "year": 2026,
  "url": ["https://www.who.int/teams/mental-health-and-substance-use/promotion-prevention/perinatal-mental-health"
           "https://policycentermmh.org/mmh-disorders/"],
  "content": "Maternal mental health conditions are among the most common complications of pregnancy and the postpartum period, affecting up to one in five women globally. Prevalence rates are higher in low and middle income countries, reaching around sixteen percent during pregnancy and twenty percent after childbirth. Risk factors include extreme stress, living in poverty, low social support, exposure to domestic or gender based violence, emergency or conflict situations, and a personal or family history of mental health disorders. Without proper screening and care, severe maternal distress can lead to poor self care, self harm, suicide, impaired mother baby bonding, infant undernutrition, and long term developmental challenges for the child."
},

    {
  "id": "DOC_HRP_189",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "maternal_depression_dysthymia_and_baby_blues",
  "source": ["World Health Organization, Policy Center for Maternal Mental Health Synthesis"],
  "year": 2026,
  "url": ["https://www.who.int/teams/mental-health-and-substance-use/promotion-prevention/perinatal-mental-health"
           "https://policycentermmh.org/mmh-disorders/"],
  "content": "Temporary mood changes after childbirth, known as the baby blues, affect up to eighty percent of new mothers due to rapid hormone shifts, usually resolving within two weeks without treatment. If a mother has a sad mood, trouble concentrating, loss of joy, or difficulty bonding persists beyond two weeks, it indicates perinatal depression. Depression affects up to twenty percent of pregnant and postpartum women, with onset occurring before pregnancy in twenty seven percent of cases, during pregnancy in thirty three percent, and after birth in forty percent. Women with long term low mood, known as dysthymia, face higher risks of severe depressive episodes during pregnancy and postpartum."
},

    {
  "id": "DOC_HRP_190",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "maternal_anxiety_ocd_and_intrusive_thoughts",
  "source": "Policy Center for Maternal Mental Health Synthesis",
  "year": 2026,
  "url": "https://policycentermmh.org/mmh-disorders/",
  "content": "Maternal anxiety disorders affect about twenty percent of pregnant and postpartum women, causing constant worry, a fast heart rate, feeling restless, and severe trouble sleeping. Perinatal obsessive compulsive disorder affects eight percent of pregnant women and seventeen percent of new mothers after birth. Almost all new mothers have scary, unwanted thoughts about bad things happening to their baby, which is a normal reaction to stress. Unlike psychosis where a person loses touch with reality, mothers with obsessive compulsive disorder stay completely in control of their actions and have no desire or intention to hurt their child."
},

    {
  "id": "DOC_HRP_191",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "bipolar_disorder_and_postpartum_psychosis",
  "source": "Policy Center for Maternal Mental Health Synthesis",
  "year": 2026,
  "url": "https://policycentermmh.org/mmh-disorders/",
  "content": "Bipolar disorder causes major mood swings between deep sadness and high energy. During high energy periods, a woman may need very little sleep, feel hyperactive, and make risky decisions. Over half of women with bipolar disorder experience mood changes during pregnancy or after giving birth. Bipolar type one carries the highest risk for postpartum psychosis, a rare medical emergency that affects about one in every thousand mothers. Postpartum psychosis usually starts suddenly in the first two weeks after birth, causing severe confusion, strange beliefs, and a break from reality that puts both mother and baby in immediate danger."
},

    {
  "id": "DOC_HRP_192",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "traumatic_birth_childbirth_ptsd_and_substance_use",
  "source": "Policy Center for Maternal Mental Health Synthesis",
  "year": 2026,
  "url": "https://policycentermmh.org/mmh-disorders/",
  "content": "A traumatic birth happens when a mother feels terrified, overwhelmed, or in severe physical danger during labor and delivery, which affects about one in three women. Up to twenty percent of mothers develop birth related post traumatic stress disorder afterward, causing painful flashbacks and extreme fear. Using alcohol, tobacco, or drugs during pregnancy affects about five percent of mothers and often happens alongside depression or anxiety. Untreated drug or alcohol use can cause early birth, low birth weight, infant withdrawal symptoms after birth, and delays in how the baby grows and learns."
},

    {
  "id": "DOC_HRP_193",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "pregnancy_loss_grief_and_community_care_models",
  "source": ["World Health Organization", "Policy Center for Maternal Mental Health Synthesis"],
  "year": 2026,
  "url": ["https://www.who.int/teams/mental-health-and-substance-use/promotion-prevention/perinatal-mental-health"
           "https://policycentermmh.org/mmh-disorders/"],
  "content": "Losing a pregnancy through miscarriage or stillbirth causes deep grief and increases the risk for long term depression and anxiety. Mental health support can easily be added into standard pregnancy clinics and local health centers. By following simple guidelines like the WHO Mental Health Gap program, local nurses and community health workers can be trained to spot warning signs early and offer effective care to support mothers and their babies."
},

    {
  "id": "DOC_HRP_194",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "emergency_medical_and_suicide_hotlines_kenya",
  "source": "Open Counseling Kenya",
  "year": 2026,
  "url": "https://blog.opencounseling.com/hotlines-ke/",
  "content": "Emergency medical and mental health hotlines in Kenya offer immediate help during severe health crises. For emergency medical care, call the Emergency Medicine Kenya Foundation toll free at 0800 723 253 or at 0710 633 855. The Kenya Red Cross Society provides emergency support at 1199, 0700 395 395, and 0738 395 395. For suicide emergencies and severe emotional distress, call Niskize at 0900 620 800 or 0718 227 440. Befrienders Kenya provides suicide, depression, and abuse support between seven in the morning and seven in the evening at 0722 178 177."
},

    {
  "id": "DOC_HRP_195",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "gender_based_violence_and_police_hotlines_kenya",
  "source": "Open Counseling Kenya",
  "year": 2026,
  "url": "https://blog.opencounseling.com/hotlines-ke/",
  "content": "Helplines in Kenya provide immediate protection and support for victims of domestic and sexual violence. Healthcare Assistance Kenya offers a national toll free hotline for gender based violence support at 1195. Kimbilio Trust provides specialized support for women facing violence at 1193. For immediate police intervention and security emergencies, contact the Kenya Police at 999, 911, or 112."
},

    {
  "id": "DOC_HRP_196",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "reproductive_health_and_parenting_hotlines_kenya",
  "source": "Open Counseling Kenya",
  "year": 2026,
  "url": "https://blog.opencounseling.com/hotlines-ke/",
  "content": "Hotlines in Kenya assist with sexual and reproductive health, maternal care, and child safety. Marie Stopes Kenya offers reproductive health guidance at 0800 720 005. The One2One Liverpool service provides youth and maternal reproductive health information at 1190. Child Helpline Kenya supports child protection and abuse prevention at 116. Missing Child Kenya provides help for lost or displaced children at 0800 223 344. NASCOP provides care and advice regarding HIV, AIDS, and sexually transmitted infections at 0800 724 848."
},

    {
  "id": "DOC_HRP_197",
  "domain": "maternal_health",
  "topic": "high_risk_pregnancy",
  "subtopic": "substance_abuse_helpline_kenya",
  "source": "Open Counseling Kenya",
  "year": 2026,
  "url": "https://blog.opencounseling.com/hotlines-ke/",
  "content": "The National Authority for the Campaign Against Alcohol and Drug Abuse, known as NACADA, provides a twenty four hour toll free helpline in Kenya for individuals and families dealing with alcohol and substance abuse issues. Mothers or families needing immediate support or treatment guidance for drug and alcohol dependence can call NACADA at 1192."
},

    # Newborn and Early Infant Care
    {
  "id": "DOC_NEIC_198",
  "domain": "maternal_health",
  "topic": "newborn_and_early_infant_care",
  "subtopic": "essential_newborn_care_and_hygiene",
  "source": "World Health Organization",
  "year": 2026,
  "url": "https://www.who.int/tools/your-life-your-health/life-phase/newborns-and-children-under-5-years/caring-for-newborns",
  "content": "Essential daily hygiene and routine care protect newborns from severe infections and support healthy growth. Healthcare providers and parents must wash their hands thoroughly with soap and water before handling the baby. Immediately after birth, the baby should be wiped clean and dry, but the first full bath must be delayed for at least twenty four hours to preserve body heat. The baby should be kept warm by wearing one or two more layers of clothes than an adult, along with a soft hat to cover the head. The mother and baby should share the same room to allow continuous rooming in and on demand feeding. Routine screening tests should be completed for eye problems, hearing issues, and newborn jaundice."
},

    {
  "id": "DOC_NEIC_199",
  "domain": "maternal_health",
  "topic": "newborn_and_early_infant_care",
  "subtopic": "umbilical_cord_care_and_infant_danger_signs",
  "source": "World Health Organization",
  "year": 2026,
  "url": "https://www.who.int/tools/your-life-your-health/life-phase/newborns-and-children-under-5-years/caring-for-newborns",
  "content": "Proper umbilical cord care is necessary to prevent dangerous blood infections in the first weeks of life. The umbilical cord stump must be kept clean and completely dry, without applying any oils, ointments, powders, or traditional substances. Caregivers must know early danger signs that require urgent medical attention at a health facility. Danger signs in a young infant include poor feeding or inability to suck, fast breathing, chest indrawing, high body temperature or feeling unusually cold, extreme sleepiness, or yellow skin and eyes. Spotting these danger signs early allows for quick medical treatment and prevents severe illness."
},

    {
  "id": "DOC_NEIC_200",
  "domain": "maternal_health",
  "topic": "newborn_and_early_infant_care",
  "subtopic": "preterm_and_low_birth_weight_definitions_and_risks",
  "source": "World Health Organization",
  "year": 2026,
  "url": "https://www.who.int/teams/maternal-newborn-child-adolescent-health-and-ageing/newborn-health/preterm-and-low-birth-weight",
  "content": "Preterm infants are babies born before thirty seven weeks of pregnancy, while low birth weight infants weigh less than two point five kilograms at birth regardless of gestational age. Worldwide, fifteen million babies are born preterm and over twenty million are born with low birth weight each year. Prematurity and low birth weight remain the leading causes of death in newborns and children under five years old. Surrounding survival, these small infants face high risks of long term developmental disabilities such as cerebral palsy and eye damage, as well as an increased risk of adult chronic conditions including diabetes and obesity."
},

    {
  "id": "DOC_NEIC_201",
  "domain": "maternal_health",
  "topic": "newborn_and_early_infant_care",
  "subtopic": "kangaroo_mother_care_for_small_infants",
  "source": "World Health Organization",
  "year": 2026,
  "url": "https://www.who.int/teams/maternal-newborn-child-adolescent-health-and-ageing/newborn-health/preterm-and-low-birth-weight",
  "content": "Preterm and low birth weight infants require extra care to maintain body heat, stabilize breathing, and grow safely. For small or premature infants, skin to skin contact, known as kangaroo mother care, should be practiced as much as possible every single day. Direct chest to chest contact keeps the small baby warm, reduces stress, stabilizes heart rate, and promotes frequent breastfeeding. Continuous skin to skin contact significantly improves survival rates for vulnerable small newborns both in health facilities and after returning home."
},

    {
  "id": "DOC_NEIC_202",
  "domain": "maternal_health",
  "topic": "newborn_and_early_infant_care",
  "subtopic": "importance_of_infant_immunization",
  "source": ["Gertrude Childrens Hospital", "Kenya National Vaccines and Immunization Program"],
  "year": 2026,
  "url": ["https://gerties.org/protect-your-childs-health/"
           "https://kenvip-home.jkuat.ac.ke/"],
  "content": "Immunization is a vital healthcare measure that protects infants and children against life threatening infectious diseases. Vaccines work by introducing a safe, harmless piece of a virus or bacterium to trigger antibody production and train the immune system. Timely vaccination prevents severe conditions such as measles, whooping cough, diphtheria, tetanus, polio, and tuberculosis. Beyond individual protection, childhood immunization creates community immunity to protect vulnerable individuals, reduces severe complications such as brain damage or hearing loss, and saves millions of lives globally each year."
},

    {
  "id": "DOC_NEIC_203",
  "domain": "maternal_health",
  "topic": "newborn_and_early_infant_care",
  "subtopic": "kenya_immunization_schedule_birth_to_ten_weeks",
  "source": "Kenya National Vaccines and Immunization Program",
  "year": 2026,
  "url": "https://kenvip-home.jkuat.ac.ke/",
  "content": "The Kenya National Vaccines and Immunization Program outlines specific infant schedules starting at birth. At birth or first contact, infants receive the single dose intradermal BCG vaccine to prevent tuberculosis, along with birth dose oral polio vaccine OPV zero. At six weeks of age, infants receive oral rotavirus dose one, oral polio vaccine OPV dose one, pentavalent vaccine dose one given as an intramuscular injection into the left thigh to protect against diphtheria, tetanus, pertussis, hepatitis B, and Haemophilus influenzae type b, and pneumococcal conjugate vaccine PCV ten dose one given intramuscularly into the right thigh. At ten weeks, infants receive rotavirus dose two, oral polio vaccine OPV dose two, pentavalent dose two, and PCV ten dose two."
},

    {
  "id": "DOC_NEIC_204",
  "domain": "maternal_health",
  "topic": "newborn_and_early_infant_care",
  "subtopic": "kenya_immunization_schedule_fourteen_weeks_to_eighteen_months",
  "source": "Kenya National Vaccines and Immunization Program",
  "year": 2026,
  "url": "https://kenvip-home.jkuat.ac.ke/",
  "content": "The routine immunization schedule in Kenya continues through early childhood to build long lasting immunity. At fourteen weeks of age, infants receive oral polio vaccine OPV dose three, inactivated polio vaccine IPV as an intramuscular injection in the right thigh, pentavalent dose three, and PCV ten dose three. At nine months of age, children receive dose one of the measles rubella MR vaccine in the right upper arm, as well as the yellow fever vaccine in high risk counties including Baringo, Elgeyo Marakwet, West Pokot, and Turkana. At eighteen months, children receive measles rubella MR dose two in the right upper arm to complete their primary infant immunization series."
},

    # Newborn Infections
    {
  "id": "DOC_NI_205",
  "domain": "maternal_health",
  "topic": "newborn_infections",
  "subtopic": "overview_and_transmission_routes_of_newborn_infections",
  "source": "MSD Manuals",
  "year": 2026,
  "url": "https://www.msdmanuals.com/home/children-s-health-issues/infections-in-newborns/overview-of-infections-in-newborns",
  "content": "Infections in newborns are serious health concerns because a newborn baby has an underdeveloped immune system that cannot fight off germs effectively. Although mothers pass protective antibodies to their babies through the placenta during pregnancy, antibody levels may still be too low to stop an illness. Newborns can catch infections in three main ways. First, in the womb through the placenta, causing conditions like rubella, cytomegalovirus, or syphilis. Second, during labor and delivery through exposure to vaginal fluid or blood, causing infections like group B streptococcus or herpes. Third, after birth through contact with infected caregivers, hospital equipment, or breast milk. Early diagnosis using blood, urine, or spinal fluid tests is vital so doctors can give prompt treatment with antibiotics or antiviral medicines."
},

    {
  "id": "DOC_NI_206",
  "domain": "maternal_health",
  "topic": "newborn_infections",
  "subtopic": "newborn_sepsis",
  "source": "MSD Manuals",
  "year": 2026,
  "url": "https://www.msdmanuals.com/home/children-s-health-issues/infections-in-newborns/overview-of-infections-in-newborns",
  "content": "Newborn sepsis is a dangerous, life threatening bloodstream infection caused by bacteria, viruses, or fungi. It is divided into early onset sepsis, which starts in the first three days of life from germs passed during birth, and late onset sepsis, which develops after three days from hospital or home exposure. Premature babies and infants with low birth weight face the highest risk. Common warning signs include listlessness, poor feeding, pale or grayish skin, unstable body temperature, rapid breathing, vomiting, and seizures. Doctors diagnose sepsis using blood cultures and spinal taps. Treatment requires immediate intravenous antibiotics, oxygen, intravenous fluids, and close monitoring in a hospital."
},

    {
  "id": "DOC_NI_207",
  "domain": "maternal_health",
  "topic": "newborn_infections",
  "subtopic": "hospital_acquired_infections_in_newborns",
  "source": "MSD Manuals",
  "year": 2026,
  "url": "https://www.msdmanuals.com/home/children-s-health-issues/infections-in-newborns/overview-of-infections-in-newborns",
  "content": "Hospital acquired infections occur when a newborn gets an infection while staying in a hospital nursery or intensive care unit rather than from the mother before birth. Premature infants and sick full term babies who stay in the hospital for long periods face the greatest risk, especially when using breathing tubes or intravenous lines. The most common hospital acquired infections are pneumonia, bloodstream infections, and staphylococcus skin infections. Symptoms include fever or low body temperature, poor feeding, extreme sleepiness, vomiting, or skin rashes. Thorough handwashing with soap or hand sanitizer by hospital staff and family members is the most effective way to prevent these infections."
},

    {
  "id": "DOC_NIC_208",
  "domain": "maternal_health",
  "topic": "newborn_infections",
  "subtopic": "newborn_pneumonia",
  "source": "MSD Manuals",
  "year": 2026,
  "url": "https://www.msdmanuals.com/home/children-s-health-issues/infections-in-newborns/overview-of-infections-in-newborns",
  "content": "Newborn pneumonia is a serious lung infection caused by bacteria, viruses, or fungi. Early onset pneumonia occurs during or shortly after birth as the baby passes through the birth canal, while late onset pneumonia develops after seven days, most often in premature babies who require breathing tubes in neonatal intensive care units. Signs of pneumonia include fast breathing, grunting sounds, pauses in breathing, increased thick mucus, fever or low body temperature, and poor feeding. Doctors confirm pneumonia using chest x rays, blood tests, and oxygen level sensors. Treatment involves intravenous antibiotics, oxygen support, and fluid care."
},

    {
  "id": "DOC_NI_209",
  "domain": "maternal_health",
  "topic": "newborn_infections",
  "subtopic": "bacterial_meningitis",
  "source": "MSD Manuals",
  "year": 2026,
  "url": "https://www.msdmanuals.com/home/children-s-health-issues/infections-in-newborns/overview-of-infections-in-newborns",
  "content": "Bacterial meningitis is a dangerous infection of the membranes covering the brain and spinal cord, usually resulting from bacteria spreading through the bloodstream. Common causes in newborns include group B streptococcus, E. coli, and listeria. Unlike older children, newborns rarely develop a stiff neck. Instead, warning signs include severe irritability, unusual drowsiness, poor feeding, vomiting, seizures, or a bulging soft spot on top of the skull. Doctors diagnose meningitis using a spinal tap procedure to test spinal fluid. Immediate intravenous antibiotics are critical, as untreated bacterial meningitis is fatal."
},

    {
  "id": "DOC_NI_210",
  "domain": "maternal_health",
  "topic": "newborn_infections",
  "subtopic": "listeriosis_in_newborns",
  "source": "MSD Manuals",
  "year": 2026,
  "url": "https://www.msdmanuals.com/home/children-s-health-issues/infections-in-newborns/overview-of-infections-in-newborns",
  "content": "Listeriosis in newborns is a bacterial infection caused by Listeria monocytogenes, which can be acquired in the womb, during delivery, or after birth. Pregnant women can contract Listeria by eating contaminated foods such as unpasteurized dairy products, soft cheeses, raw vegetables, or prepared deli meats. The bacteria can cross the placenta, potentially leading to miscarriage, preterm delivery, or stillbirth. Symptoms in newborns can appear early within hours or days of birth, often causing sepsis symptoms, or late after several weeks, typically presenting as meningitis or sepsis. Diagnosis is confirmed by culturing blood, spinal fluid, or other bodily fluids from the newborn or mother. Treatment involves intravenous ampicillin combined with an aminoglycoside antibiotic. Prevention focuses on avoiding high-risk contaminated foods during pregnancy and practicing safe food handling."
},

    {
  "id": "DOC_NI_211",
  "domain": "maternal_health",
  "topic": "newborn_infections",
  "subtopic": "conjunctivitis_in_newborns",
  "source": "MSD Manuals",
  "year": 2026,
  "url": "https://www.msdmanuals.com/home/children-s-health-issues/infections-in-newborns/overview-of-infections-in-newborns",
  "content": "Newborn conjunctivitis is an eye infection or inflammation caused by bacteria, viruses, or chemical reactions. Bacterial causes include Chlamydia and Neisseria gonorrhoeae passed from the mother during delivery. Chlamydial conjunctivitis usually starts 5 to 14 days after birth with swollen eyelids and pus discharge, treated with oral erythromycin or azithromycin. Gonococcal conjunctivitis starts 2 to 5 days after birth with severe eyelid swelling and thick pus, which can cause blindness if untreated; it requires hospital treatment with intravenous or intramuscular ceftriaxone or cefotaxime. Chemical conjunctivitis is a mild reaction to preventive eye drops that appears within hours and resolves in a few days. Routine post-delivery preventive measures include applying erythromycin ointment to the newborn's eyes and screening pregnant women during prenatal care."
},

    {
  "id": "DOC_NI_212",
  "domain": "maternal_health",
  "topic": "newborn_infections",
  "subtopic": "herpes_simplex_virus_in_newborns",
  "source": "MSD Manuals",
  "year": 2026,
  "url": "https://www.msdmanuals.com/home/children-s-health-issues/infections-in-newborns/overview-of-infections-in-newborns",
  "content": "Herpes simplex virus (HSV) infection in newborns is typically acquired during delivery through contact with infected genital secretions, even if the mother has no active symptoms. Symptoms usually begin between the first and third weeks of life and typically start as a rash of fluid-filled blisters on the skin, mouth, or eyes. The infection can remain localized to the skin, eyes, or mouth, or it can spread to the brain and central nervous system or cause a severe widespread multi-organ infection affecting the lungs, liver, and brain. Diagnosis is confirmed by viral cultures or PCR testing of blister samples, blood, or spinal fluid. Intravenous acyclovir is the primary treatment, supplemented by topical antiviral eye medications if the eyes are involved. Prevention includes giving pregnant women with a history of herpes oral antivirals starting at 36 weeks and performing a cesarean section if active lesions are present during labor."
},

    {
  "id": "DOC_NI_213",
  "domain": "maternal_health",
  "topic": "newborn_infections",
  "subtopic": "hepatitis_b_virus_in_newborns",
  "source": "MSD Manuals",
  "year": 2026,
  "url": "https://www.msdmanuals.com/home/children-s-health-issues/infections-in-newborns/overview-of-infections-in-newborns",
  "content": "Hepatitis B virus (HBV) infection in newborns occurs primarily during labor and delivery when the infant comes into contact with maternal blood or vaginal fluids. Most infected newborns show no immediate symptoms at birth but become chronic carriers of the virus, which increases the long-term risk of liver disease, cirrhosis, and liver cancer. Infrequently, infected infants develop mild acute hepatitis causing jaundice, listlessness, growth faltering, abdominal swelling, and clay-colored stools. Diagnosis is established through blood tests that check for viral markers and liver enzyme levels, along with liver ultrasound imaging. Prevention is highly effective: all pregnant women are screened, and infants born to HBV-infected mothers receive hepatitis B immune globulin and the first dose of the hepatitis B vaccine within 12 hours of delivery."
},

    {
  "id": "DOC_NI_214",
  "domain": "maternal_health",
  "topic": "newborn_infections",
  "subtopic": "cytomegalovirus_in_newborns",
  "source": "MSD Manuals",
  "year": 2026,
  "url": "https://www.msdmanuals.com/home/children-s-health-issues/infections-in-newborns/overview-of-infections-in-newborns",
  "content": "Cytomegalovirus is the most common congenital viral infection in newborns, occurring when the virus passes across the placenta from an infected mother to the fetus. Perinatal CMV can also be transmitted through vaginal fluids during birth, breast milk, or blood transfusions. While most infected infants have no symptoms, about 13 percent of congenitally infected newborns show symptoms such as premature birth, low birth weight, small head size (microcephaly), jaundice, enlarged liver and spleen, and skin bruises. Diagnosis is confirmed by detecting CMV DNA via PCR testing of urine, saliva, or blood samples taken shortly after birth. Although there is no cure, antiviral medications like ganciclovir or valganciclovir can reduce the severity of symptoms and lower the risk of progressive hearing loss."
},

    {
  "id": "DOC_NI_215",
  "domain": "maternal_health",
  "topic": "newborn_infections",
  "subtopic": "rubella_in_newborns",
  "source": "MSD Manuals",
  "year": 2026,
  "url": "https://www.msdmanuals.com/home/children-s-health-issues/infections-in-newborns/overview-of-infections-in-newborns",
  "content": "Congenital rubella occurs when a non-immune pregnant womanContracts the rubella virus during pregnancy—especially during the first 12 weeks—and passes it to the fetus through the placenta. Maternal infection can lead to miscarriage, stillbirth, or severe multiple birth defects known as congenital rubella syndrome (CRS). Common defects of CRS include low birth weight, small head size, cataracts, retinal damage, heart defects, enlarged liver and spleen, and progressive hearing loss. Diagnosis is made using antibody tests and viral detection in blood, throat, or urine samples of the mother and infant. There is no specific cure for rubella, so management involves supportive care for specific organ defects. Prevention is achieved through routine MMR vaccination of non-pregnant individuals prior to pregnancy, as the live vaccine cannot be administered during pregnancy."
}

]

## 1.3 Save the knowledge base

In [3]:
with open("maternal_health_knowledge_base.json", "w") as f:
    json.dump(maternal_documents, f, indent=4)

## 1.4 Load the knowledge base

In [4]:
with open("maternal_health_knowledge_base.json", "r") as f:
    maternal_documents = json.load(f)

print("Number of documents:", len(maternal_documents))

Number of documents: 215


## Phase 2: Data Preprocessing

## 2.1 Data cleaning and normalization

In [5]:
# Normalizes the dataset

def normalize_to_list(value):
    if isinstance(value, list):
        return value
    return [value]

for doc in maternal_documents:
    doc['source'] = normalize_to_list(doc['source'])
    doc['url'] = normalize_to_list(doc['url'])

print("Normalization applied.")

Normalization applied.


In [6]:
# Split any source name that contains a comma into two separate list items.

for doc in maternal_documents:
    new_sources = []
    for s in doc['source']:
        if ',' in s:
            # Split on the comma, and strip extra spaces from each part
            parts = [p.strip() for p in s.split(',')]
            new_sources.extend(parts)
        else:
            new_sources.append(s)
    doc['source'] = new_sources

print("Comma-joined sources split.")

Comma-joined sources split.


In [7]:
# This checks that the documents are complete and consistent across all 215 documents.
import collections

# Check 1: what data type is "source" across all documents? All of them should have the data type as list
source_types = collections.Counter(type(d['source']).__name__ for d in maternal_documents)
print("source field types:", source_types)

# Check 2: what data type is "url"?
url_types = collections.Counter(type(d['url']).__name__ for d in maternal_documents)
print("url field types:", url_types)

# Check 3: what data type is "year"? int is expected
year_types = collections.Counter(type(d['year']).__name__ for d in maternal_documents)
print("year field types:", year_types)

# Check 4: does every document have the exact same set of field names?
field_sets = collections.Counter(tuple(sorted(d.keys())) for d in maternal_documents)
print("distinct field combinations:", field_sets)

# Check 5: how many distinct, cleaned-up source names do we have now?
all_sources = [s for d in maternal_documents for s in d['source']]
print("\nunique sources:", sorted(set(all_sources)))

source field types: Counter({'list': 215})
url field types: Counter({'list': 215})
year field types: Counter({'int': 215})
distinct field combinations: Counter({('content', 'domain', 'id', 'source', 'subtopic', 'topic', 'url', 'year'): 215})

unique sources: ['Aga Khan University Hospital', 'American College of Obstetricians and Gynecologists', 'Birth Injury Justice Center', 'Centers for Disease Control and Prevention', 'Cleveland Clinic', 'Gertrude Childrens Hospital', 'International Diabetes Federation', 'Johns Hopkins Medicine', 'Kenya Ministry of Health', 'Kenya National Vaccines and Immunization Program', 'Kenyatta University Hospital', 'MSD Manuals', 'National Institute of Child Health Development', 'National Library of Medicine', 'Open Counseling Kenya', 'Perth Obstetrics and Gynaecology Specialists', 'Policy Center for Maternal Mental Health Synthesis', 'The Mother Baby Center', 'UNICEF', 'World Health Organization']


## 2.2 Experiments on which chunk size and overlap is the best

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Test each chunk size against the actual 215 documents and record how many documents get split into more than one piece.
results = []
for chunk_size, overlap in [(300, 50), (500, 75), (800, 100), (1000, 100)]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    split_docs = 0
    total_chunks = 0
    for d in maternal_documents:
        pieces = splitter.split_text(d['content'])
        total_chunks += len(pieces)
        if len(pieces) > 1:
            split_docs += 1
    results.append((chunk_size, overlap, split_docs, total_chunks))
    print(f"chunk_size={chunk_size}, overlap={overlap}: {split_docs}/{len(maternal_documents)} docs split, total chunks={total_chunks}")

print("\nLongest document in the knowledge base:", max(len(d['content']) for d in maternal_documents), "characters")

chunk_size=300, overlap=50: 182/215 docs split, total chunks=462
chunk_size=500, overlap=75: 61/215 docs split, total chunks=279
chunk_size=800, overlap=100: 15/215 docs split, total chunks=230
chunk_size=1000, overlap=100: 0/215 docs split, total chunks=215

Longest document in the knowledge base: 974 characters


In [9]:
import textwrap
# For each chunk size, show what happens to the same document (DOC_ANC_002) which is long enough to be affected by smaller sizes
target_id = "DOC_ANC_002"
target_doc = next(d for d in maternal_documents if d['id'] == target_id)

# How many characters wide each printed line should be before it wraps to the next line
WRAP_WIDTH = 140

for chunk_size, overlap in [(300, 50), (500, 75), (800, 100), (1000, 100)]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    pieces = splitter.split_text(target_doc['content'])
    print(f"\n=== Chunk_size={chunk_size} -> {len(pieces)} piece(s) ===")
    for p in pieces:
        wrapped = textwrap.fill(p, width=WRAP_WIDTH)
        print(wrapped)
        print()


=== Chunk_size=300 -> 4 piece(s) ===
WHO's 2016 ANC model recommends a minimum of eight contacts between a pregnant woman and a health provider, replacing the earlier four-visit
model, to reduce perinatal mortality and improve a woman's experience of care. The recommended schedule is: Contact 1 by 12 weeks of
gestation, Contact 2 at 20

1 by 12 weeks of gestation, Contact 2 at 20 weeks, Contact 3 at 26 weeks, Contact 4 at 30 weeks, Contact 5 at 34 weeks, Contact 6 at 36
weeks, Contact 7 at 38 weeks, and Contact 8 at 40 weeks. If the woman has not given birth by 41 weeks, she should return for assessment. Any
recommended

she should return for assessment. Any recommended intervention missed at one contact should be provided at the next. These antenatal visits
allow health professionals to monitor the health of the mother and baby. These visits help detect complications early and provide
appropriate counselling and

early and provide appropriate counselling and treatment.


=== Chunk_si

## 2.3 Actual chunking implementation

In [10]:
# Actual chunking of the knowledge base
# Chosen settings: chunk_size=1000, chunk_overlap=100
from langchain_text_splitters import RecursiveCharacterTextSplitter
import textwrap
import json

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

chunked_documents = []

for doc in maternal_documents:

    pieces = splitter.split_text(doc['content'])

    for i, piece in enumerate(pieces):
        chunk = {
            # A unique id for this specific chunk, e.g. "DOC_ANC_001_0"
            "chunk_id": f"{doc['id']}_{i}",

            # Which original document this chunk came from
            "source_doc_id": doc['id'],

            # The actual text for this chunk
            "content": piece,

            # Carry forward the rest of the original metadata
            "domain": doc['domain'],
            "topic": doc['topic'],
            "subtopic": doc['subtopic'],
            "source": doc['source'],
            "url": doc['url'],
            "year": doc['year'],
        }
        chunked_documents.append(chunk)
print("Total chunks created:", len(chunked_documents))

example = chunked_documents[0].copy()
example['content'] = textwrap.fill(example['content'], width=80)

print("\nExample chunk:")
print(json.dumps(example, indent=4))

Total chunks created: 215

Example chunk:
{
    "chunk_id": "DOC_ANC_001_0",
    "source_doc_id": "DOC_ANC_001",
    "content": "The 2016 WHO guideline on antenatal care (ANC) sets out 49 recommendations\nacross five areas: nutritional interventions, maternal and fetal assessment,\npreventive measures, management of common physiological symptoms, and health-\nsystem interventions to improve ANC quality. The guideline's central aim is a\npositive pregnancy experience, defined as maintaining physical and social\nwellbeing, preventing or managing complications, and supporting a woman's\nconfidence and autonomy through pregnancy, not simply preventing death or\nillness.",
    "domain": "maternal_health",
    "topic": "antenatal_care",
    "subtopic": "antenatal_care_overview",
    "source": [
        "World Health Organization"
    ],
    "url": [
        "https://iris.who.int/server/api/core/bitstreams/9dccde13-3593-4a22-9237-61abe5a3c6b7/content"
    ],
    "year": 2016
}


Save the chunked documents

In [11]:
import json

with open("chunked_maternal_health_kb.json", "w") as f:
    json.dump(chunked_documents, f, indent=4)

print("Saved", len(chunked_documents), "chunks to chunked_maternal_health_kb.json")

Saved 215 chunks to chunked_maternal_health_kb.json


## 2.4 Embedding

In [12]:
# Embed the chunks using sentence transformers
!pip install sentence-transformers -q

from sentence_transformers import SentenceTransformer
import numpy as np

# Load the embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Pull out just the text content from every chunk since that's the only part we're turning into numbers
chunk_texts = [chunk['content'] for chunk in chunked_documents]

# Turn all 215 pieces of text into embeddings in one call and if it works show true
embeddings = model.encode(chunk_texts, show_progress_bar=True)

# Check the shape of chunks and emebddings we get
print("Embeddings shape:", embeddings.shape)

for i, chunk in enumerate(chunked_documents):
    chunk['embedding'] = embeddings[i].tolist()

print("\nExample chunk with embedding (showing first 5 numbers of the embedding):")
print("chunk_id:", chunked_documents[0]['chunk_id'])
print("embedding (first 5 of 384 numbers):", chunked_documents[0]['embedding'][:5])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings shape: (215, 384)

Example chunk with embedding (showing first 5 numbers of the embedding):
chunk_id: DOC_ANC_001_0
embedding (first 5 of 384 numbers): [-0.06557019054889679, -0.019004935398697853, -0.044749923050403595, 0.05324581637978554, 0.0002687290252652019]


Save the embedded chunks

In [13]:
import json

with open("embedded_maternal_health_kb.json", "w") as f:
    json.dump(chunked_documents, f, indent=4)

print("Saved", len(chunked_documents), "embedded chunks to embedded_maternal_health_kb.json")

Saved 215 embedded chunks to embedded_maternal_health_kb.json


## Phase 3: Vector database

## 3.1 Set up FAISS vector database

In [14]:
# Phase 3: Set up the FAISS vector database and load our chunks
!pip install faiss-cpu -q

import faiss
import numpy as np

embedding_matrix = np.array(
    [chunk['embedding'] for chunk in chunked_documents],
    dtype='float32'
)

embedding_dimension = embedding_matrix.shape[1]

# IndexFlatL2 compares a new embedding against every stored embedding directly
index = faiss.IndexFlatL2(embedding_dimension)
index.add(embedding_matrix)

print("Number of embeddings stored in the FAISS index:", index.ntotal)

# FAISS only stores numbers, not metadata so we keep our own lookup list, in the SAME order as the FAISS index
chunk_lookup = [
    {
        "chunk_id": chunk['chunk_id'],
        "content": chunk['content'],
        "topic": chunk['topic'],
        "subtopic": chunk['subtopic'],
        "source": chunk['source'],
        "url": chunk['url'],
        "year": chunk['year'],
    }
    for chunk in chunked_documents
]

print("Lookup list length (should match the FAISS count):", len(chunk_lookup))

Number of embeddings stored in the FAISS index: 215
Lookup list length (should match the FAISS count): 215


## 3.2 Save the FAISS index and lookup list

In [15]:
faiss.write_index(index, "maternal_health_faiss.index")

with open("chunk_lookup.json", "w") as f:
    json.dump(chunk_lookup, f, indent=4)

print("Saved FAISS index to maternal_health_faiss.index")
print("Saved chunk lookup list to chunk_lookup.json")

Saved FAISS index to maternal_health_faiss.index
Saved chunk lookup list to chunk_lookup.json


## 3.3 Test retrieval with three sample questions

In [16]:
# Test retrieval with sample questions
sample_question = "What are the danger signs I should watch for during pregnancy?"

question_embedding = model.encode([sample_question])
question_embedding = np.array(question_embedding, dtype='float32')

k = 3
distances, indices = index.search(question_embedding, k)

print(f"Question: {sample_question}\n")
print(f"Top {k} matching chunks:\n")

for rank, position in enumerate(indices[0]):
    matched_chunk = chunk_lookup[position]
    distance = distances[0][rank]
    print(f"Rank {rank + 1} (distance: {distance:.4f})")
    print("chunk_id:", matched_chunk['chunk_id'])
    print("topic:", matched_chunk['topic'])
    print("content:", textwrap.fill(matched_chunk['content'], width=80))
    print()

Question: What are the danger signs I should watch for during pregnancy?

Top 3 matching chunks:

Rank 1 (distance: 0.5550)
chunk_id: DOC_DS_017_0
topic: danger_signs_pregnancy
content: Danger signs in pregnancy are symptoms that can signal a life-threatening
complication for the mother or baby and require immediate care at a health
facility, not a wait-and-see approach. Across WHO, CDC, UNICEF, and Kenya's
Ministry of Health guidance, the core danger signs are consistent: vaginal
bleeding, severe or persistent headache, blurred vision, convulsions or fits,
difficulty breathing or chest pain, severe abdominal pain, high fever, severe
swelling of the face or hands, reduced or absent fetal movement, severe nausea
and vomiting, and fluid leaking from the vagina before the due date. A pregnant
woman experiencing any of these should go to a health facility right away rather
than waiting for her next scheduled antenatal visit.

Rank 2 (distance: 0.7449)
chunk_id: DOC_HRP_181_0
topic: high_ri

In [17]:
sample_question = "What foods should I eat during pregnancy?"
question_embedding = model.encode([sample_question])
question_embedding = np.array(question_embedding, dtype='float32')

distances, indices = index.search(question_embedding, k)

print(f"Question: {sample_question}\n")
for rank, position in enumerate(indices[0]):
    matched_chunk = chunk_lookup[position]
    print(f"Rank {rank + 1} (distance: {distances[0][rank]:.4f})")
    print("chunk_id:", matched_chunk['chunk_id'])
    print("topic:", matched_chunk['topic'])
    print("content:", textwrap.fill(matched_chunk['content'], width=80))
    print()

Question: What foods should I eat during pregnancy?

Rank 1 (distance: 0.5115)
chunk_id: DOC_NUT_102_0
topic: nutrition_pregnancy
content: Pregnant women should consume a balanced diet that includes fruits, vegetables,
whole grains, dairy products and protein-rich foods. A varied diet helps ensure
adequate intake of essential nutrients needed for maternal health and fetal
development.

Rank 2 (distance: 0.5142)
chunk_id: DOC_NUT_115_0
topic: nutrition_pregnancy
content: Eating a wide variety of locally available foods during pregnancy helps ensure
mothers get all the essential vitamins and minerals needed for a healthy
pregnancy. Eating from multiple food groups such as fresh vegetables, fruits,
whole grains, beans, and lean meats prevents maternal malnutrition and supports
healthy growth and development for the baby.

Rank 3 (distance: 0.7049)
chunk_id: DOC_ANC_003_0
topic: antenatal_care
content: Counselling on healthy eating and staying physically active during pregnancy is
recommen

In [18]:
sample_question = "What causes heavy bleeding after childbirth?"
question_embedding = model.encode([sample_question])
question_embedding = np.array(question_embedding, dtype='float32')

distances, indices = index.search(question_embedding, k)

print(f"Question: {sample_question}\n")
for rank, position in enumerate(indices[0]):
    matched_chunk = chunk_lookup[position]
    print(f"Rank {rank + 1} (distance: {distances[0][rank]:.4f})")
    print("chunk_id:", matched_chunk['chunk_id'])
    print("topic:", matched_chunk['topic'])
    print("content:", textwrap.fill(matched_chunk['content'], width=80))
    print()

Question: What causes heavy bleeding after childbirth?

Rank 1 (distance: 0.4170)
chunk_id: DOC_POST_135_0
topic: postnatal_care
content: Heavy vaginal bleeding after childbirth is a life-threatening emergency that can
indicate postpartum hemorrhage. Mothers who soak through more than two sanitary
towels in an hour, pass large blood clots, or experience sudden severe bleeding
must go to a health facility immediately for emergency medical care.

Rank 2 (distance: 0.4504)
chunk_id: DOC_PPH_055_0
topic: postpartum_hemorrhage
content: Physical trauma during childbirth, such as tears or lacerations to the vagina,
cervix, perineum, or uterus, is a major cause of postpartum hemorrhage. These
birth canal injuries can cause rapid, significant blood loss or allow blood to
collect internally, requiring surgical repair or stitches to stop the bleeding.

Rank 3 (distance: 0.4638)
chunk_id: DOC_PPH_046_0
topic: postpartum_hemorrhage
content: Postpartum hemorrhage is heavy, excessive bleeding that oc

## 3.4 Build a helper to search chunks by keyword

In [19]:
# This looks through every chunk's text and prints the ones that contain the keyword you give it, along with their chunk_id
def search_by_keyword(keyword):
    matches_found = 0
    for chunk in chunk_lookup:
        if keyword.lower() in chunk['content'].lower():
            print(chunk['chunk_id'], '(topic:', chunk['topic'] + ') -', chunk['content'][:100]) # shoes first 100 characters
            matches_found += 1
    if matches_found == 0:
        print("No chunks found containing:", keyword)

In [20]:
search_by_keyword("mental health")

DOC_DS_030_0 (topic: danger_signs_pregnancy) - Thoughts of harming oneself or the baby during pregnancy are serious warning signs that require imme
DOC_HRP_188_0 (topic: high_risk_pregnancy) - Maternal mental health conditions are among the most common complications of pregnancy and the postp
DOC_HRP_193_0 (topic: high_risk_pregnancy) - Losing a pregnancy through miscarriage or stillbirth causes deep grief and increases the risk for lo
DOC_HRP_194_0 (topic: high_risk_pregnancy) - Emergency medical and mental health hotlines in Kenya offer immediate help during severe health cris


In [21]:
# Print the FULL content of one specific chunk, by its chunk_id.
# This will help in the next step when checking retrieval quality using precision@k, recall@k,MRR and nDCG
def show_full_chunk(chunk_id):
    for chunk in chunk_lookup:
        if chunk['chunk_id'] == chunk_id:
            print("chunk_id:", chunk['chunk_id'])
            print("topic:", chunk['topic'])
            print("content:", textwrap.fill(chunk['content'], width=80))
            return
    print("No chunk found with id:", chunk_id)

show_full_chunk("DOC_HRP_188_0")

chunk_id: DOC_HRP_188_0
topic: high_risk_pregnancy
content: Maternal mental health conditions are among the most common complications of
pregnancy and the postpartum period, affecting up to one in five women globally.
Prevalence rates are higher in low and middle income countries, reaching around
sixteen percent during pregnancy and twenty percent after childbirth. Risk
factors include extreme stress, living in poverty, low social support, exposure
to domestic or gender based violence, emergency or conflict situations, and a
personal or family history of mental health disorders. Without proper screening
and care, severe maternal distress can lead to poor self care, self harm,
suicide, impaired mother baby bonding, infant undernutrition, and long term
developmental challenges for the child.


In [22]:
for cid in ["DOC_ANC_002_0", "DOC_PRE_032_0", "DOC_GDM_066_0", "DOC_IPV_163_0", "DOC_LAB_144_0", "DOC_LAB_145_0"]:
    show_full_chunk(cid)
    print("---")

chunk_id: DOC_ANC_002_0
topic: antenatal_care
content: WHO's 2016 ANC model recommends a minimum of eight contacts between a pregnant
woman and a health provider, replacing the earlier four-visit model, to reduce
perinatal mortality and improve a woman's experience of care. The recommended
schedule is: Contact 1 by 12 weeks of gestation, Contact 2 at 20 weeks, Contact
3 at 26 weeks, Contact 4 at 30 weeks, Contact 5 at 34 weeks, Contact 6 at 36
weeks, Contact 7 at 38 weeks, and Contact 8 at 40 weeks. If the woman has not
given birth by 41 weeks, she should return for assessment. Any recommended
intervention missed at one contact should be provided at the next. These
antenatal visits allow health professionals to monitor the health of the mother
and baby. These visits help detect complications early and provide appropriate
counselling and treatment.
---
chunk_id: DOC_PRE_032_0
topic: preeclampsia
content: Pre-eclampsia is clinically diagnosed by the onset of hypertension (blood
pressure 

In [23]:
# Gold-labeled test questions for evaluating retrieval quality
# Each entry has a question and the chunk_id(s) that I have confirmed are the correct/relevant answers.
# This is the "answer key" the retrieval metrics compare FAISS's real results against.

gold_questions = [
    {
        "question": "How many antenatal care visits does WHO recommend during pregnancy?",
        "relevant_chunk_ids": ["DOC_ANC_002_0"],
    },
    {
        "question": "How is preeclampsia diagnosed?",
        "relevant_chunk_ids": ["DOC_PRE_032_0"],
    },
    {
        "question": "What causes gestational diabetes?",
        "relevant_chunk_ids": ["DOC_GDM_066_0"],
    },
    {
        "question": "How common is intimate partner violence during pregnancy?",
        "relevant_chunk_ids": ["DOC_IPV_163_0"],
    },
    {
        "question": "What are the early signs that labor is starting?",
        "relevant_chunk_ids": ["DOC_LAB_144_0", "DOC_LAB_145_0"],
    },
]

print("Number of gold questions so far:", len(gold_questions))

Number of gold questions so far: 5


In [24]:
for cid in ["DOC_HRP_177_0", "DOC_PRE_034_0", "DOC_PRE_035_0", "DOC_INF_085_0", "DOC_NI_206_0", "DOC_POST_138_0", "DOC_BP_123_0", "DOC_BP_126_0"]:
    show_full_chunk(cid)
    print("---")

chunk_id: DOC_HRP_177_0
topic: high_risk_pregnancy
content: Health conditions that a woman has before getting pregnant can make a pregnancy
high risk for both mother and baby. High blood pressure before pregnancy can
damage the mother's kidneys and cause low birth weight or severe high blood
pressure during pregnancy. Having diabetes before pregnancy means blood sugar
must be carefully controlled before and during pregnancy to prevent birth
defects, unusually large babies, and low blood sugar in the newborn. Polycystic
ovary syndrome (PCOS) increases the risk of miscarriage before 20 weeks,
pregnancy diabetes, and preeclampsia. Kidney disease makes it harder to get
pregnant and raises the chance of early delivery or preeclampsia—in fact, nearly
1 in 5 women who get early preeclampsia actually have hidden kidney disease.
Autoimmune diseases like lupus increase the risk of early labor and stillbirth,
meaning doctors must carefully manage the mother's treatment so her medications
do not h

In [25]:
gold_questions.extend([
    {
        "question": "What pre-existing health conditions can make a pregnancy high-risk?",
        "relevant_chunk_ids": ["DOC_HRP_177_0", "DOC_PRE_034_0", "DOC_PRE_035_0"]
    },
    {
        "question": "How is Group B Streptococcus detected during pregnancy?",
        "relevant_chunk_ids": ["DOC_INF_085_0"],
    },
    {
        "question": "What is newborn sepsis?",
        "relevant_chunk_ids": ["DOC_NI_206_0"],
    },
    {
        "question": "What does a high fever with severe headache after childbirth indicate?",
        "relevant_chunk_ids": ["DOC_POST_138_0"],
    },
    {
        "question": "What should be included in a birth preparedness plan?",
        "relevant_chunk_ids": ["DOC_BP_126_0"],
    },
])

print("Number of gold questions so far:", len(gold_questions))

Number of gold questions so far: 10


In [26]:
for cid in ["DOC_IFA_120_0", "DOC_PPH_054_0", "DOC_DS_024_0", "DOC_NEIC_199_0", "DOC_ANC_010_0"]:
    show_full_chunk(cid)
    print("---")

chunk_id: DOC_IFA_120_0
topic: nutrition_pregnancy
content: The World Health Organization recommends that pregnant women take a daily
supplement containing thirty to sixty milligrams of elemental iron and four
hundred micrograms or zero point four milligrams of folic acid. Taking this
daily combination helps maintain healthy iron levels, prevents maternal anemia,
and supports the baby's early physical and brain development.
---
chunk_id: DOC_PPH_054_0
topic: postpartum_hemorrhage
content: Uterine atony is the most common cause of postpartum hemorrhage, responsible for
up to eighty percent of cases. It happens when the uterine muscles remain soft
and weak after delivery instead of contracting firmly. Strong contractions are
necessary to squeeze shut the open blood vessels where the placenta attached, so
when the uterus fails to contract, heavy bleeding continues.
---
chunk_id: DOC_DS_024_0
topic: danger_signs_pregnancy
content: Mild swelling of the feet and ankles is common and normal i

In [27]:
gold_questions.extend([
    {
        "question": "How much iron should a pregnant woman take daily?",
        "relevant_chunk_ids": ["DOC_IFA_120_0"],
    },
    {
        "question": "What is the most common cause of postpartum hemorrhage?",
        "relevant_chunk_ids": ["DOC_PPH_054_0"],
    },
    {
        "question": "Is swelling of the feet and ankles during pregnancy dangerous?",
        "relevant_chunk_ids": ["DOC_DS_024_0"],
    },
    {
        "question": "How should the umbilical cord stump be cared for?",
        "relevant_chunk_ids": ["DOC_NEIC_199_0"],
    },
    {
        "question": "What can relieve heartburn during pregnancy?",
        "relevant_chunk_ids": ["DOC_ANC_010_0"],
    },
])

print("Number of gold questions so far:", len(gold_questions))

Number of gold questions so far: 15


In [28]:
for cid in ["DOC_ANC_015_0", "DOC_ANC_016_0"]:
    show_full_chunk(cid)
    print("---")

chunk_id: DOC_ANC_015_0
topic: continuity_of_care
content: Each pregnant woman is recommended to carry her own case notes during pregnancy,
whether paper-based or electronic, to improve continuity and quality of care and
her overall pregnancy experience. Women who carry their own notes report greater
access to pregnancy-related information and a stronger sense of involvement in
their own care. Health systems implementing this should be mindful that women
with lower health literacy may need support understanding their notes, and that
data privacy and sensitive information (such as HIV status) need appropriate
safeguards.
---
chunk_id: DOC_ANC_016_0
topic: continuity_of_care
content: In settings with well-functioning midwifery programmes, midwife-led continuity-
of-care models are recommended, where a known midwife or small group of known
midwives supports a woman through pregnancy, birth, and the postnatal period.
This model requires enough trained midwives for each woman to see the sam

In [29]:
gold_questions.extend([
    {
        "question": "Why should pregnant women carry their own case notes?",
        "relevant_chunk_ids": ["DOC_ANC_015_0"],
    },
    {
        "question": "What is a midwife-led continuity-of-care model?",
        "relevant_chunk_ids": ["DOC_ANC_016_0"],
    },
    ])

print("Number of gold questions so far:", len(gold_questions))

Number of gold questions so far: 17


In [30]:
candidates = ["DOC_NUT_108_0", "DOC_IFA_122_0", "DOC_PRE_033_0", "DOC_PRE_042_0",
              "DOC_POST_137_0", "DOC_POST_140_0", "DOC_ANC_011_0", "DOC_ANC_013_0",
              "DOC_INF_088_0", "DOC_INF_090_0", "DOC_NEIC_201_0", "DOC_NEIC_203_0",
              "DOC_HRP_194_0"]
for cid in candidates:
    show_full_chunk(cid)
    print("---")

chunk_id: DOC_NUT_108_0
topic: nutrition_pregnancy
content: Iron is an essential micronutrient during pregnancy required for maternal
hemoglobin production, red blood cell synthesis, and fetal oxygen supply. Severe
iron deficiency leads to maternal anemia, increased fatigue, heightened risk of
infection, impaired fetal growth, and higher susceptibility to severe
complications during childbirth such as postpartum hemorrhage and low birth
weight.
---
chunk_id: DOC_IFA_122_0
topic: nutrition_pregnancy
content: Folic acid supplementation should ideally begin before getting pregnant or as
early as possible during the first weeks of pregnancy. Taking folic acid early
in pregnancy provides critical protection when the baby's spine and brain are
forming, significantly lowering the risk of severe birth defects known as neural
tube defects.
---
chunk_id: DOC_PRE_033_0
topic: preeclampsia
content: If pre-eclampsia is not properly managed, it can progress to eclampsia.
Eclampsia is a severe, life-

In [31]:
gold_questions.extend([
    {
        "question": "Why is iron important during pregnancy?",
        "relevant_chunk_ids": ["DOC_NUT_108_0"],
    },
    {
        "question": "When should folic acid supplementation begin during pregnancy?",
        "relevant_chunk_ids": ["DOC_IFA_122_0"],
    },
    {
        "question": "What is eclampsia and how does it relate to preeclampsia?",
        "relevant_chunk_ids": ["DOC_PRE_033_0"],
    },
    {
        "question": "What medication is used to prevent seizures in severe preeclampsia?",
        "relevant_chunk_ids": ["DOC_PRE_042_0"],
    },
    {
        "question": "What breathing problems after childbirth should prompt urgent care?",
        "relevant_chunk_ids": ["DOC_POST_137_0"],
    },
    {
        "question": "What urinary symptoms after childbirth need medical attention?",
        "relevant_chunk_ids": ["DOC_POST_140_0"],
    },
    {
        "question": "How can leg cramps during pregnancy be relieved?",
        "relevant_chunk_ids": ["DOC_ANC_011_0"],
    },
    {
        "question": "What helps with constipation during pregnancy?",
        "relevant_chunk_ids": ["DOC_ANC_013_0"],
    },
    {
        "question": "How can listeriosis be prevented during pregnancy?",
        "relevant_chunk_ids": ["DOC_INF_088_0"],
    },
    {
        "question": "Why is rubella infection dangerous during early pregnancy?",
        "relevant_chunk_ids": ["DOC_INF_090_0"],
    },
    {
        "question": "How should preterm or low birth weight babies be kept warm?",
        "relevant_chunk_ids": ["DOC_NEIC_201_0"],
    },
    {
        "question": "What vaccines does a baby receive at birth in Kenya?",
        "relevant_chunk_ids": ["DOC_NEIC_203_0"],
    },
    {
        "question": "What emergency hotline can be called for urgent medical help in Kenya?",
        "relevant_chunk_ids": ["DOC_HRP_194_0"],
    },
])

print("Number of gold questions so far:", len(gold_questions))

Number of gold questions so far: 30


In [32]:
# Helper: run one question through the same retrieval pipeline

# Turn the question into an embedding, search FAISS,
# and return just the list of chunk_ids

def get_retrieved_ids(question, k):
    q_embedding = model.encode([question])         # turn the question into 384 numbers
    q_embedding = np.array(q_embedding, dtype='float32')  # FAISS needs float32
    distances, indices = index.search(q_embedding, k)     # find the k closest chunks
    # indices[0] holds the positions of the matches; we convert
    # those into actual chunk_ids using our chunk_lookup list.
    return [chunk_lookup[i]['chunk_id'] for i in indices[0]]

## 3.5 Define the four retrieval quality metrics

In [33]:
# The four retrieval quality metrics from the proposal

import numpy as np

def precision_at_k(retrieved_ids, relevant_ids):
    # This answers "of what we retrieved, how much was actually
    # correct?" A low precision means the system is returning a lot
    # of irrelevant chunks alongside the good ones.
    hits = sum(1 for cid in retrieved_ids if cid in relevant_ids)
    return hits / len(retrieved_ids)

def recall_at_k(retrieved_ids, relevant_ids):
    # This answers "of everything that was relevant, how much
    # did we actually find?" A low recall means the system is
    # missing correct answers that exist in the knowledge base.
    hits = sum(1 for cid in retrieved_ids if cid in relevant_ids)
    return hits / len(relevant_ids)

def reciprocal_rank(retrieved_ids, relevant_ids):
    # Which answer occurs at rank 1 counts? If nothing relevant
    # was found at all, the score is 0.
    for rank, cid in enumerate(retrieved_ids, start=1):
        if cid in relevant_ids:
            return 1 / rank
    return 0

def ndcg_at_k(retrieved_ids, relevant_ids):
    # Instead of only caring about the first correct answer, this
    # rewards every correct answer found, weighted by how early it
    # appeared.
    dcg = 0
    for rank, cid in enumerate(retrieved_ids, start=1):
        if cid in relevant_ids:
            dcg += 1 / np.log2(rank + 1)
    ideal_dcg = sum(1 / np.log2(rank + 1) for rank in range(1, len(relevant_ids) + 1))
    return dcg / ideal_dcg if ideal_dcg > 0 else 0

## 3.6 Run the thirty questions using the four retrieval quality metrics and average the results

In [34]:
# Run each gold question through retrieval, print its own scores then show the averages at the end

k = 3
all_precision, all_recall, all_mrr, all_ndcg = [], [], [], []

# Normal counting from Q1 to Q30
for question_number, item in enumerate(gold_questions, start=1):
    retrieved = get_retrieved_ids(item['question'], k)
    relevant = item['relevant_chunk_ids']

    p = precision_at_k(retrieved, relevant)
    r = recall_at_k(retrieved, relevant)
    mrr = reciprocal_rank(retrieved, relevant)
    ndcg = ndcg_at_k(retrieved, relevant)

    all_precision.append(p)
    all_recall.append(r)
    all_mrr.append(mrr)
    all_ndcg.append(ndcg)

    # Print this question's number, text, and its own result.
    print(f"Q{question_number}: {item['question']}")
    print(f"   Precision={p:.2f}  Recall={r:.2f}  MRR={mrr:.2f}  nDCG={ndcg:.2f}")

print("\nAverages across all 30 questions")
print(f"Average Precision@{k}: {np.mean(all_precision):.3f}")
print(f"Average Recall@{k}: {np.mean(all_recall):.3f}")
print(f"Average MRR: {np.mean(all_mrr):.3f}")
print(f"Average nDCG@{k}: {np.mean(all_ndcg):.3f}")

Q1: How many antenatal care visits does WHO recommend during pregnancy?
   Precision=0.33  Recall=1.00  MRR=1.00  nDCG=1.00
Q2: How is preeclampsia diagnosed?
   Precision=0.33  Recall=1.00  MRR=0.50  nDCG=0.63
Q3: What causes gestational diabetes?
   Precision=0.33  Recall=1.00  MRR=1.00  nDCG=1.00
Q4: How common is intimate partner violence during pregnancy?
   Precision=0.33  Recall=1.00  MRR=1.00  nDCG=1.00
Q5: What are the early signs that labor is starting?
   Precision=0.33  Recall=0.50  MRR=1.00  nDCG=0.61
Q6: What pre-existing health conditions can make a pregnancy high-risk?
   Precision=0.67  Recall=0.67  MRR=1.00  nDCG=0.70
Q7: How is Group B Streptococcus detected during pregnancy?
   Precision=0.33  Recall=1.00  MRR=1.00  nDCG=1.00
Q8: What is newborn sepsis?
   Precision=0.33  Recall=1.00  MRR=1.00  nDCG=1.00
Q9: What does a high fever with severe headache after childbirth indicate?
   Precision=0.33  Recall=1.00  MRR=1.00  nDCG=1.00
Q10: What should be included in a bir

## Save the retrieval quality evaluation results

In [35]:
# Save the final retrieval evaluation results
results_summary = {
    "k": k,
    "num_questions": len(gold_questions),
    "average_precision_at_k": float(np.mean(all_precision)),
    "average_recall_at_k": float(np.mean(all_recall)),
    "average_mrr": float(np.mean(all_mrr)),
    "average_ndcg_at_k": float(np.mean(all_ndcg)),
}

with open("retrieval_evaluation_results.json", "w") as f:
    json.dump(results_summary, f, indent=4)

print("Saved:", results_summary)

Saved: {'k': 3, 'num_questions': 30, 'average_precision_at_k': 0.3444444444444445, 'average_recall_at_k': 0.9722222222222221, 'average_mrr': 0.95, 'average_ndcg_at_k': 0.9403284847504655}


## Phase 4: LLM Integration

## 4.1 Danger sign keyword list

In [36]:
# This keyword list is drawn directly from the documents in the danger_signs_pregnancy topic
danger_sign_keywords = [
    # From DOC_DS_017 and DOC_DS_018
    "vaginal bleeding", "bleeding",

    # From DOC_DS_019
    "severe headache", "persistent headache", "blurred vision", "seeing spots",

    # From DOC_DS_020
    "convulsions", "seizure", "fits",

    # From DOC_DS_021
    "difficulty breathing", "trouble breathing", "chest pain", "fast heartbeat",

    # From DOC_DS_022
    "severe abdominal pain", "severe stomach pain",

    # From DOC_DS_023
    "high fever", "fever",

    # From DOC_DS_024
    "severe swelling", "sudden swelling", "swelling of the face",

    # From DOC_DS_025
    "reduced fetal movement", "baby is not moving", "baby stopped moving", "no fetal movement",

    # From DOC_DS_026
    "severe vomiting", "persistent vomiting", "can't stop vomiting",

    # From DOC_DS_027
    "fluid leaking", "water broke", "waters have broken",

    # From DOC_DS_028 / DOC_DS_029
    "fainting", "dizziness", "paleness",

    # From DOC_DS_030
    "harm myself", "hurt myself", "harm the baby", "hurt the baby", "thoughts of harming",

    "not kicked", "has not kicked", "hasn't kicked", "not kicking", "stopped kicking",
    "cannot breathe", "can't breathe", "cant breathe", "hard to breathe",
    "short of breath", "shortness of breath", "struggling to breathe",
]

def check_for_danger_signs(user_message):
    """Check if the user's message contains any known danger-sign
    phrase. Returns True and the matched phrase if found, otherwise
    returns False."""
    message_lower = user_message.lower()
    for phrase in danger_sign_keywords:
        if phrase in message_lower:
            return True, phrase
    return False, None

# Test with a few example messages
test_messages = [
    "I have a severe headache and blurred vision",
    "Is it normal to feel tired in the third trimester?",
    "My baby is not moving",
]

for msg in test_messages:
    is_danger, matched = check_for_danger_signs(msg)
    print(f"Message: {msg}")
    print(f"  Danger sign detected: {is_danger} (matched phrase: {matched})")
    print()

Message: I have a severe headache and blurred vision
  Danger sign detected: True (matched phrase: severe headache)

Message: Is it normal to feel tired in the third trimester?
  Danger sign detected: False (matched phrase: None)

Message: My baby is not moving
  Danger sign detected: True (matched phrase: baby is not moving)



## 4.2 Testing with messages that indicate a danger sign but do not match the danger keyword list as it is written

In [37]:
test_messages_2 = [
    "I don't feel my baby moving today",
    "No movement from the baby since morning",
    "I think something is wrong, I can't feel the baby",
]

for msg in test_messages_2:
    is_danger, matched = check_for_danger_signs(msg)
    print(f"Message: {msg}")
    print(f"  Danger sign detected: {is_danger} (matched phrase: {matched})")
    print()

Message: I don't feel my baby moving today
  Danger sign detected: False (matched phrase: None)

Message: No movement from the baby since morning
  Danger sign detected: False (matched phrase: None)

Message: I think something is wrong, I can't feel the baby
  Danger sign detected: False (matched phrase: None)



## 4.3 Try solving the danger keyword list problem using semantic similarity

In [38]:
# Danger keyword list might not always capture a high risk symptom as true
# since I cannot predict the phrasing that the mother will use in her question.
# For this, we can try use semantic smilarity which looks at the meaning.

danger_sign_chunk_ids = [f"DOC_DS_0{n}_0" for n in range(17, 31)]  # DOC_DS_017 to DOC_DS_030
danger_sign_positions = [i for i, c in enumerate(chunk_lookup) if c['chunk_id'] in danger_sign_chunk_ids]

def check_semantic_danger_sign(user_message, similarity_threshold=0.6):
    """Embeds the user's message and checks how close it is to our
    known danger-sign chunks. If it's close enough (above the
    threshold), we treat this as a POSSIBLE danger sign, even if no
    exact keyword matched."""
    msg_embedding = model.encode([user_message])[0]

    best_similarity = -1
    best_match_id = None
    for pos in danger_sign_positions:
        chunk_embedding = np.array(chunked_documents[pos]['embedding'])
        # Cosine similarity which measures how closely two embeddings
        # point in the same direction. 1.0 = identical meaning,
        # 0 = unrelated, negative = opposite meaning.
        similarity = np.dot(msg_embedding, chunk_embedding) / (
            np.linalg.norm(msg_embedding) * np.linalg.norm(chunk_embedding)
        )
        if similarity > best_similarity:
            best_similarity = similarity
            best_match_id = chunk_lookup[pos]['chunk_id']

    is_danger = best_similarity >= similarity_threshold
    return is_danger, best_similarity, best_match_id

# Test it on the three messages that the keyword check missed
for msg in test_messages_2:
    is_danger, similarity, matched_id = check_semantic_danger_sign(msg)
    print(f"Message: {msg}")
    print(f"  Semantic danger sign detected: {is_danger} (similarity: {similarity:.3f}, closest: {matched_id})")
    print()

Message: I don't feel my baby moving today
  Semantic danger sign detected: False (similarity: 0.557, closest: DOC_DS_025_0)

Message: No movement from the baby since morning
  Semantic danger sign detected: True (similarity: 0.617, closest: DOC_DS_025_0)

Message: I think something is wrong, I can't feel the baby
  Semantic danger sign detected: False (similarity: 0.472, closest: DOC_DS_030_0)



## 4.4 Try with messages showing no danger signs

In [39]:
safe_messages = [
    "Is it normal to feel tired in the third trimester?",
    "What foods should I avoid during pregnancy?",
    "How much weight should I gain by month 6?",
]

for msg in safe_messages:
    is_danger, similarity, matched_id = check_semantic_danger_sign(msg, similarity_threshold=0.5)
    print(f"Message: {msg}")
    print(f"  Detected at threshold 0.5: {is_danger} (similarity: {similarity:.3f}, closest: {matched_id})")
    print()

Message: Is it normal to feel tired in the third trimester?
  Detected at threshold 0.5: False (similarity: 0.447, closest: DOC_DS_028_0)

Message: What foods should I avoid during pregnancy?
  Detected at threshold 0.5: False (similarity: 0.373, closest: DOC_DS_020_0)

Message: How much weight should I gain by month 6?
  Detected at threshold 0.5: False (similarity: 0.189, closest: DOC_DS_024_0)



## 4.5 Set a threshold to capture danger signs using both keyword and semantic similarity search

In [40]:
# Final threshold, chosen based on testing against both danger messages and safe messages
SAFETY_SIMILARITY_THRESHOLD = 0.45

In [41]:
# Combined safety check: keyword match or semantic similarity
SAFETY_SIMILARITY_THRESHOLD = 0.45 # anything with a score of 0.45 or higher indicates a danger sign

def check_danger_sign_combined(user_message): # looks at the mother's question to see if it indicates danger
    keyword_hit, matched_phrase = check_for_danger_signs(user_message) # looks at the question to see if matches any danger sign in the keyword list
    semantic_hit, similarity, matched_chunk = check_semantic_danger_sign( # Performs semantic similarity search
        user_message, similarity_threshold=SAFETY_SIMILARITY_THRESHOLD
    )

    is_danger = bool(keyword_hit or semantic_hit)

    return {
        "is_danger": is_danger,
        "keyword_match": matched_phrase,
        "semantic_similarity": round(float(similarity), 3),
        "semantic_closest_chunk": matched_chunk,
    }

all_test_messages = [
    "I have a severe headache and blurred vision",
    "Is it normal to feel tired in the third trimester?",
    "I don't feel my baby moving today",
    "How much weight should I gain by month 6?",
    "No movement from the baby since morning",
    "I think something is wrong, I can't feel the baby",
    "What foods should I avoid during pregnancy?",
]

for msg in all_test_messages:
    result = check_danger_sign_combined(msg)
    print(f"Message: {msg}")
    print(f"  {result}")
    print()

Message: I have a severe headache and blurred vision
  {'is_danger': True, 'keyword_match': 'severe headache', 'semantic_similarity': 0.497, 'semantic_closest_chunk': 'DOC_DS_019_0'}

Message: Is it normal to feel tired in the third trimester?
  {'is_danger': False, 'keyword_match': None, 'semantic_similarity': 0.447, 'semantic_closest_chunk': 'DOC_DS_028_0'}

Message: I don't feel my baby moving today
  {'is_danger': True, 'keyword_match': None, 'semantic_similarity': 0.557, 'semantic_closest_chunk': 'DOC_DS_025_0'}

Message: How much weight should I gain by month 6?
  {'is_danger': False, 'keyword_match': None, 'semantic_similarity': 0.189, 'semantic_closest_chunk': 'DOC_DS_024_0'}

Message: No movement from the baby since morning
  {'is_danger': True, 'keyword_match': None, 'semantic_similarity': 0.617, 'semantic_closest_chunk': 'DOC_DS_025_0'}

Message: I think something is wrong, I can't feel the baby
  {'is_danger': True, 'keyword_match': None, 'semantic_similarity': 0.472, 'sema

## 4.6 Set up Microsoft Phi-3 Mini as the LLM

In [42]:
# Install libraries
!pip install transformers accelerate torch -q

# Load model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "microsoft/Phi-3-mini-4k-instruct"

# The tokenizer converts text into the numeric format the model actually understands, and converts the model's numeric output
# back into readable text.
llm_tokenizer = AutoTokenizer.from_pretrained(model_name)

# The model itself
llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print("Phi-3-mini loaded successfully.")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

... (4 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Phi-3-mini loaded successfully.


## 4.7 Function that generates an answer only from relevant retrieved document chunks

In [43]:
# Generate a grounded answer using retrieved chunks
def generate_answer(question, k=3, max_new_tokens=200):
    # Step A: Retrieve the top-k most relevant chunks.
    retrieved_ids = get_retrieved_ids(question, k)
    retrieved_chunks = [c for c in chunk_lookup if c['chunk_id'] in retrieved_ids]

    # Step B: Combine the retrieved chunks into one block of context text, so the model can read all of them together.
    context_text = "\n\n".join([c['content'] for c in retrieved_chunks])

    # Step C: Build the actual prompt. Only use the provided context, and if the information is not there, say it.
    prompt = f"""You are a helpful maternal health assistant. Answer the question using ONLY the information in the context below. If the answer is not in the context, say you don't have that information.

Context:
{context_text}

Question: {question}

Answer:"""

    # Step D: Phi-3-mini expects input in a specific "chat" format (a list of role/content messages), rather than plain text.
    messages = [{"role": "user", "content": prompt}]
    inputs = llm_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(llm_model.device)

    # Step E: Generate the answer.
    outputs = llm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    # Step F: Keep only the text generated.
    input_length = inputs['input_ids'].shape[1]
    generated_text = llm_tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    # Step G: Return the question, the final answer , and which chunks were
    # actually used, so we can verify the answer is properly is correct.
    return {
        "question": question,
        "answer": generated_text.strip(),
        "retrieved_chunk_ids": retrieved_ids,
    }

## 4.8 Test with three real maternal health question

In [44]:
# Try it on one of your real gold questions, so we can check the
# answer is both fluent and grounded in the retrieved chunks.
result = generate_answer("What are the danger signs I should watch for during pregnancy?")

print("Question:", result['question'])
print("\nAnswer:", result['answer'])
print("\nBased on chunks:", result['retrieved_chunk_ids'])

Question: What are the danger signs I should watch for during pregnancy?

Answer: The danger signs you should watch for during pregnancy are vaginal bleeding, severe or persistent headache, blurred vision, convulsions or fits, difficulty breathing or chest pain, severe abdominal pain, high fever, severe swelling of the face or hands, reduced or absent fetal movement, severe nausea and vomiting, and fluid leaking from the vagina before the due date. Additionally, thoughts of harming oneself or the baby during pregnancy are serious warning signs that require immediate attention.

Based on chunks: ['DOC_DS_017_0', 'DOC_HRP_181_0', 'DOC_DS_030_0']


In [45]:
result = generate_answer("How should I prepare myself as I head towards labour?")

print("Question:", result['question'])
print("\nAnswer:", result['answer'])
print("\nBased on chunks:", result['retrieved_chunk_ids'])

Question: How should I prepare myself as I head towards labour?

Answer: As you head towards labor, you should prepare yourself by gathering essential childbirth supplies ahead of time, including clean clothes and soft blankets for drying, warming, and wrapping the newborn baby. Additionally, discuss and review your birth preparedness plan with a midwife or healthcare provider during regular antenatal clinic visits. Reviewing and updating the plan as the expected due date gets closer ensures that all arrangements for travel, funds, and emergency contacts remain clear and ready.

Based on chunks: ['DOC_BP_132_0', 'DOC_BP_133_0', 'DOC_BP_123_0']


In [46]:
result = generate_answer("How and where can I get professional help with issues to do with maternal health in Kenya?")

print("Question:", result['question'])
print("\nAnswer:", result['answer'])
print("\nBased on chunks:", result['retrieved_chunk_ids'])

Question: How and where can I get professional help with issues to do with maternal health in Kenya?

Answer: For professional help with issues related to maternal health in Kenya, you can contact the following organizations:

1. Marie Stopes Kenya: They offer reproductive health guidance at 0800 720 005.
2. One2One Liverpool service: They provide youth and maternal reproductive health information at 1190.
3. NASCOP: They provide care and advice regarding HIV, AIDS, and sexually transmitted infections at 0800 724 848.

Based on chunks: ['DOC_HRP_196_0', 'DOC_HRP_195_0', 'DOC_HRP_194_0']


## 4.9 Try with a question that does not involve maternal health

In [47]:
result = generate_answer("What is the capital city of Kenya?")
print("Answer:", result['answer'])

Answer: I'm sorry, but I don't have that information.


## 4.10 Danger Sign Detection Layer

In [48]:
def get_full_response(user_message, k=3, max_new_tokens=200):
    # Step 1: safety check, using the combined keyword + semantic function built.
    safety_result = check_danger_sign_combined(user_message)

    # Step 2: if a danger sign was detected, return the fixed alert immediately
    if safety_result['is_danger']:
        alert_message = (
            "This sounds like it could be a serious warning sign during "
            "pregnancy. Please go to your nearest hospital or health facility "
            "immediately, or call your emergency healthcare provider. "
            "Do not wait to see if it gets better on its own."
        )
        return {
            "user_message": user_message,
            "final_answer": alert_message,
            "danger_detected": True,
            "safety_details": safety_result,
            "retrieved_chunk_ids": [],  # nothing was retrieved, since we skipped that step
        }

    # Step 3: If no danger signs exist, retrieve relevant chunks and generate a context aware answer answer.
    answer_result = generate_answer(user_message, k=k, max_new_tokens=max_new_tokens)

    return {
        "user_message": user_message,
        "final_answer": answer_result['answer'],
        "danger_detected": False,
        "safety_details": safety_result,
        "retrieved_chunk_ids": answer_result['retrieved_chunk_ids'],
    }

In [49]:
test_cases = [
    "I have severe bleeding and can't feel the baby moving",   # TP scenario hence should trigger an alert
    "Why should I take supplements during pregnancy?",         # TN scenario hence should not trigger alert
    "I've had a mild headache on and off today",               # borderline scenario
]

for msg in test_cases:
    result = get_full_response(msg)
    print(f"User: {msg}")
    print(f"Danger detected: {result['danger_detected']}")
    print(f"Response:\n{result['final_answer']}")
    print("\n")

User: I have severe bleeding and can't feel the baby moving
Danger detected: True
Response:
This sounds like it could be a serious warning sign during pregnancy. Please go to your nearest hospital or health facility immediately, or call your emergency healthcare provider. Do not wait to see if it gets better on its own.


User: Why should I take supplements during pregnancy?
Danger detected: False
Response:
You should take supplements during pregnancy to support rapid fetal cell division, enable neural tube development, expand maternal red blood cell mass to prevent anemia, protect maternal bone health, and reduce the risks of pre-eclampsia, low birth weight, and congenital anomalies. Taking iron, folic acid, and calcium supplements helps fill nutritional gaps, prevents maternal anemia and bone loss, and lowers the risk of birth defects and pregnancy complications.


User: I've had a mild headache on and off today
Danger detected: False
Response:
I'm sorry, but I don't have information

## Phase 5: Testing with open and closed ended questions

## 5.1 Scenario testing (TP, TN and Bordeline)

In [50]:
# Scenario testing (TP / TN / Borderline)
safety_scenarios = [
    # True Positive: clear danger signs, system should trigger an alert
    {"message": "I have severe bleeding and can't feel the baby moving", "category": "True Positive"},
    {"message": "I have a severe headache and blurred vision", "category": "True Positive"},
    {"message": "I just had a seizure", "category": "True Positive"},
    {"message": "I'm having real difficulty breathing and chest pain", "category": "True Positive"},
    {"message": "The pain in my stomach is severe and won't go away", "category": "True Positive"},
    {"message": "I have a very high fever and feel extremely weak", "category": "True Positive"},
    {"message": "My face and hands are swelling a lot nowadays", "category": "True Positive"},
    {"message": "No movement from the baby at all since this morning", "category": "True Positive"},
    {"message": "I can't stop vomiting and feel dehydrated", "category": "True Positive"},
    {"message": "Sometimes I think about hurting myself and the baby", "category": "True Positive"},

    # True Negative: normal, no danger, system should not trigger an alert
    {"message": "What foods should I avoid during pregnancy?", "category": "True Negative"},
    {"message": "Is it normal to feel tired in the third trimester?", "category": "True Negative"},
    {"message": "How much weight should I gain by month 6?", "category": "True Negative"},
    {"message": "What exercises are safe during pregnancy?", "category": "True Negative"},
    {"message": "How many antenatal visits should I attend?", "category": "True Negative"},
    {"message": "What vitamins should I be taking?", "category": "True Negative"},
    {"message": "When should I start preparing my hospital bag?", "category": "True Negative"},
    {"message": "Is it safe to travel during pregnancy?", "category": "True Negative"},
    {"message": "What is a midwife-led continuity-of-care model?", "category": "True Negative"},
    {"message": "How should I care for my baby's umbilical cord stump?", "category": "True Negative"},

    # Borderline: mild, or early-stage symptoms
    {"message": "I've had a mild headache on and off today", "category": "Borderline"},
    {"message": "My ankles are a bit swollen by the end of the day", "category": "Borderline"},
    {"message": "I feel a little more tired than usual and slightly dizzy", "category": "Borderline"},
    {"message": "The baby has been less active than usual today", "category": "Borderline"},
    {"message": "I've had mild cramping on and off since this morning", "category": "Borderline"},
    {"message": "I feel a bit nauseous today, more than normal", "category": "Borderline"},
    {"message": "I have a slight fever, nothing too bad", "category": "Borderline"},
    {"message": "My vision felt a little blurry for a moment earlier", "category": "Borderline"},
    {"message": "I've been feeling unusually anxious and overwhelmed lately", "category": "Borderline"},
    {"message": "I noticed a small amount of spotting today", "category": "Borderline"},
]

print("Total scenarios:", len(safety_scenarios))

Total scenarios: 30


In [51]:
# Run every scenario through the full pipeline and record results

safety_test_results = []

for scenario in safety_scenarios:
    result = get_full_response(scenario['message'])
    safety_test_results.append({
        "message": scenario['message'],
        "category": scenario['category'],
        "danger_detected": result['danger_detected'],
        "response_preview": result['final_answer'][:100],
    })

# Print results
for r in safety_test_results:
    print(f"[{r['category']:14}] detected={r['danger_detected']}  -  {r['message']}")

[True Positive ] detected=True  -  I have severe bleeding and can't feel the baby moving
[True Positive ] detected=True  -  I have a severe headache and blurred vision
[True Positive ] detected=True  -  I just had a seizure
[True Positive ] detected=True  -  I'm having real difficulty breathing and chest pain
[True Positive ] detected=True  -  The pain in my stomach is severe and won't go away
[True Positive ] detected=True  -  I have a very high fever and feel extremely weak
[True Positive ] detected=False  -  My face and hands are swelling a lot nowadays
[True Positive ] detected=True  -  No movement from the baby at all since this morning
[True Positive ] detected=True  -  I can't stop vomiting and feel dehydrated
[True Positive ] detected=True  -  Sometimes I think about hurting myself and the baby
[True Negative ] detected=False  -  What foods should I avoid during pregnancy?
[True Negative ] detected=False  -  Is it normal to feel tired in the third trimester?
[True Negative ] de

## Save the results

In [52]:
import json
with open("safety_scenario_test_results.json", "w") as f:
    json.dump(safety_test_results, f, indent=4)
print("Saved the results.")

Saved the results.


## 5.2 Testing with the 30 gold questions used when testing retrieval in Phase 3

In [53]:
# Generate answers for all 30 gold questions
# This runs each gold question through the full system (retrieval + LLM generation) and saves the actual answer produced.

closed_question_results = []

for item in gold_questions:
    result = generate_answer(item['question'])
    closed_question_results.append({
        "question": item['question'],
        "expected_chunk_ids": item['relevant_chunk_ids'],
        "generated_answer": result['answer'],
        "retrieved_chunk_ids": result['retrieved_chunk_ids'],
    })

print("Generated answers for", len(closed_question_results), "questions.")

Generated answers for 30 questions.


In [54]:
# Print each question + answer for manual scoring
for i, r in enumerate(closed_question_results, start=1):
    print(f"Q{i}: {r['question']}")
    print(f"Expected source chunk(s): {r['expected_chunk_ids']}")
    print(f"Answer: {r['generated_answer']}")
    print()

Q1: How many antenatal care visits does WHO recommend during pregnancy?
Expected source chunk(s): ['DOC_ANC_002_0']
Answer: The WHO recommends a minimum of eight contacts between a pregnant woman and a health provider during pregnancy.

Q2: How is preeclampsia diagnosed?
Expected source chunk(s): ['DOC_PRE_032_0']
Answer: Preeclampsia is clinically diagnosed by the onset of hypertension (blood pressure ≥140/90 mmHg) and proteinuria (protein in the urine) or end-organ dysfunction after 20 weeks of gestation.

Q3: What causes gestational diabetes?
Expected source chunk(s): ['DOC_GDM_066_0']
Answer: Gestational diabetes is caused by the pancreas not being able to produce enough insulin to meet the increased demands of pregnancy. During gestation, hormones released by the placenta cause insulin resistance by blocking how the mother's body uses insulin, leading to elevated blood sugar levels if the body cannot compensate.

Q4: How common is intimate partner violence during pregnancy?
Expect

## 5.3 Testing with other open ended questions (More natural and realistic)

In [55]:
# Open ended question testing
# It checks whether the system can respond well to questions that are more natural and what a mother would normally ask

open_ended_questions = [
    # Informational
    "What should I know about staying healthy during my third trimester?",
    "How can I have a smooth and healthy pregnancy overall?",

    # Emotional
    "I'm really scared about labor, what should I expect?",
    "This is my first pregnancy and I'm nervous about everything, where do I start?",

    # Combined concerns
    "I'm 28 weeks pregnant and have been having mild swelling and trouble sleeping, is this normal?",
    "I've been feeling really tired and a bit nauseous again, and I'm not sure if I'm eating enough, what should I do?",

    # Profile blended (medical history, gestational period and symptoms present in the question)
    "I'm in my first trimester and keep feeling nauseous, what can help?",
    "I'm 36 weeks along now, what should I be preparing for at this stage?",
    "I just gave birth a few days ago and I'm exhausted and overwhelmed, is that normal?",
    "I have a history of high blood pressure and I'm newly pregnant, what should I be careful about?",
]

print("Total open-ended questions:", len(open_ended_questions))

Total open-ended questions: 10


In [56]:
# Generate answers for all 10 open ended questions
open_ended_results = []

for q in open_ended_questions:
    result = get_full_response(q)
    open_ended_results.append({
        "question": q,
        "danger_detected": result['danger_detected'],
        "answer": result['final_answer'],
        "retrieved_chunk_ids": result['retrieved_chunk_ids'],
    })

for i, r in enumerate(open_ended_results, start=1):
    print(f"Q{i}: {r['question']}")
    print(f"Danger detected: {r['danger_detected']}")
    print(f"Answer: {r['answer']}")
    print()

Q1: What should I know about staying healthy during my third trimester?
Danger detected: False
Answer: During your third trimester, it's important to consume an additional three hundred to four hundred fifty calories daily, representing a ten to fifteen percent increase over your pre-pregnancy baseline needs. This increase in energy intake is necessary to fuel healthy fetal tissue development and maternal weight gain. Additionally, maintaining a balanced diet that includes fruits, vegetables, whole grains, dairy products, and protein-rich foods is crucial to ensure adequate intake of essential nutrients for both maternal health and fetal development.

Q2: How can I have a smooth and healthy pregnancy overall?
Danger detected: False
Answer: To have a smooth and healthy pregnancy overall, it is important to consume a balanced diet that includes fruits, vegetables, whole grains, dairy products, and protein-rich foods. Eating from multiple food groups such as fresh vegetables, fruits, whol

## Save the results for 5.2 and 5.3

In [57]:
import json

with open("5_2_gold_question_test_results.json", "w") as f:
    json.dump(closed_question_results, f, indent=4)

with open("5_3_open_ended_test_results.json", "w") as f:
    json.dump(open_ended_results, f, indent=4)

print("Saved 5.2 results:", len(closed_question_results), "questions")
print("Saved 5.3 results:", len(open_ended_results), "questions")

Saved 5.2 results: 30 questions
Saved 5.3 results: 10 questions


# Part 3: The Kiswahili layer

**What it does:** lets a mother write in Kiswahili and get an answer in Kiswahili, **without changing** the English pipeline.

## Phase 6: Add translation model and review the full Kiswahili layer works

## 6.1 Keyword fix and extra danger keywords

In [58]:
import re

def check_for_danger_signs(user_message):
    """Match a danger phrase only at the start of a word ('fits' no longer matches 'benefits')."""
    message_lower = user_message.lower()
    for phrase in danger_sign_keywords:
        if re.search(r"\b" + re.escape(phrase), message_lower):
            return True, phrase
    return False, None

danger_sign_keywords += [
    "vision changes", "visual changes", "visual disturbances", "blurry",
    "spotting", "spots of blood", "blood spots", "a little blood", "small amount of blood",
]

print(check_danger_sign_combined("What are the benefits of folic acid?")["keyword_match"])   # expect: None
print(check_for_danger_signs("I have had seizures"))                                          # expect: a match

None
(True, 'seizure')


## 6.2 Translation model

In [59]:
import re, time
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MT_NAME = "facebook/nllb-200-distilled-600M"   # license: CC-BY-NC 4.0

mt_device = "cuda" if torch.cuda.is_available() else "cpu"
mt_dtype = torch.float16 if mt_device == "cuda" else torch.float32

mt_tokenizer = AutoTokenizer.from_pretrained(MT_NAME)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_NAME, torch_dtype=mt_dtype).to(mt_device)
mt_model.eval()
mt_model.generation_config.max_length = None

print("Translation model loaded on", mt_device)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Translation model loaded on cuda


In [60]:
LANG_CODES = {"en": "eng_Latn", "sw": "swh_Latn"}

def split_sentences(text):
    # The model works best one sentence at a time, so long answers are split first
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p for p in parts if p]

def translate(text, src, tgt, num_beams=4, max_new_tokens=200):
    """Translate text between 'en' and 'sw'."""
    if src == tgt or not text.strip():
        return text

    mt_tokenizer.src_lang = LANG_CODES[src]
    target_id = mt_tokenizer.convert_tokens_to_ids(LANG_CODES[tgt])

    sentences = split_sentences(text)
    translated = []
    for i in range(0, len(sentences), 8):
        batch = sentences[i:i + 8]
        inputs = mt_tokenizer(
            batch, return_tensors="pt", padding=True, truncation=True, max_length=256
        ).to(mt_model.device)
        with torch.no_grad():
            generated = mt_model.generate(
                **inputs,
                forced_bos_token_id=target_id,
                num_beams=num_beams,
                max_new_tokens=max_new_tokens,
            )
        translated.extend(mt_tokenizer.batch_decode(generated, skip_special_tokens=True))
    return " ".join(translated)

def numbers_preserved(source, translated):
    """Medical text lives and dies by its numbers (doses, weeks, hotlines)."""
    find = lambda s: sorted(re.findall(r'\d+(?:[.,]\d+)?', s))
    return find(source) == find(translated), find(source), find(translated)

**Quick test:** English to Kiswahili and back, to see how faithful the translation is.

In [61]:
for en in ["Please go to your nearest hospital or health facility immediately. Do not wait to see if it gets better on its own.",
           "WHO recommends a minimum of eight antenatal contacts, with the first contact by 12 weeks of gestation.",
           "Is it normal to feel tired in the third trimester?"]:
    sw = translate(en, "en", "sw")
    back = translate(sw, "sw", "en")
    print("EN  :", en)
    print("SW  :", sw)
    print("BACK:", back)
    print("Numbers kept:", numbers_preserved(en, back)[0], "\n")

EN  : Please go to your nearest hospital or health facility immediately. Do not wait to see if it gets better on its own.
SW  : Tafadhali tembelea hospitali au kituo cha afya kilicho karibu nawe mara moja. Usisubiri ili uone ikiwa itaboresha.
BACK: Please visit the nearest hospital or health center immediately. Do not wait to see if it improves.
Numbers kept: True 

EN  : WHO recommends a minimum of eight antenatal contacts, with the first contact by 12 weeks of gestation.
SW  : Shirika la Afya Ulimwenguni (WHO) hupendekeza angalau mawasiliano nane kabla ya kuzaa, na mawasiliano ya kwanza hufanywa baada ya majuma 12 ya ujauzito.
BACK: The World Health Organization (WHO) recommends at least eight prenatal contacts, the first being after 12 weeks of pregnancy.
Numbers kept: True 

EN  : Is it normal to feel tired in the third trimester?
SW  : Je, ni jambo la kawaida kuhisi uchovu katika miezi mitatu?
BACK: Is it normal to feel tired within three months?
Numbers kept: True 



## 6.3 Build glossary

*What it does:** the knowledge base uses specific English terms ("umbilical cord stump", "preeclampsia"). Each line below is `kiswahili term = English term`. `apply_glossary()` swaps the Kiswahili terms for the English ones **before** the question goes to the translator.

**Why:** without it, *kikonyo cha kitovu* is translated as "central lining" and retrieval fails. The glossary is the biggest single improvement in Part 4.

In [62]:
import re

# Write the glossary

glossary_text = """# --- from my own test questions ---
kikonyo cha kitovu = umbilical cord stump
kikonyo cha kitovu cha mtoto = umbilical cord stump
kikonyo = umbilical cord stump
kitovu = umbilical cord
kifafa cha mimba = preeclampsia
kifafa cha mimba kilichokomaa = eclampsia
shinikizo kubwa la damu la ujauzito = preeclampsia
shinikizo la damu = high blood pressure

# --- conditions & infections ---
kisukari cha ujauzito = gestational diabetes
kisukari cha mimba = gestational diabetes
maambukizi ya damu kwa mtoto mchanga = newborn sepsis
sepsis ya mtoto mchanga = newborn sepsis
ukatili wa majumbani = intimate partner violence
ukatili kutoka kwa mwenza = intimate partner violence
ukatili wa kijinsia = intimate partner violence
listeriosis = listeriosis
maambukizi ya listeriosis = listeriosis
surua ya ujerumani = rubella
rubella = rubella
group b streptococcus = group b streptococcus
bakteria wa group b streptococcus = group b streptococcus

# --- symptoms & physiological conditions ---
kiungulia = heartburn
kufunga choo = constipation
pata choo kwa shida = constipation
kukaza kwa misuli ya miguu = leg cramps
kutetemeka kwa misuli ya miguu = leg cramps
homa kali = fever
kichwa kuuma sana = severe headache
kuvimba kwa miguu na vifundo vya miguu = swelling of the feet and ankles
kuvimba kwa miguu = swelling
kuvuja damu kwa wingi baada ya kujifungua = postpartum hemorrhage
kuvuja damu baada ya kujifungua = postpartum hemorrhage
kifafa = seizures
kutetemeka = seizures
matatizo ya kupumua = breathing problems
dalili za mfumo wa mkojo = difficulty when urinating
madoa madogo ya damu = spotting
matone ya damu = spotting
madoa madogo ya damu = spotting
fifia = blurred vision
kuona giza = blurred vision
kifua kinabana = chest pains
maji ya uzazi yametoka = waters have broken

# --- care, plans & interventions ---
kliniki za uja uzito = antenatal care
kliniki ya ujauzito = antenatal care
huduma za kabla ya kujifungua = antenatal care
mpango wa kujiandaa na uzazi = birth preparedness
mpango wa uzazi = birth preparedness
madini ya chuma = iron
vidonge vya iron = iron
vidonge vya folic acid = folic acid
folic acid = folic acid
kadi za kliniki = case notes
rekodi za kliniki = case notes
kadi ya hospitali = case notes
huduma endelevu za uzazi zinazoongozwa na wakunga = midwife-led continuity-of-care
huduma za wakunga = midwife-led continuity-of-care
wakunga = midwife
mtoto aliyezaliwa kabla ya wakati = preterm
mtoto njiti = preterm
njiti = preterm
uzito mdogo = low birth weight
watoto wenye uzito mdogo = low birth weight
chanjo = vaccines
nambari ya simu ya dharura = hotlines
namba ya dharura = hotlines

# --- trimesters ---
trimesta ya kwanza = first trimester
trimesta ya pili = second trimester
trimesta ya tatu = third trimester
mwezi wa kwanza wa ujauzito = first trimester
mwezi wa pili wa ujauzito = second trimester
mwezi wa tatu wa ujauzito = third trimester

"""

GLOSSARY = {}
for line in glossary_text.strip().split("\n"):
    line = line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    sw_term, en_term = line.split("=", 1)
    GLOSSARY[sw_term.strip().lower()] = en_term.strip()

def apply_glossary(text):
    # Longest phrase first. The term must start at the beginning of a word.
    for sw_term in sorted(GLOSSARY, key=len, reverse=True):
        text = re.sub(r"\b" + re.escape(sw_term), GLOSSARY[sw_term], text, flags=re.IGNORECASE)
    return text

In [63]:
q = "Kikonyo cha kitovu cha mtoto kinapaswa kuhudumiwaje?"
print("Before:", q)
print("After :", apply_glossary(q))

Before: Kikonyo cha kitovu cha mtoto kinapaswa kuhudumiwaje?
After : umbilical cord stump kinapaswa kuhudumiwaje?


## 6.4 Kiswahili danger sign detection

In [64]:
# Kiswahili danger phrases
sw_danger_text = """
kutokwa na damu
damu nyingi
maumivu makali ya kichwa
kuona kizinguzungu
kutoona vizuri
degedege
kifafa
kufifia
kupumua kwa shida
maumivu ya kifua
maumivu makali ya tumbo
homa kali
kuvimba uso
mtoto hachezi
mtoto wangu hasongi
kutapika sana
maji yamepasuka
maji yamevunjika
kuzimia
kizunguzungu
kujiumiza
kumuumiza mtoto
mtoto hasongi
kifua kinabana
maji ya uzazi yametoka
"""
SW_DANGER_PHRASES = [l.strip().lower() for l in sw_danger_text.strip().split("\n") if l.strip()]

# Kiswahili verbs change shape, so these stems match inside the word
SW_STEMS = [
    "tapika sana", "tapika mfululizo", "tapika kila",     # natapika sana, anatapika sana...
    "tokwa na damu", "damu nyingi",                        # natokwa na damu, anatokwa na damu...
    "madoa madogo ya damu", "madoa ya damu", "matone ya damu", "fifia", "ukungu",   # spotting, blurred vision

]

def check_sw_danger(text):
    t = text.lower()
    for phrase in SW_DANGER_PHRASES:
        if re.search(r"\b" + re.escape(phrase), t):
            return True, phrase
    for stem in SW_STEMS:
        if stem in t:
            return True, stem
    return False, None

def check_original_danger(text):
    # Check both phrase lists on the original text, in case she mixes languages
    hit, phrase = check_sw_danger(text)
    if hit:
        return True, phrase
    return check_for_danger_signs(text)

# Swelling of the face or hands (both languages)
def swelling_rule(text):
    t = text.lower()
    swell = re.search(r"swell|swollen|puffy|vimevimba|umevimba|kuvimba|navimba|nimevimba", t)
    part = re.search(r"\b(face|hands|uso|mikono)\b", t)
    return bool(swell and part)

def danger_only(msg, language):
    hit, phrase = check_original_danger(msg)
    q_en = translate(apply_glossary(msg), "sw", "en") if language == "sw" else msg
    eng = check_danger_sign_combined(q_en)
    rule = swelling_rule(msg) or swelling_rule(q_en)
    if rule and not phrase:
        phrase = "swelling of face/hands (rule)"
    return (hit or eng["is_danger"] or rule), phrase, q_en, eng["semantic_similarity"]

## 6.5 Alert texts and the full multilingual response

In [65]:
# Fixed alert texts
ALERT_EN = (
    "This sounds like it could be a serious warning sign during pregnancy. "
    "Please go to your nearest hospital or health facility immediately, or call your "
    "emergency healthcare provider. Do not wait to see if it gets better on its own."
)
# Alert message in Kiswahili
ALERT_SW = (
    "Hii inaweza kuwa ishara kubwa ya hatari wakati wa ujauzito. "
    "Tafadhali nenda hospitali au kituo cha afya kilicho karibu nawe mara moja, "
    "au mpigie mtoa huduma wako wa afya wa dharura. "
    "Usisubiri kuona kama itapona yenyewe."
)

# A softer line added to a normal answer when only the meaning check was unsure
CAUTION_EN = "If you are having symptoms right now, please see a health worker."
CAUTION_SW = "Ikiwa unapata dalili sasa hivi, tafadhali muone mhudumu wa afya."

# Fixes for words the translator gets wrong when it turns an English ANSWER into Kiswahili.
# Add a new pair whenever you find a wrong word. Left = wrong, right = correct.
ANSWER_FIXES_SW = {
    "mjomba": "mkunga",                                              # midwife
    "tarehe inayotarajiwa ya kukamilika": "tarehe ya kujifungua",   # due date
}

def fix_answer_sw(text):
    for wrong, right in ANSWER_FIXES_SW.items():
        # Whole words only, and keep a capital letter if the word started a sentence
        def swap(match, right=right):
            return right.capitalize() if match.group(0)[0].isupper() else right
        text = re.sub(r"\b" + re.escape(wrong) + r"\b", swap, text, flags=re.IGNORECASE)
    return text

def trim_to_last_sentence(text):
    """If the answer was cut off in the middle of a sentence, remove the unfinished part."""
    text = text.strip()
    if text.endswith((".", "!", "?")):
        return text
    cut = max(text.rfind("."), text.rfind("!"), text.rfind("?"))
    return text[:cut + 1] if cut > 0 else text

def get_full_response_multilingual(user_message, language="en", k=3, max_new_tokens=300):
    alert = ALERT_SW if language == "sw" else ALERT_EN
    caution = CAUTION_SW if language == "sw" else CAUTION_EN

    # 1. Danger check on the original text (Kiswahili phrases, English keywords)
    original_hit, original_phrase = check_original_danger(user_message)

    # 2. Translate the question to English if she wrote in Kiswahili
    question_en = (translate(apply_glossary(user_message), "sw", "en")
                   if language == "sw" else user_message)

    # 3. Danger check on the English text (keywords + similarity), and the swelling rule
    english_check = check_danger_sign_combined(question_en)
    swelling_hit = swelling_rule(user_message) or swelling_rule(question_en)

    # RED ALERT only when something definite fired: a keyword, a phrase, or the swelling rule
    definite = original_hit or bool(english_check["keyword_match"]) or swelling_hit
    # CAUTION when only the meaning (similarity) check was unsure
    unsure = english_check["is_danger"] and not definite

    if definite:
        return {
            "language": language, "user_message": user_message, "question_en": question_en,
            "danger_detected": True,
            "danger_from": original_phrase or english_check["keyword_match"] or "swelling rule",
            "final_answer": alert, "answer_en": None, "retrieved_chunk_ids": [],
        }

    # 4. Normal path: your existing retrieval and Phi-3 generation, in English
    result = generate_answer(question_en, k=k, max_new_tokens=max_new_tokens)
    answer_en = trim_to_last_sentence(result["answer"])

    # 5. Translate the answer back if she asked in Kiswahili, then fix known wrong words
    final_answer = fix_answer_sw(translate(answer_en, "en", "sw")) if language == "sw" else answer_en

    # 6. Add the caution line if only the meaning check was unsure
    if unsure:
        final_answer = final_answer + "\n\n" + caution

    return {
        "language": language, "user_message": user_message, "question_en": question_en,
        "danger_detected": False, "danger_from": None, "caution": unsure,
        "final_answer": final_answer, "answer_en": answer_en,
        "retrieved_chunk_ids": result["retrieved_chunk_ids"],
    }

**Quick test:** Using six messages

In [66]:
tests = [
    ("sw", "Nina maumivu makali ya kichwa na ninaona kizunguzungu"),    # expect: alert
    ("sw", "Ni vyakula gani napaswa kuepuka wakati wa ujauzito?"),      # expect: normal answer
    ("sw", "Je, ni kawaida kuhisi uchovu katika trimesta ya tatu?"),    # expect: normal answer
    ("en", "I have severe bleeding and the baby is not moving"),        # expect: alert
    ("en", "What foods should I avoid during pregnancy?"),              # expect: normal answer
    ("sw", "Mtoto wangu hasongi leo, nina wasiwasi"),                   # expect: alert
]

for lang, msg in tests:
    r = get_full_response_multilingual(msg, language=lang)
    print(f"[{lang}] {msg}")
    print("   question in English:", r["question_en"])
    print("   danger:", r["danger_detected"], "|", r["danger_from"])
    print("   answer:", r["final_answer"])
    print()

[sw] Nina maumivu makali ya kichwa na ninaona kizunguzungu
   question in English: I have severe headaches and feel dizzy
   danger: True | maumivu makali ya kichwa
   answer: Hii inaweza kuwa ishara kubwa ya hatari wakati wa ujauzito. Tafadhali nenda hospitali au kituo cha afya kilicho karibu nawe mara moja, au mpigie mtoa huduma wako wa afya wa dharura. Usisubiri kuona kama itapona yenyewe.

[sw] Ni vyakula gani napaswa kuepuka wakati wa ujauzito?
   question in English: What foods should I avoid during pregnancy?
   danger: False | None
   answer: Naomba radhi, lakini sina habari hiyo.

[sw] Je, ni kawaida kuhisi uchovu katika trimesta ya tatu?
   question in English: Is it normal to feel tired in the third trimester?
   danger: False | None
   answer: Ndiyo, ni jambo la kawaida kuhisi uchovu katika miezi mitatu. Mahitaji ya nishati huendelea kuwa ya chini sana wakati wa trimester ya kwanza, lakini huongeza hadi zaidi ya kalori mia tatu hadi mia nne na hamsini kila siku wakati wa tr

# Part 4: Evaluation: English vs Kiswahili

Two questions, answered with the same tests in both languages:
1. **Does retrieval still find the right chunks?** (Precision@3, Recall@3, MRR, nDCG@3)
2. **Are danger signs still caught, without false alarms?**

## Test retrieval quality using the 30 gold questions created during phase 3.5

In [ ]:
gold_questions_sw = [
    "Shirika la Afya Duniani (WHO) linapendekeza kliniki ngapi za uja uzito wakati wa ujauzito?",
    "Shinikizo kubwa la damu la ujauzito hugunduliwaje?",
    "Ni nini kinachosababisha kisukari cha ujauzito?",
    "Je, ukatili wa majumbani au kutoka kwa mwenza una uenezi gani wakati wa ujauzito?",
    "Ni zipi dalili za mapema zinazoashiria kuwa uchungu wa uzazi umeanza?",
    "Ni hali gani za kiafya zilizopo awali zinazoweza kufanya ujauzito kuwa wa hatari kubwa?",
    "Bakteria wa Group B Streptococcus hugunduliwaje wakati wa ujauzito?",
    "Maambukizi ya damu kwa mtoto mchanga (newborn sepsis) ni nini?",
    "Homa kali na kichwa kuuma sana baada ya kujifungua vinaashiria nini?",
    "Ni mambo gani yanayopaswa kujumuishwa katika mpango wa kujiandaa na uzazi?",
    "Mwanamke mjamzito anapaswa kutumia kiasi gani cha madini ya chuma (iron) kila siku?",
    "Ni nini kisababishi kikuu cha kuvuja damu kwa wingi baada ya kujifungua?",
    "Je, kuvimba kwa miguu na vifundo vya miguu wakati wa ujauzito ni hatari?",
    "Kikonyo cha kitovu cha mtoto kinapaswa kuhudumiwaje?",
    "Ni nini kinachoweza kupunguza au kutuliza kiungulia(heartburn) wakati wa ujauzito?",
    "Kwa nini wanawake wajawazito wanapaswa kutembea na kadi/rekodi zao za kliniki?",
    "Ni nini mfumo wa huduma endelevu za uzazi unaoongozwa na wakunga?",
    "Kwa nini madini ya chuma (iron) ni muhimu wakati wa ujauzito?",
    "Matumizi ya vidonge vya folic acid yanapaswa kuanza lini wakati wa ujauzito?",
    "Kifafa cha mimba kilichokomaa ni nini na inahusianaje na preeclampsia?",
    "Ni dawa gani inayotumika kuzuia kifafa wakati wa preeclampsia kali?",
    "Ni matatizo gani ya kupumua baada ya kujifungua yanayohitaji matibabu ya dharura?",
    "Ni dalili gani za mfumo wa mkojo baada ya kujifungua zinazohitaji uangalizi wa kitatibu?",
    "Ni vipi kukaza kwa misuli ya miguu (leg cramps) wakati wa ujauzito kunavyoweza kutulizwa?",
    "Ni nini kinachosaidia kupunguza au kutibu kufunga choo (constipation) wakati wa ujauzito?",
    "Maambukizi ya listeriosis yanawezaje kuzuiwa wakati wa ujauzito?",
    "Kwa nini maambukizi ya ugonjwa wa rubella ni hatari katika miezi ya kwanza ya ujauzito?",
    "Watoto waliozaliwa kabla ya wakati (njiti) au wenye uzito mdogo wanapaswa kuwekwa joto kwa njia gani?",
    "Ni chanjo gani ambazo mtoto mchanga hupewa anapozaliwa nchini Kenya?",
    "Ni nambari gani ya simu ya dharura inayoweza kupigiwa kwa ajili ya msaada wa haraka wa kitatibu nchini Kenya?",


]
print(len(gold_questions_sw), "Kiswahili questions")

30 Kiswahili questions


## Define the four retrieval quality metrics

In [ ]:
# The four metric functions (precision_at_k, recall_at_k, reciprocal_rank, ndcg_at_k) come from Phase 3.5.
# evaluate() runs questions through retrieval (optionally translating to English first).
# summarize() prints the four averages.
def evaluate(questions, gold, k=3, to_english=None):
    rows = []
    for q, g in zip(questions, gold):
        q_en = to_english(q) if to_english else q
        retrieved = get_retrieved_ids(q_en, k)
        rel = g["relevant_chunk_ids"]
        rows.append({
            "original": q,
            "english_used": q_en,
            "retrieved": retrieved,
            "precision": precision_at_k(retrieved, rel),
            "recall": recall_at_k(retrieved, rel),
            "mrr": reciprocal_rank(retrieved, rel),
            "ndcg": ndcg_at_k(retrieved, rel),
        })
    return rows

def summarize(rows, label, k=3):
    print(f"{label}  Precision@{k}={np.mean([r['precision'] for r in rows]):.3f}  "
          f"Recall@{k}={np.mean([r['recall'] for r in rows]):.3f}  "
          f"MRR={np.mean([r['mrr'] for r in rows]):.3f}  "
          f"nDCG@{k}={np.mean([r['ndcg'] for r in rows]):.3f}")

**The comparison.** Six versions of retrieval on the same 30 questions:
- English (the baseline)
- Kiswahili translated only
- Kiswahili translated with the glossary
- the same two without the English words in brackets (the fairer test, because brackets such as *(newborn sepsis)* give the system a hint)
- Kiswahili raw, with no translation at all (to show translation is needed)

In [ ]:
def strip_brackets(t):
    return re.sub(r"\s*\([^)]*\)", "", t).strip()

gold_sw_plain = [strip_brackets(q) for q in gold_questions_sw]          # no English hints
to_en       = lambda t: translate(t, "sw", "en")
to_en_gloss = lambda t: translate(apply_glossary(t), "sw", "en")

en_rows      = evaluate([g["question"] for g in gold_questions], gold_questions)
sw_rows      = evaluate(gold_questions_sw, gold_questions, to_english=to_en)
sw_rows_g    = evaluate(gold_questions_sw, gold_questions, to_english=to_en_gloss)
plain_rows   = evaluate(gold_sw_plain,     gold_questions, to_english=to_en)
plain_rows_g = evaluate(gold_sw_plain,     gold_questions, to_english=to_en_gloss)
raw_rows     = evaluate(gold_sw_plain,     gold_questions)              # no translation

summarize(en_rows,      "English                           ")
summarize(sw_rows,      "Kiswahili, translation only       ")
summarize(sw_rows_g,    "Kiswahili + glossary              ")
summarize(plain_rows,   "No brackets, translation only     ")
summarize(plain_rows_g, "No brackets + glossary            ")
summarize(raw_rows,     "Raw Kiswahili, no translation     ")

print("\nStill losing a correct chunk (no brackets + glossary):")
for i, (e, s) in enumerate(zip(en_rows, plain_rows_g), start=1):
    if s["recall"] < e["recall"]:
        print(f"Q{i}: {s['original']}")
        print("   became:", s["english_used"])
        print("   got   :", s["retrieved"], "| expected:", gold_questions[i - 1]["relevant_chunk_ids"])

English                             Precision@3=0.344  Recall@3=0.972  MRR=0.950  nDCG@3=0.940
Kiswahili, translation only         Precision@3=0.311  Recall@3=0.894  MRR=0.811  nDCG@3=0.812
Kiswahili + glossary                Precision@3=0.333  Recall@3=0.961  MRR=0.900  nDCG@3=0.896
No brackets, translation only       Precision@3=0.300  Recall@3=0.861  MRR=0.794  nDCG@3=0.791
No brackets + glossary              Precision@3=0.333  Recall@3=0.961  MRR=0.917  nDCG@3=0.908
Raw Kiswahili, no translation       Precision@3=0.044  Recall@3=0.133  MRR=0.078  nDCG@3=0.092

Still losing a correct chunk (no brackets + glossary):
Q6: Ni hali gani za kiafya zilizopo awali zinazoweza kufanya ujauzito kuwa wa hatari kubwa?
   became: What preexisting health conditions can make pregnancy more dangerous?
   got   : ['DOC_PRE_034_0', 'DOC_HRP_176_0', 'DOC_HRP_181_0'] | expected: ['DOC_HRP_177_0', 'DOC_PRE_034_0', 'DOC_PRE_035_0']


## Danger-sign tests

**Labels:** *True Positive* = real danger (must alert). *True Negative* = ordinary question (must **not** alert). *Borderline* = could go either way.

Two sets: the same 30 Engish scenario questions written in Kiswahili, and a mixed Kiswahili/English set (the way people really type).

In [ ]:
# 30 scenarios written in Kiswahili (same order and categories as the English ones),
# plus a mixed Kiswahili/English set. TP = must alert, TN = must NOT alert.

sw_scen_text = """Ninatokwa na damu nyingi sana na simsikii mtoto akisogea kabisa
Nina maumivu makali ya kichwa na macho yangu yanaona giza
Nimepata kifafa cha mimba hivi punde tu
Napata shida kubwa sana kupumua na kifua kinauma
Maumivu ya tumbo langu ni makali mno hayapoi
Nina homa kali sana na najisikia mnyonge kupita kiasi
Uso wangu na mikono yangu vimevimba sana siku hizi
Mtoto hasongi kabisa tangu asubuhi ya leo
Natapika sana na najisikia nimekauka maji mwilini
Wakati mwingine nawaza kujiumiza mwenyewe na kumuumiza mtoto
Ni vyakula gani ninapaswa kuviepuka nikiwa mjamzito?
Je, ni kawaida kujisikia nimechoka katika miezi ya tatu ya ujauzito?
Natakiwa kuongezeka uzito kiasi gani kufikia mwezi wa sita?
Ni mazoezi gani yaliyo salama wakati wa ujauzito?
Natakiwa kuhudhuria kliniki mara ngapi?
Ni vitamini gani ninapaswa kuwa natumia?
Ni lini nianze kuandaa begi langu la kwenda nalo hospitali?
Je, ni salama kusafiri wakati wa ujauzito?
Mfumo wa huduma endelevu za uzazi unaoongozwa na mkunga ni nini?
Nitunze vipi kikonyo cha kitovu cha mtoto wangu?
Leo nimekuwa na maumivu madogo ya kichwa yanayokuja na kuondoka
Vifundo vyangu vya miguu vimevimba kidogo kufikia jioni
Najisikia nimechoka zaidi ya kawaida na kizunguzungu kidogo
Mtoto amekuwa kimya au hasogei kama kawaida yake leo
Nimekuwa na maumivu madogo ya tumbo yanayokuja na kuondoka tangu asubuhi
Leo najisikia kichefuchefu kupita kiasi cha kawaida
Nina homa ndogo tu, sio kitu kibaya sana
Macho yangu yalionekana kufifia au kubibisha kwa muda mfupi tu mapema leo
Nimekuwa nikijisikia na hofu isiyo ya kawaida na kuzidiwa hivi karibuni
Nimegundua kuna madoa madogo ya damu leo"""

mixed_text = """TP | Nina bleeding nyingi sana na simsikii mtoto akisonga kabisa
TP | Niko na severe headache na macho yanabibisha vibaya sana
TP | Nimepata seizure sasa hivi niko hospitali
TP | Niko na trouble kubwa kupumua na chest pain haipoi
TP | Maumivu ya tummy ni severe sana na hayapoi hata kidogo
TP | Nina high fever na najisikia extremely weak mwili wote
TP | Face na hands zimevimba sana nowadays nimeshtuka
TP | Mtoto hasongi kabisa movement hakuna tangu morning
TP | Siwezi stop kutapika na naskia very dehydrated
TP | Wakati mwingine natink kujihurt na kudhuru mtoto
TN | What foods should i avoid nikiwa mjamzito hivi sasa?
TN | Je ni normal kujisikia tired kwenye third trimester?
TN | How much weight natakiwa niongezeke by month 6?
TN | Ni mazoezi gani safe kufanya wakati wa pregnancy?
TN | Antenatal visits ngapi ninapaswa kuhudhuria Strathmore au kliniki?
TN | Ni vitamins gani napaswa kumeza kwa sasa?
TN | When should i start kuandaa hospital bag yangu?
TN | Je ni safe kusafiri travel wakati wa pregnancy?
TN | What is a midwife led continuity of care model?
TN | How should i take care ya umbilical cord stump ya mtoto?
TN | What foods should I avoid during pregnancy?
TN | Is it normal to feel tired in the third trimester?
TN | How much weight should I gain by month 6?
TN | What exercises are safe during pregnancy?
TN | How many antenatal visits should I attend?
TN | What vitamins should I be taking?
TN | When should I start preparing my hospital bag?
TN | Is it safe to travel during pregnancy?
TN | What is a midwife-led continuity-of-care model?
TN | How should I care for my baby's umbilical cord stump?
TP | Nina bleeding na damu haikatiki tangu asubuhi
TP | Niko na severe headache na blurry vision kali sana
TP | Nimepatwa na convulsions na mwili unatetemeka
TP | Nina severe abdominal pain na simsikii baby kabisa
TP | Nina high fever kali na niko weak sana"""

sw_scen = [l.strip() for l in sw_scen_text.strip().split("\n") if l.strip()]
mixed = [tuple(p.strip() for p in l.split("|", 1)) for l in mixed_text.strip().split("\n") if "|" in l]

print(len(sw_scen), "Kiswahili scenarios;", len(mixed), "mixed-language messages")
assert len(sw_scen) == 30, f"Expected 30 scenarios but found {len(sw_scen)}"

30 Kiswahili scenarios; 35 mixed-language messages


**Run the three sets** and count alerts, false alarms and misses.

In [ ]:
def run_set(msgs, cats, language):
    rows = []
    for m, c in zip(msgs, cats):
        flag, phrase, q_en, sim = danger_only(m, language)
        rows.append({"cat": c, "msg": m, "alert": flag, "phrase": phrase, "en": q_en, "sim": sim})
    return rows

cats = [s["category"] for s in safety_scenarios]
en_rows_s = run_set([s["message"] for s in safety_scenarios], cats, "en")
sw_rows_s = run_set(sw_scen, cats, "sw")
mx_rows   = [{**r, "cat": "True Positive" if c == "TP" else "True Negative"}
             for (c, m), r in zip(mixed, run_set([m for _, m in mixed], ["x"] * len(mixed), "sw"))]

def report(rows, label):
    tp = [r for r in rows if r["cat"] == "True Positive"]
    tn = [r for r in rows if r["cat"] == "True Negative"]
    bl = [r for r in rows if r["cat"] == "Borderline"]
    print(f"\n=== {label} ===")
    if tp: print(f"True Positive: {sum(r['alert'] for r in tp)}/{len(tp)} alerted")
    if tn: print(f"True Negative: {sum(r['alert'] for r in tn)}/{len(tn)} false alarms")
    if bl: print(f"Borderline:    {sum(r['alert'] for r in bl)}/{len(bl)} alerted (review by hand)")
    for r in tp:
        if not r["alert"]: print("  MISSED:", r["msg"], "| became:", r["en"], "| similarity:", r["sim"])
    for r in tn:
        if r["alert"]: print("  FALSE ALARM:", r["msg"], "| similarity:", r["sim"])

report(en_rows_s, "English")
report(sw_rows_s, "Kiswahili")
report(mx_rows,   "Mixed Kiswahili and English")


=== English ===
True Positive: 10/10 alerted
True Negative: 1/10 false alarms
Borderline:    6/10 alerted (review by hand)
  FALSE ALARM: Is it safe to travel during pregnancy? | similarity: 0.537

=== Kiswahili ===
True Positive: 10/10 alerted
True Negative: 2/10 false alarms
Borderline:    6/10 alerted (review by hand)
  FALSE ALARM: Je, ni kawaida kujisikia nimechoka katika miezi ya tatu ya ujauzito? | similarity: 0.461
  FALSE ALARM: Je, ni salama kusafiri wakati wa ujauzito? | similarity: 0.537

=== Mixed Kiswahili and English ===
True Positive: 14/15 alerted
True Negative: 2/20 false alarms
  MISSED: Siwezi stop kutapika na naskia very dehydrated | became: I can't stop spitting and skiing very dehydrated | similarity: 0.161
  FALSE ALARM: Je ni safe kusafiri travel wakati wa pregnancy? | similarity: 0.537
  FALSE ALARM: Is it safe to travel during pregnancy? | similarity: 0.537


## Test using fresh messages

In [ ]:
def layers_fired(msg, language):
    kw_orig, _ = check_original_danger(msg)
    q_en = translate(apply_glossary(msg), "sw", "en") if language == "sw" else msg
    kw_en, _ = check_for_danger_signs(q_en)
    rule = swelling_rule(msg) or swelling_rule(q_en)
    sem, sim, _ = check_semantic_danger_sign(q_en, similarity_threshold=SAFETY_SIMILARITY_THRESHOLD)
    return {"kw_original": bool(kw_orig), "kw_english": bool(kw_en), "rule": bool(rule),
            "similarity": bool(sem), "sim": round(float(sim), 3), "en": q_en}

In [ ]:
# Fresh danger messages (the fair test)

fresh_text = """TP | sw | Natokwa damu sana hapa chini na simsikii mtoto kabisa akicheza
TP | en | My head is throbbing so bad and my vision is totally blurry right now
TP | mx | Nimepata seizure hapa home niko na convulsion kali sana
TP | sw | Napata shida kali sana ya kupumua na kifua kinabana kwelikweli
TP | en | The baby has not moved since morning at all i am scared
TP | sw | Tumbo linauma sana linaniuma vibaya mno haliishi hata kidogo
TP | en | I have a very high fever and my body feels super weak
TP | mx | Uso na hands zimevimba vibaya sana leo asubuhi nimeshtuka
TP | sw | Maji ya uzazi yametoka yote kabla ya wakati kufika
TP | en | Sometimes i feel like hurting myself and the baby too
TP | mx | Nina severe tummy pain na simsikii baby kabisa tangu jana
TP | sw | Nina kizunguzungu kikali na natapika kila kitu hakikai
TN | sw | Nianze lini kula vidonge vya chuma kliniki?
TN | en | What foods should I avoid during the first trimester?
TN | mx | Je ni normal kupata leg cramps wakati wa pregnancy?
TN | sw | Nitaongezeka uzito kiasi gani nikifika mwezi wa sita?
TN | en | When should I start packing my hospital bag for delivery?
TN | mx | Ni mazoezi gani safe kufanya nikiwa mjamzito?"""

In [ ]:
fresh = []
for line in fresh_text.strip().split("\n"):
    if line.count("|") >= 2 and not line.startswith("#"):
        cat, lang, msg = [p.strip() for p in line.split("|", 2)]
        fresh.append({"cat": "True Positive" if cat.upper() == "TP" else "True Negative",
                      "lang": "en" if lang.lower() == "en" else "sw", "msg": msg})
print(len(fresh), "fresh messages")

fresh_rows = []
for item in fresh:
    L = layers_fired(item["msg"], item["lang"])
    alert = L["kw_original"] or L["kw_english"] or L["rule"] or L["similarity"]
    fresh_rows.append({**item, **L, "alert": bool(alert)})

tp = [r for r in fresh_rows if r["cat"] == "True Positive"]
tn = [r for r in fresh_rows if r["cat"] == "True Negative"]
print(f"Danger messages caught: {sum(r['alert'] for r in tp)}/{len(tp)}")
print(f"False alarms:           {sum(r['alert'] for r in tn)}/{len(tn)}")
for r in tp:
    if not r["alert"]: print("MISSED:", r["msg"], "| English:", r["en"], "| similarity:", r["sim"])
for r in tn:
    if r["alert"]:     print("FALSE ALARM:", r["msg"], "| similarity:", r["sim"])
only_sim = [r for r in tp if r["alert"] and not (r["kw_original"] or r["kw_english"] or r["rule"])]
print("Caught by similarity only:", len(only_sim))


18 fresh messages
Danger messages caught: 12/12
False alarms:           1/6
FALSE ALARM: Je ni normal kupata leg cramps wakati wa pregnancy? | similarity: 0.575
Caught by similarity only: 4


## Save the results

In [ ]:
import os
os.makedirs("sw_experiment", exist_ok=True)
json.dump({"english": en_rows, "kiswahili": sw_rows, "kiswahili_glossary": sw_rows_g,
           "no_brackets": plain_rows, "no_brackets_glossary": plain_rows_g, "raw": raw_rows,
           "glossary": GLOSSARY},
          open("sw_experiment/retrieval_all_variants.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
json.dump({"english": en_rows_s, "kiswahili": sw_rows_s, "mixed": mx_rows, "fresh": fresh_rows},
          open("sw_experiment/danger_tests.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Saved to sw_experiment/")

Saved to sw_experiment/


## End-to-end check on Kiswahili questions

In [ ]:
for n in [1, 3, 5, 10, 14]:
    q_sw = gold_questions_sw[n - 1]
    r = get_full_response_multilingual(q_sw, language="sw")
    print(f"--- Q{n}: {q_sw}")
    print("English the pipeline saw:", r["question_en"])
    if r["danger_detected"]:
        print("ALERT shown instead of an answer")
    else:
        print("ANSWER (English)  :", r["answer_en"])
        print("ANSWER (Kiswahili):", r["final_answer"])
        print("Numbers kept:", numbers_preserved(r["answer_en"], r["final_answer"])[0])
    print()

--- Q1: Shirika la Afya Duniani (WHO) linapendekeza kliniki ngapi za uja uzito wakati wa ujauzito?
English the pipeline saw: How many pregnancy clinics does the World Health Organization (WHO) recommend?
ANSWER (English)  : I'm sorry, but the context provided does not include information on the number of pregnancy clinics recommended by the World Health Organization (WHO).
ANSWER (Kiswahili): Naomba radhi, lakini muktadha uliotolewa haujumuishi habari juu ya idadi ya kliniki za ujauzito zinazopendekezwa na Shirika la Afya Ulimwenguni (WHO).
Numbers kept: True

--- Q3: Ni nini kinachosababisha kisukari cha ujauzito?
English the pipeline saw: What causes gestational diabetes?
ANSWER (English)  : Gestational diabetes is caused by the pancreas not being able to produce enough insulin to meet the increased demands of pregnancy. During gestation, hormones released by the placenta cause insulin resistance by blocking how the mother's body uses insulin, leading to elevated blood sugar levels i

# Part 5: Results
## Retrieval, k = 3 (30 questions)

| Version | Precision@3 | Recall@3 | MRR | nDCG@3 |
|---|---|---|---|---|
| English (baseline) | 0.344 | 0.972 | 0.950 | 0.940 |
| Kiswahili, translation only | 0.311 | 0.894 | 0.811 | 0.812 |
| Kiswahili + glossary | 0.333 | 0.961 | 0.900 | 0.896 |
| No brackets, translation only | 0.300 | 0.861 | 0.794 | 0.791 |
| No brackets + glossary | 0.333 | 0.961 | 0.917 | 0.908 |
| Kiswahili, no translation | 0.044 | 0.133 | 0.078 | 0.092 |

**What this says:** translating helps alot (recall 0.13 to 0.86), and the glossary closes most of the remaining gap (0.86 to 0.96) Recall for English is
highest at 0.97. Raw KIswahili has the lowest recall score because the model
understands English only.

## Danger signs

| Set | Danger caught | False alarms (ordinary questions) |
|---|---|---|
| English scenarios | 10 / 10 | 1 / 10 |
| Kiswahili scenarios | 10 / 10 | 2 / 10 |
| Mixed Kiswahili/English | 14 / 15 | 2 / 20 |
| **Fresh messages (the fair test)** | **10 / 12** | **1 / 6** |



# Part 6: RAGAS Evaluation

# Part 7: Serve the pipeline to the website

In [73]:
NO_INFO_PHRASES = [
    "don't have that information", "do not have that information",
    "don't have information", "do not have information",
    "not mentioned in the context", "not provided in the context",
    "does not contain", "doesn't contain", "no information",
]

In [75]:
# Part 7: Serve the pipeline to the website (demo only)
import threading, subprocess, re, time, requests
from flask import Flask, request, jsonify

# The secret word. The website must send the same word, or the request is refused.
RAG_SECRET = "PUT-YOUR-SECRET-WORD-HERE"

rag_app = Flask(__name__)

@rag_app.route("/ask", methods=["POST"])
def ask():
    if request.headers.get("X-RAG-Key") != RAG_SECRET:
        return jsonify(error="Not allowed."), 403

    data = request.get_json(silent=True) or {}
    question = str(data.get("question", "")).strip()
    language = "sw" if data.get("language") == "sw" else "en"
    if not question:
        return jsonify(error="No question."), 400

    # The whole pipeline, already built above
    result = get_full_response_multilingual(question, language=language)

    # Turn the retrieved chunk ids into source names
    sources = []
    for chunk in chunk_lookup:
        if chunk["chunk_id"] in result["retrieved_chunk_ids"]:
            names = chunk["source"] if isinstance(chunk["source"], list) else [chunk["source"]]
            for name in names:
                if name not in sources:
                    sources.append(name)

    return jsonify(
        danger=result["danger_detected"],
        answer=result["final_answer"],
        sources=sources,
    )

# threaded=False: the GPU answers one question at a time
threading.Thread(
    target=lambda: rag_app.run(port=5001, threaded=False, use_reloader=False),
    daemon=True,
).start()
print("Server started.")

 * Serving Flask app '__main__'
Server started.
 * Debug mode: off


Address already in use
Port 5001 is in use by another program. Either identify and stop that program, or start the server with a different port.


## Get a public address

In [76]:
# Download the tunnel program and open it
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:5001"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

public_url = None
start = time.time()
while time.time() - start < 60:
    line = tunnel.stdout.readline()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

# Keep reading the tunnel's messages so it never gets stuck
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

print("Your public address:", public_url)

cloudflared: Text file busy
Your public address: https://vancouver-reviewed-practitioners-residents.trycloudflare.com


## Test the public address

In [77]:
time.sleep(5)  # the new address needs a few seconds to become reachable

tests = [
    ("What is folic acid used for?", "en"),
    ("Folic acid inasaidia nini?", "sw"),
    ("What should I prepare before giving birth?", "en"),
    ("Nijiandaeje kabla ya kujifungua?", "sw"),
    ("I have a severe headache", "en"),
]

for question, language in tests:
    reply = requests.post(
        public_url + "/ask",
        json={"question": question, "language": language},
        headers={"X-RAG-Key": RAG_SECRET},
        timeout=180,
    )
    data = reply.json()

    print("QUESTION  :", question, f"({language})")
    print("STATUS    :", reply.status_code)
    print("DANGER    :", data.get("danger"))
    print("ANSWER    :", data.get("answer"))

    sources = data.get("sources", [])
    if sources:
        print("SOURCES   :")
        for name in sources:
            print("   -", name)
    elif data.get("danger"):
        print("SOURCES   : none (danger alert, so nothing was searched)")
    else:
        print("SOURCES   : NONE  <-- a normal question should have sources")
    print("-" * 60)

INFO:werkzeug:127.0.0.1 - - [03/Nov/2026 18:20:56] "POST /ask HTTP/1.1" 200 -


QUESTION  : What is folic acid used for? (en)
STATUS    : 200
DANGER    : False
ANSWER    : Folic acid is used for cellular division, rapid tissue growth, and proper neural tube development in early pregnancy. It helps in the complete closure of the embryonic neural tube within twenty-eight days of conception, reducing the risk of major congenital brain and spinal cord defects such as spina bifida and anencephaly by up to seventy percent.
SOURCES   :
   - World Health Organization
   - Johns Hopkins Medicine
------------------------------------------------------------


INFO:werkzeug:127.0.0.1 - - [03/Nov/2026 18:21:02] "POST /ask HTTP/1.1" 200 -


QUESTION  : Folic acid inasaidia nini? (sw)
STATUS    : 200
DANGER    : False
ANSWER    : Asidi ya foli ni muhimu kwa ajili ya kugawanywa kwa chembe, ukuaji wa tishu wa haraka, na maendeleo sahihi ya chembe za neva wakati wa ujauzito wa mapema. Hufanya iwezekane kufungwa kabisa kwa chombo cha neva cha kiinitete ndani ya siku ishirini na nane baada ya mimba, ikipunguza hatari ya kutokea kwa kasoro kubwa za kiinitete za ubongo na uti wa mgongo kama vile spina bifida na anencefalia kwa asilimia sabini.
SOURCES   :
   - World Health Organization
   - Johns Hopkins Medicine
------------------------------------------------------------


INFO:werkzeug:127.0.0.1 - - [03/Nov/2026 18:21:09] "POST /ask HTTP/1.1" 200 -


QUESTION  : What should I prepare before giving birth? (en)
STATUS    : 200
DANGER    : False
ANSWER    : Before giving birth, you should prepare essential childbirth supplies such as clean clothes and soft blankets for drying, warming, and wrapping the newborn baby. It's also important to gather other necessary items to promote good hygiene and ensure proper immediate care after birth. Additionally, discuss and review your birth preparedness plan with a midwife or healthcare provider during regular antenatal clinic visits, and update the plan as the expected due date gets closer to ensure all arrangements for travel, funds, and emergency contacts are clear and ready.
SOURCES   :
   - World Health Organization
   - Kenyatta University Hospital
------------------------------------------------------------


INFO:werkzeug:127.0.0.1 - - [03/Nov/2026 18:21:16] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/Nov/2026 18:21:16] "POST /ask HTTP/1.1" 200 -


QUESTION  : Nijiandaeje kabla ya kujifungua? (sw)
STATUS    : 200
DANGER    : False
ANSWER    : Ili kujitayarisha kwa ajili ya kujifungua, unapaswa kukusanya vifaa muhimu vya kujifungua mapema, kutia ndani nguo safi na blanketi laini kwa ajili ya kukausha, kupasha joto, na kumfunga mtoto mchanga. Kwa kuongezea, zungumzia na kuchunguza mpango wako wa maandalizi ya kujifungua pamoja na mkunga au mtoa huduma wa afya wakati wa ziara za kawaida za kliniki kabla ya kujifungua. Tathmini na sasisha mpango huo wakati tarehe ya kujifungua inakaribia ili kuhakikisha kwamba mipango yote ya usafiri, fedha, na mawasiliano ya dharura inabaki wazi na tayari.
SOURCES   :
   - World Health Organization
   - Kenyatta University Hospital
------------------------------------------------------------
QUESTION  : I have a severe headache (en)
STATUS    : 200
DANGER    : True
ANSWER    : This sounds like it could be a serious warning sign during pregnancy. Please go to your nearest hospital or health facility 

## Fix the sources problem showing in non-related maternal health questions

In [78]:
def ask_v2():
    if request.headers.get("X-RAG-Key") != RAG_SECRET:
        return jsonify(error="Not allowed."), 403

    data = request.get_json(silent=True) or {}
    question = str(data.get("question", "")).strip()
    language = "sw" if data.get("language") == "sw" else "en"
    if not question:
        return jsonify(error="No question."), 400

    result = get_full_response_multilingual(question, language=language)

    # If the answer says it has no information, do not list sources
    english_answer = (result.get("answer_en") or result["final_answer"] or "")
    english_answer = english_answer.lower().replace("\u2019", "'")
    no_information = any(phrase in english_answer for phrase in NO_INFO_PHRASES)

    sources = []
    if not no_information:
        for chunk in chunk_lookup:
            if chunk["chunk_id"] in result["retrieved_chunk_ids"]:
                names = chunk["source"] if isinstance(chunk["source"], list) else [chunk["source"]]
                for name in names:
                    if name not in sources:
                        sources.append(name)

    return jsonify(
        danger=result["danger_detected"],
        answer=result["final_answer"],
        sources=sources,
    )

rag_app.view_functions["ask"] = ask_v2
print("Updated.")

Updated.


In [79]:
ordinary = [
    "Why are iron supplements important during pregnancy?",
    "What are the symptoms of anaemia?",
    "Is it safe to exercise during pregnancy?",
    "How much calcium do I need?",
    "What vitamins should I take?",
    "Is it normal to feel tired in the third trimester?",
    "How can I sleep better while pregnant?",
    "What should I eat to gain healthy weight?",
    "Is it safe to travel during pregnancy?",
    "What is morning sickness?",
]
real_danger = [
    "I feel very dizzy and weak",
    "I am bleeding heavily",
    "My baby has not kicked since morning",
    "I keep fainting",
    "My vision is blurry and my head hurts",
    "I have chest pain",
    "My water broke early",
    "I cannot breathe properly",
]

def show(title, messages):
    print(title)
    for msg in messages:
        r = check_danger_sign_combined(msg)
        layer = "keyword" if r["keyword_match"] else ("meaning" if r["is_danger"] else "-")
        print(f'  {r["is_danger"]!s:5}  score {r["semantic_similarity"]:.3f}  layer: {layer:8} {r["semantic_closest_chunk"]}  | {msg}')
    print()

show("ORDINARY QUESTIONS (should be False)", ordinary)
show("REAL DANGER (should be True)", real_danger)

ORDINARY QUESTIONS (should be False)
  True   score 0.462  layer: meaning  DOC_DS_029_0  | Why are iron supplements important during pregnancy?
  False  score 0.448  layer: -        DOC_DS_029_0  | What are the symptoms of anaemia?
  False  score 0.434  layer: -        DOC_DS_025_0  | Is it safe to exercise during pregnancy?
  False  score 0.033  layer: -        DOC_DS_029_0  | How much calcium do I need?
  False  score 0.181  layer: -        DOC_DS_029_0  | What vitamins should I take?
  False  score 0.447  layer: -        DOC_DS_028_0  | Is it normal to feel tired in the third trimester?
  False  score 0.381  layer: -        DOC_DS_028_0  | How can I sleep better while pregnant?
  False  score 0.032  layer: -        DOC_DS_029_0  | What should I eat to gain healthy weight?
  True   score 0.537  layer: meaning  DOC_DS_017_0  | Is it safe to travel during pregnancy?
  True   score 0.455  layer: meaning  DOC_DS_026_0  | What is morning sickness?

REAL DANGER (should be True)
  True   sc